# Unified format - from a survey export to the IPCC software (25 November 2026)

*25 November 2026. From one or several survey exports (Collect Earth, Collect Earth Online) to one format, the same for every survey and country, then the classes of the country, the areas, and the files for the IPCC software and mocaredd. The steps: several files read together (Step 1); survey summary and description, with how the land names were read (Step 2; one sample in Step 2b); preliminary quality check: duplicate plots and plots at the same place stopped, gaps in the years listed, changes that do not match the land (Step 3); unified format with one row per sample per year: the land use and the subdivision class are the basic information, with the attributes of the class, e.g. the canopy before and after (Step 4); the history of each sample as recorded: the pairs of years and the years side by side (Step 5); the country's rules, only if needed, in one workbook (transitions between forest types, canopy cover transitions when the survey records the canopy, reclassify, merge), with the classes recorded and reported and the history reported (Step 6); the areas with their standard errors, systematic or stratified sample, the matrices and the transitions for mocaredd (Step 7); the units of land for the IPCC software (Step 8); a review of a few complex samples (Step 9); the activity of a period, e.g. reference or monitoring: one activity per sample, with its area (Step 10). All the code is in Step 0.*

| Step | | You get |
|---|---|---|
| 1 | Read the export(s): one file, several files or a folder | |
| 2 | Describe the survey | `<country>_step2_survey_structure_<date>.html` (with the questions) |
| 2b | One sample (optional) | `<country>_step2_sample_<id>_<date>.html` |
| 3 | Quality check: duplicates, same place, gaps, inconsistencies | `_step3_errors_` CSV, `_step3_to_correct_` CSV |
| 4 | Unified format, with the canopy before and after | `_step4_unified_format_` CSV |
| 5 | History of the samples | `_step5_pairs_` CSV, `_step5_history_` CSV (one row per plot: land use, subdivision, canopy of every year side by side) |
| 6 | Country rules and REDD+ classes | `_step6_country_rules_template_` Excel, `_step6_classes_` CSV (or `_reported_` and `_recorded_`), `_step5_history_reported_` CSV |
| 7 | Area, standard errors and matrices | `_step7_strata_` CSV, `_step7_class_areas_` CSV, `_step7_matrices_` Excel, `_step7_transitions_` CSV, `_step7_mocaredd_transitions_` Excel |
| 8 | IPCC software | `_step8_ipcc_subcategories_` Excel, `_step8_ipcc_units_of_land_` Excel |
| 9 | Review of a few samples (optional) | `_step9_review_samples_` HTML and CSV |
| 10 | The activity of a period (optional) | `_step10_history_lookup_template_` Excel, `_step10_period_activity_` CSV, `_step10_period_areas_` CSV |

**Fill in:** `file_path` (Step 1), `end_year` (Step 2, Collect Earth), one file in Steps 6 and 8, `total_area_ha` (Step 7).

## Step 0 - Setup
**What you do:** run it once (internet needed the first time).

In [ ]:
#@title Step 0 - Setup
"""Unified format - from survey exports to the IPCC software. 24 November 2026: v19 + several files (Step 1), duplicates,
same place and gaps (Step 3), canopy before / after (Step 4), canopy rule (Step 6), review of samples (Step 9).
VERSION 19 (20 November 2026): v18 with the lookup word 'Stable' read as 'Stable forest' and the forest types recorded kept after 'Stable forest' and 'Not possible'. v18 (20 November 2026): v17 + the history after the country's rules (Step 6, _step5_history_reported_). v17 (17 November 2026): v16 + density order (Step 6) + sampling design, weights, standard errors and the mocaredd
transitions (Step 7). v16: v15 + classes recorded / reported in Step 6 (reclassify, merge) + fix of CHANGE_ATTRIBUTES.
All the code of the notebook. What each step does, the columns and the files: Unified_Format_Tool_v8_description.docx."""

import io
import re
import unicodedata
from collections import Counter, defaultdict
from datetime import date
from html import escape

import pandas as pd

# ----------------------------------------------------------------------------------------------- the file

EXCEL_ROWS = 1_048_575          # rows of data an Excel sheet holds (plus the header)


def kind(record):
    """'plot', 'state' or 'change' from the record name (P, S0, C1...)."""
    r = str(record)
    return "plot" if r == "P" else "state" if r.startswith("S") else "change" if r.startswith("C") else ""

# ----------------------------------------------------------------------------------------------- values


def plain(text):
    """lower case, no accents, no punctuation: 'Terres_Forestières' -> 'terres forestieres'."""
    text = unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", " ", text.lower()).strip()


IPCC_NAMES = {
    "Forest land": ["f", "forest", "forestland", "forest land", "forests", "terres forestieres", "terre forestiere", "foret",
                    "forets", "tf", "bosque", "bosques", "tierras forestales", "terras florestais", "floresta"],
    "Cropland": ["c", "cropland", "croplands", "crop land", "terres cultivees", "terre cultivee", "tc", "cultivos",
                 "tierras de cultivo", "terras agricolas"],
    "Grassland": ["g", "grassland", "grasslands", "prairies", "prairie", "terres gramineennes", "tg", "pastizales",
                  "praderas", "pastagens"],
    "Wetlands": ["w", "wetland", "wetlands", "terres humides", "zones humides", "th", "humedales", "areas umidas"],
    "Settlements": ["s", "settlement", "settlements", "etablissements", "etablissement", "etablissement humain",
                    "etablissements humains", "eh", "asentamientos", "assentamentos"],
    "Other land": ["o", "other land", "otherland", "other lands", "autres terres", "autre terre", "at", "otras tierras",
                   "outras terras"],
}
for _cat, _code in (("Forest land", "fl"), ("Cropland", "cl"), ("Grassland", "gl"), ("Wetlands", "wl"),
                    ("Settlements", "sl"), ("Other land", "ol")):
    IPCC_NAMES[_cat].append(_code)                       # two-letter codes, e.g. Kenya ipcc1990 = FL, CL, GL...
for _cat, _more in (("Forest land", ["tierra forestal", "terra florestal", "terras florestais", "florestas", "foresta", "terre forestali"]),
                    ("Cropland", ["tierra de cultivo", "tierras agricolas", "tierra agricola", "terra agricola", "terras cultivadas", "cultivo"]),
                    ("Grassland", ["pastizal", "pradera", "pastagem", "terres de prairie", "pascolo"]),
                    ("Wetlands", ["humedal", "area umida", "zone humide", "zona umida"]),
                    ("Settlements", ["asentamiento", "assentamento", "insediamenti"]),
                    ("Other land", ["otra tierra", "outra terra", "altre terre"])):
    IPCC_NAMES[_cat] += [n for n in _more if n not in IPCC_NAMES[_cat]]        # 2027: more spellings
IPCC_LOOKUP = {plain(n): cat for cat, names in IPCC_NAMES.items() for n in names}
YES = {"yes", "true", "oui", "si", "sim", "1"}
NO = {"no", "false", "non", "nao", "0"}


def ipcc_category(value, label=None):
    """the IPCC category of a land-use answer (code or label), or None if it is not one."""
    for v in (value, label):
        if v is not None and plain(v) in IPCC_LOOKUP:
            return IPCC_LOOKUP[plain(v)]
    return None


def yes_no(value):
    p = plain(value)
    return "yes" if p in YES else "no" if p in NO else None


def given(value):
    """True when an answer is filled in."""
    return value is not None and not (isinstance(value, float) and pd.isna(value)) and str(value).strip() not in ("", "nan", "None")


def fix_accents(text):
    """repair accents broken by a double encoding ('ForÃªt' -> 'Forêt')."""
    if isinstance(text, str) and ("Ã" in text or "â€" in text or "Â" in text):
        try:
            return text.encode("cp1252").decode("utf-8")
        except (UnicodeEncodeError, UnicodeDecodeError):
            return text
    return text

# ----------------------------------------------------------------------------------------------- Step 1: read


def one_hot_to_answers(df):
    """CEO plot-data export: one column per answer ('Question:Answer' = share of samples, 0-100).
    Turned into one column per question holding the answer with the largest share (ties: joined with ' | ')."""
    groups = defaultdict(list)
    for c in df.columns:
        if ":" in c:
            q, a = c.split(":", 1)
            groups[q].append((c, a))
    if not groups:
        return df
    out = df[[c for c in df.columns if ":" not in c]].copy()
    new = {}
    for q, cols in groups.items():
        vals = df[[c for c, _ in cols]].apply(pd.to_numeric, errors="coerce").fillna(0)
        best = vals.max(axis=1)
        answers = []
        for i in range(len(df)):
            if best.iat[i] <= 0:
                answers.append(None)
            else:
                answers.append(" | ".join(a for (c, a), v in zip(cols, vals.iloc[i]) if v == best.iat[i]))
        new[q] = answers
    return pd.concat([out, pd.DataFrame(new, index=df.index)], axis=1)

# ----------------------------------------------------------------------------------------------- Step 2: describe


# plot information (level 0): column name (plain) -> variable
PLOT_INFO = {
    "location x": "x", "location y": "y", "lon": "x", "lat": "y", "center lon": "x", "center lat": "y",
    "operator": "interpreter", "email": "interpreter", "collection time": "date",
    "country": "country", "province": "province", "district": "district",
    "elevation": "elevation", "slope": "slope", "aspect": "aspect",
    "climate": "climate_zone", "soil": "soil_type", "gez": "ecological_zone",
    "uncert": "confidence", "confidence": "confidence", "confiance": "confidence",
    "uncertainty reason": "difficulty_reason", "reason for difficulty": "difficulty_reason",
    "difficulty in interpretation": "interpretation_difficulty", "difficulty interpretation": "interpretation_difficulty",
    "most useful imagery": "imagery", "most useful tool": "tool", "imagery title": "imagery",
    "landscape": "landscape", "sample location": "point_location",
    "exp factor": "expansion_factor", "expansion factor": "expansion_factor", "plot weight": "expansion_factor",
    "strata ceo": "stratum", "strata": "stratum", "stratum": "stratum", "strate": "stratum",
    "flagged reason": "flag_reason", "group": "project_group",
    "further comments": "comment", "comment": "comment", "ajouter commentaire": "comment", "notes": "comment",
}
# columns that carry no observation (identifiers, technical or calculated fields)
NOT_USED = re.compile(
    r"^(id|plotid|sampleid|sample_internal_id|round|location_srs|plot_file|actively_saved|actively_saved_on_.*|"
    r"calculated_.*|land_use[a-z]*_lu|base_year|flag|flagged|flagged_reason|analysis_duration|used_geodash|used_kml|"
    r"imagery_attributions|imagery_used|sample_geom|shape|size_m|sample_points|pl_extra_plot_info|smpl_extra_plot_info|"
    r"canopy_cover_percent.*|source_file|.*_label|no|uid)$", re.I)

# state attributes (level 3): question stem (plain, without year or suffix) -> variable
STATE_ATTRIBUTES = {
    "f canopy cover": "canopy_cover", "canopy cover": "canopy_cover", "canopycover": "canopy_cover",
    "f canopy cover before": "canopy_cover", "f canopy cover before e": "canopy_cover",
    "canopy cover tof": "tree_cover", "trees outside": "tree_cover",
    "presence of trees": "trees_present", "presence trees": "trees_present",
    "native species": "native_species", "native spss": "native_species",
    "commercial use": "commercial_use", "commercial": "commercial_use",
    "f patch size": "patch_size", "forest patch size": "patch_size", "patch size": "patch_size", "forest size": "patch_size",
    "managed grassland": "managed", "grassland management": "managed",
    "cyclical system": "cyclical_system", "lenght cyclical": "cyclical_system",
    "temporarily un stocked": "temporarily_unstocked", "temporarily unstocked": "temporarily_unstocked",
}
# change attributes (level 3) and change information (level 4)
CHANGE_ATTRIBUTES = {
    "change size": "change_size", "disturbance size": "change_size", "enhancement size": "change_size",
    "canopy cover reduction": "canopy_cover_decrease", "canopy cover increase": "canopy_cover_increase",
    "extension of a previous change": "extends_previous_change", "extension of previous change": "extends_previous_change",
    "extension of a previous disturbance": "extends_previous_change", "extension of a previous enhancement": "extends_previous_change",
    "extension": "extends_previous_change",
    "temporary loss cyclical system": "temporary_loss", "temporary lu": "temporary_loss",
    "within distance buffer": "within_buffer", "disturbance within": "within_buffer", "disturbance within 1 ha": "within_buffer",
    "duration non f before gain": "nonforest_duration_before", "how long was non forest before": "nonforest_duration_before",
    "was forest before the non forest period": "forest_before_nonforest", "was forest": "forest_before_nonforest",
}
CHANGE_INFO = {
    "driver anthropogenic": "driver", "driver natural": "driver", "disturbance": "driver", "driver regrowth": "driver",
    "antro driver": "driver", "nat driver": "driver",
    "anthropogenic cause": "driver_origin", "lu change type": "change_label",
}


def stem(column):
    """'F Canopy cover before-SDFD1' -> 'f canopy cover before'; 'Native species 2023' -> 'native species'."""
    c = re.sub(r"\.\d+$", "", column)                     # pandas suffix of a repeated name
    c = re.sub(r"[-\s](L|SDFD|DWC|G|SDFR|E|NFLUC|NFSDC|FD|FR|NFC)\d+(\s*\(\d+\))?\s*$", "", c)
    c = re.sub(r"\s(19|20)\d\d\s*$", "", c.strip())
    return plain(c)


def normalise_layout(df):
    """A Collect Earth export merged or flattened after export: the change slots are suffixes (change_yr, change_yr2,
    change_yr3...) instead of event[1]_change_yr, event[2]_change_yr..., and a merge may have left _x / _y.
    Renamed in memory to the standard layout; returns (data, note)."""
    cols = list(df.columns)
    if "lu_t2" not in cols or any(re.match(r"^(event|change)\[\d+\]_", c) for c in cols):
        return df, ""
    bases = {m.group(1) for c in cols for m in [re.match(r"^(.+?\D)(\d+)$", c)]
             if m and int(m.group(2)) >= 2 and (m.group(1) in cols or m.group(1) + "_y" in cols)}
    if "change_yr" not in bases:
        return df, ""
    rename = {}
    for b in bases:
        first = b + "_y" if b + "_y" in cols else b
        rename[first] = f"event[1]_{b}"
        if b + "_x" in cols:
            rename[b + "_x"] = b                      # the plot-level question of the same name
        for c in cols:
            m = re.match(rf"^{re.escape(b)}(\d+)$", c)
            if m and int(m.group(1)) >= 2:
                rename[c] = f"event[{int(m.group(1))}]_{b}"
    return df.rename(columns=rename), f"merged layout: change slots renamed in memory ({len(rename)} columns, e.g. change_yr2 -> event[2]_change_yr)"


def detect_design(df):
    """tool and design, from the column names only."""
    cols = set(df.columns)
    if any(re.match(r"^event\[\d+\]_", c) for c in cols) and "lu_t2" in cols:
        return "CE", "event", "backward from the end year (event slots)"
    if any(re.match(r"^change\[\d+\]_", c) for c in cols) and "lu_t2" in cols:
        return "CE", "change", "backward from the end year (change blocks)"
    if "plotid" in cols:
        if any(re.match(r"^LU in ", c) for c in cols) and any(re.match(r"^1st change", c) for c in cols):
            return "CEO", "cards", "backward from the end year (change cards)"
        years = {m.group(0) for c in cols for m in [re.search(r"(19|20)\d\d", c)] if m}
        if len(years) >= 2:
            return "CEO", "annual", "the land at several dates"
        return "CEO", "single", "one date"
    if is_ipcc_form(cols):
        return "CE", "ipcc_form", "backward from the end year (Collect Earth IPCC form)"
    raise ValueError("Unknown export: neither Collect Earth (lu_t2 + event[...]/change[...]) nor Collect Earth Online (plotid).")


def end_year_from_columns(df):
    for c in df.columns:
        m = re.match(r"^LU in ((19|20)\d\d)", c)
        if m:
            return int(m.group(1))
    return None


def describe_survey(df, survey_id, end_year=0, numbering="newest"):
    """Step 2: what the export contains and the role of each column. Returns a dict 'survey'."""
    raw = df
    df, layout_note = normalise_layout(df)
    tool, design, design_text = detect_design(df)
    if tool == "CEO" and any(":" in c for c in df.columns):      # plot-data export: one column per answer
        df = one_hot_to_answers(df)
    found_end = end_year_from_columns(df)
    if design == "annual":
        years = sorted({int(m.group(0)) for c in df.columns for m in [re.search(r"(19|20)\d\d", c)] if m})
        found_end = years[-1]
    end = int(end_year) if end_year else found_end
    default_id = "id" if tool == "CE" else "plotid"
    point_col = "sampleid" if "sampleid" in df.columns and df[default_id].duplicated().any() else None
    id_col, id_note = pick_id_column(df, default_id, point_col)
    survey = {"survey_id": survey_id, "tool": tool, "design": design, "design_text": design_text,
              "end_year": end, "numbering": numbering, "data": df, "raw": raw, "layout_note": layout_note,
              "id_col": id_col, "id_note": id_note, "point_col": point_col, "plots": df[id_col].nunique()}
    survey["roles"] = roles_table(df, tool, design)
    if id_col != default_id:
        survey["roles"].loc[survey["roles"]["export column"] == id_col, ["record", "note"]] = ["-", "the plot id"]
    prepare_roles(survey)
    return survey


def pick_id_column(df, default, point_col=None):
    """the column that identifies the plots: the tool's own (id / plotid) when every value is unique; otherwise another
    column with 'id' in its name whose values are all given and unique (letters and numbers allowed, e.g. Kenya UID)."""
    if point_col or df[default].notna().all() and not df[default].duplicated().any():
        return default, ""
    for c in df.columns:
        if c != default and re.search(r"(^|_)u?id$|^uid|plot_?id", c, re.I) and df[c].notna().all() and not df[c].duplicated().any():
            return c, (f"'{default}' repeats ({int(df[default].duplicated().sum())} times), so '{c}' is used as the plot id "
                       f"(unique for every row)")
    return default, f"'{default}' repeats ({int(df[default].duplicated().sum())} times) and no other column is a unique id"


def roles_table(df, tool, design):
    """one row per column of the export: record, level, variable - the sheet 'where each column went'."""
    rows = []
    for c in df.columns:
        vals = df[c].dropna().astype(str).str.strip()
        vals = vals[vals != ""]
        years_only = len(vals) > 0 and vals.str.fullmatch(r"(19|20)\d\d(\.0)?").mean() > 0.9
        content = None
        if design == "annual" and re.search(r"(19|20)\d\d", c):     # snapshots: the role is read from the answers
            sample = vals.drop_duplicates().head(300)
            content = {"ipcc": sample.map(lambda v: ipcc_category(v) is not None).mean() if len(sample) else 0,
                       "numeric": len(sample) > 0 and bool(sample.str.fullmatch(r"-?\d+(\.\d+)?").all()),
                       "yes_no": len(sample) > 0 and bool(sample.map(lambda v: yes_no(v) is not None).all())}
        record, level, variable, note = role_of(c, tool, design, years_only, content)
        n = int(df[c].map(given).sum())
        rows.append({"export column": c, "record": record, "level": level, "variable": variable,
                     "note": note, "plots with an answer": n})
    return pd.DataFrame(rows)


def role_of(c, tool, design, years_only=False, content=None):
    """(record, level, variable, note) of one column. Records: P, S0 (end year), Sk / Ck (change k as asked)."""
    if design == "ipcc_form":
        r = role_ipcc_form(c)
        if r:
            return r
    if NOT_USED.match(c) or c in ("plotid", "sampleid", "id"):
        return "-", "", "", "not used (identifier or technical)"
    s = stem(c)
    if tool == "CE":
        m = re.match(r"^(event|change)\[(\d+)\]_(.+)$", c)
        if m:
            field = m.group(3)
            f = plain(field)
            if field in ("event_num", "change_num", "to_lu", "to_sub", "to_lu_label", "to_sub_label", "previous_label",
                         "from_lu_actual", "from_sub_actual", "from_lu_no_lu_change", "from_sub_no_change"):
                return "-", "", "", "not used (calculated in the form)"
            if field == "change_yr":
                return "Ck", "", "(years)", "year of the change"
            if field == "from_lu":
                return "Sk", 1, "ipcc_category", "land use before change k"
            if field == "from_sub":
                return "Sk", 2, "subcategory_1", "subcategory before change k"
            if f in STATE_ATTRIBUTES:
                return "Sk", 3, STATE_ATTRIBUTES[f], "state before change k"
            if f in CHANGE_ATTRIBUTES:
                return "Ck", 3, CHANGE_ATTRIBUTES[f], "attribute of change k"
            if f in CHANGE_INFO:
                return "Ck", 4, CHANGE_INFO[f], "change k"
            return "Ck", 3, field, "not known by the tool: only in raw_data"
        if c == "lu_t2":
            return "S0", 1, "ipcc_category", "land use in the end year"
        if c == "sub_t2":
            return "S0", 2, "subcategory_1", "subcategory in the end year"
    if tool == "CEO" and design == "cards":
        if re.match(r"^LU in ", c):
            return "S0", 1, "ipcc_category", "land use in the end year"
        m = re.match(r"^(Forest|Cropland|Grassland|Wetland|Settlement|Other land) subdivision(\s+(19|20)\d\d)?\s*$", c, re.I)
        if m:
            return "S0", 2, "subcategory_1", "subcategory in the end year"
        card = re.search(r"[-\s](L|SDFD|DWC|G|SDFR|E|NFLUC|NFSDC|FD|FR|NFC)(\d+)(\s*\(\d+\))?(\.\d+)?\s*$", c)
        if re.match(r"^(1st|2nd|3rd|\d+th) change", c):
            return "Ck", 4, "change_label", "the card answered for change k"
        if re.match(r"^(Forest disturbance|Forest recovery|NF change)", c):
            return "Ck", 4, "change_label", "kind of change k (card)"
        if card:
            if re.match(r"^Year-", c):
                return "Ck", "", "(years)", "year of the change"
            if re.match(r"^(Subdivision before change|.*subdivision-)", c, re.I):
                return "Sk", 2, "subcategory_1", "subcategory before change k"
            if re.match(r"^(LU before change|SD before change)", c):
                return "Sk", 1, "ipcc_category", "land use before change k"
            if s in CHANGE_INFO:
                return "Ck", 4, CHANGE_INFO[s], "change k"
            if s in CHANGE_ATTRIBUTES:
                return "Ck", 3, CHANGE_ATTRIBUTES[s], "attribute of change k"
            if s in STATE_ATTRIBUTES:
                return "Sk", 3, STATE_ATTRIBUTES[s], "state before change k"
            return "Ck", 3, s.replace(" ", "_"), "not known by the tool: only in raw_data"
    if tool == "CEO" and design == "annual":
        n_years = len(re.findall(r"(?:19|20)\d\d", c))
        if n_years >= 2:
            return period_role(c, years_only, content or {})
        if n_years == 1:
            return snapshot_role(c, re.search(r"(19|20)\d\d", c).group(0), content or {})
    if s in PLOT_INFO or plain(c) in PLOT_INFO:
        return "P", 0, PLOT_INFO.get(s, PLOT_INFO.get(plain(c))), "plot information"
    if re.match(r"^(pl|smpl)_", c):
        return "P", 0, c, "extra plot information of the project"
    if s in STATE_ATTRIBUTES:
        return "S0", 3, STATE_ATTRIBUTES[s], "state in the end year"
    return "P", 0, c, "not known by the tool: only in raw_data"

CANOPY_WORDS = r"fcc|canopy|cover|couvert|cobertura"


def snapshot_role(c, y, content):
    """a question asked for one date (e.g. Occupation_2021, ipcc1990, lc1990, fcc1990): its role is read from its answers.
    IPCC categories -> level 1; numbers under a canopy / cover name -> canopy cover; a class code -> not used."""
    p = plain(c)
    if re.search(r"\bcode\b", p) and content.get("numeric"):
        return "-", "", "", "not used (the code of a class column)"
    if content.get("ipcc", 0) >= 0.8:
        return f"S@{y}", 1, "ipcc_category", f"land use in {y} (IPCC categories)"
    if re.search(CANOPY_WORDS, p):
        if content.get("numeric"):
            return f"S@{y}", 3, "canopy_cover", f"canopy cover in {y}"
        return f"S@{y}", 3, "canopy_class", f"canopy cover class in {y}"
    if re.search(r"niveau\s*3|level\s*3", p):
        return f"S@{y}", 2, "subcategory_2", f"subcategory (level 3) in {y}"
    return f"S@{y}", 2, "subcategory_1", f"subcategory in {y}"


def period_role(c, years_only, content):
    """a question asked for a period (two years in its name, e.g. Changement_2021-2025, type_change1_2016_2024):
    the change itself (label, year, driver, attributes) or the land after it (class, canopy cover)."""
    p = plain(re.sub(r"(19|20)\d\d", " ", c))
    if content.get("yes_no"):
        return "-", "", "", "not used (yes / no: a change was recorded)"
    if re.search(r"cause gain|gain cause|gain origin|origin gain|planted or natural|natural or planted", p):
        return "PERIOD", 3, "gain_origin", "how the gain happened (planted / natural)"
    if re.search(r"moteur|driver|cause|motor", p):
        return "PERIOD", 4, "driver", "driver of the change in the period"
    if years_only or re.search(r"\b(annee|year|yr|ano)\b", p):
        return "PERIOD", "", "(years)", "year of the change in the period"
    if re.search(r"\b(after|apres|despues)\b", p):
        if re.search(CANOPY_WORDS, p):
            if content.get("numeric"):
                return "PERIOD-to", 3, "canopy_cover", "canopy cover after the change"
            return "PERIOD-to", 3, "canopy_class", "canopy cover class after the change"
        if content.get("ipcc", 0) >= 0.8:
            return "PERIOD-to", 1, "ipcc_category", "land use after the change"
        return "PERIOD-to", 2, "subcategory_1", "class after the change"
    if re.search(r"type loss|loss type", p):
        return "PERIOD", 3, "loss_type", "kind of loss (e.g. true loss, cyclical)"
    if re.search(r"\d ?yrs|duration|how long", p):
        return "PERIOD", 3, "nonforest_duration_before", "how long the land was non-forest before"
    return "PERIOD", 4, "change_label", "the change in the period, as labelled by the interpreter"


PERIOD_RE = re.compile(r"(\d)?[_\-\s]?((?:19|20)\d\d)[_\-\s]+((?:19|20)\d\d)\s*$")


def period_of(c):
    """(start year, end year, change number or None, tag) of a period question: 'fcc_after_deg1_2016_2024' -> (2016, 2024, 1, 'deg')."""
    m = PERIOD_RE.search(c)
    if not m:
        return None, None, None, ""
    head = c[:m.start()].lower()
    tag = re.split(r"[_\s-]", head)[-1] if head else ""
    return int(m.group(2)), int(m.group(3)), int(m.group(1)) if m.group(1) else None, tag


# ----------------------------------------------------------------------------------------------- Step 3: rebuild

CARD_SUFFIX = re.compile(r"[-\s](L|SDFD|DWC|G|SDFR|E|NFLUC|NFSDC)(\d+)(\s*\(\d+\))?(\.\d+)?\s*$")


def prepare_roles(survey):
    """group the roles once, so that each plot is read quickly."""
    all_recs = [r for r in survey["roles"].to_dict("records") if r["record"] != "-"]
    # columns the tool does not know: not in the table; written with their plot id in 'unrecognised data'
    survey["unknown_cols"] = [r["export column"] for r in all_recs if str(r["note"]).startswith("not known")]
    recs = [r for r in all_recs if not str(r["note"]).startswith("not known")]
    survey["P_cols"] = [(r["export column"], r["variable"]) for r in recs if r["record"] == "P"]
    survey["S0_cols"] = [(r["export column"], r["level"], r["variable"]) for r in recs if r["record"] == "S0"]
    survey["blocks"] = defaultdict(list)          # CE: k -> [(column, record, level, variable)]
    survey["cards"] = defaultdict(list)           # CEO: (kind, k) -> [(column, record, level, variable)]
    survey["years"] = defaultdict(list)           # annual: year -> [(column, level, variable)]
    for r in recs:
        c = r["export column"]
        m = re.match(r"^(event|change)\[(\d+)\]_", c)
        if m and survey["tool"] == "CE":
            survey["blocks"][int(m.group(2))].append((c, r["record"], r["level"], r["variable"]))
        m = CARD_SUFFIX.search(c)
        if m and survey["tool"] == "CEO" and r["record"] in ("Sk", "Ck"):
            survey["cards"][(m.group(1), int(m.group(2)))].append((c, r["record"], r["level"], r["variable"]))
        if str(r["record"]).startswith("S@"):
            survey["years"][int(r["record"][2:])].append((c, r["level"], r["variable"]))
    survey["period_cols"] = []
    for r in recs:
        if r["record"] in ("PERIOD", "PERIOD-to"):
            y1, y2, k, tag = period_of(r["export column"])
            survey["period_cols"].append({"col": r["export column"], "level": r["level"], "variable": r["variable"],
                                          "side": "to" if r["record"] == "PERIOD-to" else "change",
                                          "y1": y1, "y2": y2, "k": k, "tag": tag})
    survey["numbered_periods"] = any(p["k"] for p in survey["period_cols"])
    if survey["numbered_periods"]:
        whole = [p["col"] for p in survey["period_cols"] if not p["k"] and p["variable"] == "change_label"]
        survey["roles"].loc[survey["roles"]["export column"].isin(whole), "note"] = \
            "one label for the whole period (a summary, not a change): the numbered changes are used"
    # a yes / no question per change of a period, answered in words (e.g. change1_2016_2024 = yes): it must agree with what is recorded for that change
    survey["period_flags"] = [(r["export column"],) + period_of(r["export column"])[:3] for r in survey["roles"].to_dict("records")
                              if str(r["note"]).startswith("not used (yes / no: a change was recorded)") and period_of(r["export column"])[2]
                              and not survey["data"][r["export column"]].dropna().astype(str).str.fullmatch(r"\s*-?\d+([.,]\d+)?\s*").all()]
    # the classes the survey uses at its dates: an answer in these classes can be compared with the land at a date
    survey["date_classes"] = {str(v).strip() for cols in survey["years"].values() for c, lvl, var in cols
                              if var == "subcategory_1" for v in survey["data"][c] if given(v)}
    survey["class_pairs"] = class_pairs(survey)
    cols = list(survey["data"].columns)
    survey["card_cols"] = [c for c in cols if re.match(r"^(1st|2nd|3rd|\d+th) change", c)]
    survey["lu_end_col"] = next((c for c in cols if re.match(r"^LU in ", c)), None)
    return survey


def class_pairs(survey):
    """the survey's own legend: which IPCC category each subcategory belongs to, from the dates where both were asked
    (e.g. Kenya: lc2016 'Montane - Moderate' with ipcc2016 'FL'). Used only for a state where the survey recorded the
    subcategory but not the category (the land after a change between two dates)."""
    pairs = defaultdict(Counter)
    data = survey["data"]
    for y, cols in survey.get("years", {}).items():
        l1 = next((c for c, lvl, v in cols if v == "ipcc_category"), None)
        l2 = next((c for c, lvl, v in cols if v == "subcategory_1"), None)
        if l1 and l2:
            for a, b in zip(data[l2], data[l1]):
                if given(a) and given(b):
                    pairs[str(a).strip()][ipcc_category(b) or str(b).strip()] += 1
    out = {}
    for sub, cnt in pairs.items():
        cat, n = cnt.most_common(1)[0]
        if n >= 0.95 * sum(cnt.values()):
            out[sub] = cat
    return out


def reframe_plots(survey):
    """Step 3: place every answer of every plot in the structure: the plot (P), states (S), changes (C).
    Only what was collected; nothing is deduced. Returns (plots, findings)."""
    prepare_roles(survey)
    reader = {"CE": place_ce_ipcc_form if survey["design"] == "ipcc_form" else place_ce,
              "CEO": place_ceo_cards if survey["design"] == "cards" else place_snapshots}[survey["tool"]]
    plots, findings = [], []
    aside = set(map(str, survey.get("set_aside", ())))
    id_col = survey["id_col"]
    for row in survey["data"].to_dict("records"):
        if not given(row.get(id_col)) or str(row.get(id_col)) in aside:
            continue                                      # no plot id, or set aside by the quality check
        plot = reader(row, survey)
        plots.append(plot)
        findings += check_plot(plot, survey.get("end_year")) + plot.pop("findings", [])
    return plots, findings


def new_plot(row, survey, id_col, point_col=None):
    plot = {"plot_id": row[id_col], "point_id": row[point_col] if point_col and given(row.get(point_col)) else "",
            "info": [], "states": {}, "changes": {}}
    for c, variable in survey["P_cols"]:
        if given(row.get(c)):
            plot["info"].append((0, variable, value_of(variable, row[c]), c))
    if survey["tool"] == "CE" and given(row.get("actively_saved_on_year")):
        d = "-".join(str(row.get(f"actively_saved_on_{p}", "")).split(".")[0].zfill(2) for p in ("year", "month", "day"))
        plot["info"].append((0, "date", d, "actively_saved_on_year/month/day"))
    return plot


YES_NO_VARIABLES = {"trees_present", "native_species", "managed", "temporarily_unstocked", "canopy_cover_decrease",
                    "canopy_cover_increase", "extends_previous_change", "temporary_loss", "within_buffer",
                    "forest_before_nonforest", "confidence", "commercial_use"}
EXTRA_YES_NO = {"commercial": "yes", "non commercial": "no", "cyclical": "yes", "true loss": "no"}   # v14: the kind of loss


def value_of(variable, value, label=None):
    """the value as written in the file."""
    v = str(value).strip()
    if variable == "ipcc_category":
        return ipcc_category(v, label) or v
    if variable in YES_NO_VARIABLES:
        yn = yes_no(v) or EXTRA_YES_NO.get(plain(v))
        if yn:
            return yn
    if variable == "driver_origin":
        yn = yes_no(v)
        return "anthropogenic" if yn == "yes" else "natural" if yn == "no" else v
    if variable == "date":
        return v[:10]
    return v[:-2] if re.match(r"^-?\d+\.0$", v) else v


def add_rows(target, row, columns):
    """add (level, variable, value, source) for the given (column, level, variable)."""
    for c, level, variable in columns:
        if given(row.get(c)):
            label = row.get(c + "_label") if given(row.get(c + "_label")) else None
            target.append((level, variable, value_of(variable, row[c], label), c))


def year_of(code, label=None):
    """(year_from, year_to) of a change year answer: '2018' -> (2018, 2018); 'before 2015' -> (None, 2014); a code -> (None, None)."""
    for v in (label, code):
        if not given(v):
            continue
        t = str(v).strip()
        m = re.match(r"^(?:before|avant|antes de)\s+((19|20)\d\d)$", t, re.I)
        if m:
            return None, int(m.group(1)) - 1
        m = re.match(r"^((19|20)\d\d)(\.0)?$", t)
        if m:
            return int(m.group(1)), int(m.group(1))
    return None, None


def exact_year(ch):
    """the year of a change when it is known exactly (not a range, not a code)."""
    return ch["year_to"] if ch.get("year_to") and ch.get("year_from") == ch.get("year_to") else None


def place_backward(plot, survey, n):
    """States and changes as asked (change 1 = the most recent). A state gets the years the changes around it give:
    it starts in the year of the change that led to it and ends the year before the next change. The state in the
    end year ends in the end year; it starts in the end year only when no change was recorded.
    from / to: the state the survey asks before the change, and the one after it."""
    end = survey["end_year"]
    for k in range(n + 1):
        st = plot["states"].setdefault(k, {"rows": []})
        newer = plot["changes"].get(k)                   # the change that ends this state (k >= 1)
        older = plot["changes"].get(k + 1)               # the change that led to this state
        if k == 0:
            st["year_to"] = end
        else:
            y = newer["year_from"] or newer["year_to"]
            st["year_to"] = y - 1 if y else None
        st["year_from"] = exact_year(older) if older else (end if k == 0 else None)
        st["order"] = n - k
    for k in range(1, n + 1):
        ch = plot["changes"][k]
        ch["from"], ch["to"], ch["order"] = k, k - 1, n - k + 1
        ch["link"] = "as asked: the land before this change, and after it"
    return plot


def place_ce(row, survey):
    """Collect Earth: lu_t2 / sub_t2 = the end year; event[k] or change[k] = change k, 1 = most recent."""
    plot = new_plot(row, survey, survey["id_col"])
    add_rows(plot["states"].setdefault(0, {"rows": []})["rows"], row, survey["S0_cols"])
    block = survey["design"]
    ks = [k for k in sorted(survey["blocks"]) if given(row.get(f"{block}[{k}]_change_yr"))]
    for n, k in enumerate(ks, start=1):
        yc = f"{block}[{k}]_change_yr"
        y_from, y_to = year_of(row[yc], row.get(yc + "_label"))
        ch = plot["changes"][n] = {"rows": [], "year_from": y_from, "year_to": y_to}
        st = plot["states"].setdefault(n, {"rows": []})["rows"]
        if not y_to:
            ch["rows"].append((4, "year_code", value_of("", row[yc]), yc))
        for c, record, level, variable in survey["blocks"][k]:
            if variable == "(years)" or not given(row.get(c)):
                continue
            if variable == "driver" and plain(row[c]) in ("none", "aucune", "ninguna", "nenhuma", "none or regrowth"):
                continue
            label = row.get(c + "_label") if given(row.get(c + "_label")) else None
            (st if record == "Sk" else ch["rows"]).append((level, variable, value_of(variable, row[c], label), c))
        add_kind_from_question(ch)
    return place_backward(plot, survey, len(ks))


KIND_WORDS = (("regrowth", "regrowth"), ("recovery", "recovery"), ("gain", "gain"), ("disturbance", "disturbance"),
              ("degradation", "degradation"), ("deforestation", "deforestation"))


def add_kind_from_question(ch):
    """the kind of a change, as collected: when the survey has no 'kind of change' answer, the question answered says
    it (e.g. Collect Earth event[1]_disturbance -> 'disturbance', event[1]_driver_regrowth -> 'regrowth')."""
    if any(v == "change_label" for _, v, _, _ in ch["rows"]):
        return
    for _, v, _, src in ch["rows"]:
        if v == "driver":
            word = next((w for key, w in KIND_WORDS if key in plain(src)), None)
            if word:
                ch["rows"].append((4, "change_label", word, f"{src} (the question answered)"))
                return


CARD_FAMILY = {"forest disturbance": "FD", "forest recovery": "FR", "nf change": "NFC"}


def place_ceo_cards(row, survey):
    """CEO change cards: 'LU in <year>' = the end year; '1st change', '2nd change' = change 1, 2 (1 = most recent)."""
    plot = new_plot(row, survey, survey["id_col"], survey.get("point_col"))
    lu_end = row.get(survey["lu_end_col"]) if survey["lu_end_col"] else None
    add_rows(plot["states"].setdefault(0, {"rows": []})["rows"], row,
             [(c, lvl, v) for c, lvl, v in survey["S0_cols"] if v != "subcategory_1" or same_category(c, lu_end)])
    n = 0
    for card_col in survey["card_cols"]:
        answer = str(row.get(card_col) or "").strip()
        if not given(row.get(card_col)) or plain(answer).startswith("no change"):
            break
        fam = next((f for f in CARD_FAMILY if plain(answer).startswith(f)), None)
        if fam is None:
            break
        k = int(re.match(r"^(\d+)", card_col).group(1))
        n += 1
        prefix = CARD_FAMILY[fam]
        kind_answer, sub_kind = "", None
        for c in survey["data"].columns:
            if plain(c).startswith(fam) and re.search(rf"\D{k}\s*$", c) and given(row.get(c)):
                m = re.search(r"-(L|SDFD|DWC|G|SDFR|E|NFLUC|NFSDC)\d+\s*$", str(row[c]).strip())
                if m:
                    kind_answer, sub_kind = str(row[c]).strip(), m.group(1)
        ch = plot["changes"][n] = {"rows": [(4, "change_label", " / ".join(x for x in (answer, kind_answer) if x), card_col)]}
        ycol = f"Year-{prefix}{k}"
        ch["year_from"], ch["year_to"] = year_of(row.get(ycol))
        if given(row.get(ycol)) and not ch["year_to"]:
            ch["rows"].append((4, "year_code", value_of("", row[ycol]), ycol))
        st = plot["states"].setdefault(n, {"rows": []})["rows"]
        if sub_kind in ("L", "SDFD", "SDFR"):     # these cards are asked only when the land before was forest
            st.append((1, "ipcc_category", "Forest land", f"{kind_answer} (card asked for forest before the change)"))
        lu_before = next((row[c] for c, rec, lvl, v in survey["cards"][(sub_kind, k)] if v == "ipcc_category" and given(row.get(c))), None)
        for c, record, level, variable in survey["cards"][(sub_kind, k)]:
            if not given(row.get(c)) or variable == "(years)":
                continue
            if variable == "subcategory_1" and "subdivision-" in c.lower() and not same_category(c, lu_before):
                continue
            (st if record == "Sk" else ch["rows"]).append((level, variable, value_of(variable, row[c]), c))
    return place_backward(plot, survey, n)


def same_category(column, land_use):
    """a '<Category> subdivision' column belongs to a state only if the state's land use is that category."""
    cat = re.match(r"^(Forest|Cropland|Grassland|Wetland|Settlement|Other land)", column, re.I)
    return not cat or land_use is None or ipcc_category(land_use) == ipcc_category(cat.group(1))


STABLE = {"stable", "no change", "pas de changement", "sin cambio", "stable forest", "stable non forest"}


def place_snapshots(row, survey):
    """States at several dates (snapshots), e.g. a class for every year (RCI 2021-2025) or every few years (Kenya
    1990-2000-2008-2016-2024), with the changes the survey recorded between them:
      - numbered changes in a period (Kenya type_change1_2016_2024, yr_change1_..., lc_after_change1_...): each has its
        year, and the land after it becomes a state of its own (from its year until the next change or date);
      - changes asked once for a period (RCI Changement_2021-2025, Annee_Deforestation_..., Moteur_...): linked to
        the dates before and after the year of the change.
    No change is deduced from two classes being different."""
    plot = new_plot(row, survey, survey["id_col"], survey.get("point_col"))
    plot["findings"] = []
    snaps = {}
    for y in sorted(survey["years"]):
        rows = []
        add_rows(rows, row, survey["years"][y])
        if rows:
            snaps[y] = {"rows": rows, "year_from": y, "year_to": y, "snap": True}
    states = list(snaps.values())
    changes = []
    pcols = survey["period_cols"]
    numbered = defaultdict(list)
    for p in pcols:
        if p["k"]:
            numbered[(p["y1"], p["y2"], p["k"])].append(p)
    # 1. numbered changes in a period
    by_period = defaultdict(list)
    partial = set()
    said_of = {}                                          # (y1, y2, k) -> (type of change, year): to find an answer given again
    for key, cols in numbered.items():
        lab_ = next((code(row, p["col"]) for p in cols if p["variable"] == "change_label" and given(row.get(p["col"]))), "")
        yr_ = next((year_of(row[p["col"]])[1] for p in cols if p["variable"] == "(years)" and given(row.get(p["col"]))), None)
        said_of[key] = (lab_, yr_)

    def given_again(p, here):
        """the same answer given for a change of its kind in another period: ', the same in 2016–2024 (Degradation 2018)'."""
        kind = kind_of(p["tag"])
        for key, cols in numbered.items():
            lab_, yr_ = said_of[key]
            if key == here or not lab_ or kind_of(lab_) != kind:
                continue
            if any(q["variable"] == p["variable"] and q["side"] == p["side"] and kind_of(q["tag"]) == kind
                   and code(row, q["col"]) == code(row, p["col"]) for q in cols):
                return f", the same in {key[0]}–{key[1]} ({lab_}{' ' + str(yr_) if yr_ else ''})"
        return ""
    for y in snaps:                                       # canopy cover at the dates: a percentage
        snaps[y]["rows"] += canopy_range_notes(row, [c for c, lvl, var in survey["years"][y] if var == "canopy_cover"], "")
    for (y1, y2, k), cols in sorted(numbered.items()):
        answered = [p for p in cols if given(row.get(p["col"]))]
        tags = defaultdict(list)
        for p in answered:
            tags[p["tag"]].append(p)
        # a group of answers about one kind of change holding only zeros = not answered (e.g. fcc_after_refo1 = 0)
        answered = [p for t, ps in tags.items() if not all(str(row[q["col"]]).strip() in ("0", "0.0") for q in ps) for p in ps]
        y = next((year_of(row[p["col"]])[1] for p in answered if p["variable"] == "(years)" and year_of(row[p["col"]])[1]), None)
        label = next((code(row, p["col"]) for p in answered if p["variable"] == "change_label"), "")
        other_kind = [p for p in answered if label and kind_of(p["tag"]) and kind_of(p["tag"]) != kind_of(label)]
        answered = [p for p in answered if p not in other_kind]   # an answer of another kind of change is not used
        kind_notes = [(4, "note", f"{y1}–{y2}: {p['col']} = {code(row, p['col'])}, but the change is {label}{given_again(p, (y1, y2, k))}", "")
                      for p in other_kind if code(row, p["col"]) not in ("0", "0.0")
                      or all(code(row, q["col"]) in ("0", "0.0") for q in other_kind)]
        if y and not (y1 <= y <= y2):
            ycol = next(p["col"] for p in answered if p["variable"] == "(years)")
            kind_notes.append((4, "note", f"{y1}–{y2}: {ycol} = {y}, outside the period", ""))
        kind_notes += canopy_range_notes(row, [p["col"] for p in answered if p["variable"] == "canopy_cover"], f"{y1}–{y2}: ")
        to_rows, ch_rows = [], []
        for p in answered:
            v = p["variable"]
            if v == "(years)":
                continue
            if p["side"] == "to":
                if any(x[1] == v for x in to_rows):       # the same thing asked again for another kind of change
                    continue
                to_rows.append((p["level"], v, value_of(v, row[p["col"]]), p["col"]))
            elif not (v == "driver" and plain(row[p["col"]]) in NO_ANSWER):
                ch_rows.append((p["level"], v, value_of(v, row[p["col"]]), p["col"]))
        if not y and not to_rows:
            continue                                      # nothing recorded (e.g. 'stable_natural_FL' without a year)
        if not y and not any(x[1] == "change_label" for x in ch_rows) and not any(is_land(x) for x in to_rows):
            # only some answers (a driver, a canopy): no change is made, the answers are reported as they are
            shown = [p for p in answered if p["variable"] != "(years)" and code(row, p["col"]) not in ("0", "0.0")] or \
                    [p for p in answered if p["variable"] != "(years)"]
            said = ", ".join(f"{p['col']} = {code(row, p['col'])}" for p in shown)
            flag = next((col for col, f1, f2, kk in survey.get("period_flags", []) if (f1, f2, kk) == (y1, y2, k)), None)
            flag_said = f"but {flag} = {code(row, flag)}" if flag and given(row.get(flag)) else "with no type, year or land after"
            flag_said += next((a for a in (given_again(p, (y1, y2, k)) for p in shown) if a), "")
            at = snaps.get(y2) or snaps.get(y1)
            if at is not None and said:
                at["rows"].append((4, "note", f"{y1}–{y2}: {said}, {flag_said}", ""))
            partial.add((y1, y2, k))
            continue
        if to_rows and not any(x[1] == "ipcc_category" for x in to_rows):
            sub = next((x[2] for x in to_rows if x[1] == "subcategory_1"), None)
            if sub in survey["class_pairs"]:
                to_rows.append((1, "ipcc_category", survey["class_pairs"][sub],
                                f"the survey's own classes: '{sub}' is {survey['class_pairs'][sub]} at the dates"))
        by_period[(y1, y2)].append({"rows": ch_rows + kind_notes, "year_from": y or y1 + 1, "year_to": y or y2, "k": k, "to_rows": to_rows})
    for col, f1, f2, k in survey.get("period_flags", []):
        flag = yes_no(row.get(col)) if given(row.get(col)) else None
        recorded = [ch for ch in by_period.get((f1, f2), []) if ch["k"] == k]
        if flag == "no" and recorded:
            recorded[0]["rows"].append((4, "note", f"{col} = no, but a change is recorded", ""))
        elif flag == "yes" and not recorded and (f1, f2, k) not in partial and snaps.get(f2) is not None:
            snaps[f2]["rows"].append((4, "note", f"{f1}–{f2}: {col} = yes, with no type, year or land after", ""))
    for (y1, y2), chs in sorted(by_period.items()):
        chs.sort(key=lambda ch: (ch["year_to"], ch["k"]))
        prev = snaps.get(y1)
        for i, ch in enumerate(chs):
            ch["from_state"] = prev
            ch["rows"] += canopy_direction_note(ch, prev, y1, y2)
            if ch["to_rows"] and exact_year(ch) == y2 and snaps.get(y2) is not None:
                # the change happened in the year of the date: the land after it is the land at that date
                snap = snaps[y2]
                sub_a = next((x[2] for x in ch["to_rows"] if x[1] == "subcategory_1"), None)
                sub_b = next((x[2] for x in snap["rows"] if x[1] == "subcategory_1"), None)
                if sub_a and sub_b and sub_a != sub_b:
                    ch["rows"].append(land_after_note(survey, sub_a, sub_b, y2))
                snap["rows"] += [x for x in ch["to_rows"] if not any(r[1] == x[1] for r in snap["rows"])]
                ch["to_state"] = prev = snap
            elif ch["to_rows"]:
                nxt = next((c2["year_from"] for c2 in chs[i + 1:] if exact_year(c2)), None)
                y = exact_year(ch)
                st = {"rows": ch["to_rows"], "year_from": y,
                      "year_to": (nxt - 1) if nxt else (y2 - 1 if y and y < y2 else y), "snap": False}
                states.append(st)
                ch["to_state"] = prev = st
            else:
                ch["to_state"] = None
            changes.append(ch)
        for i, ch in enumerate(chs):                      # no land recorded after a change: the next state in time
            if ch["to_state"] is None:
                ch["to_state"] = next((c2["to_state"] for c2 in chs[i + 1:] if c2["to_state"]), snaps.get(y2))
        last = chs[-1]["to_state"]
        if last is not None and snaps.get(y2) is not None and last is not snaps[y2] and not last.get("snap"):
            # no change recorded after the last one: from its year, the land is the land recorded at the date ending the period
            sub_a = next((x[2] for x in last["rows"] if x[1] == "subcategory_1"), None)
            sub_b = next((x[2] for x in snaps[y2]["rows"] if x[1] == "subcategory_1"), None)
            if sub_a and sub_b and sub_a != sub_b:
                maker = next(ch for ch in reversed(chs) if ch["to_state"] is last)
                maker["rows"].append(land_after_note(survey, sub_a, sub_b, y2))
            land = [x for x in snaps[y2]["rows"] if is_land(x)]
            if land:
                last["rows"] = [x for x in last["rows"] if not is_land(x)] + land
    # 2. changes asked once for a period (no number)
    unnumbered = [p for p in pcols if not p["k"]]
    labels_ = [(p, str(row[p["col"]]).strip()) for p in unnumbered if p["variable"] == "change_label" and given(row.get(p["col"]))]
    drivers = [(p, str(row[p["col"]]).strip()) for p in unnumbered if p["variable"] == "driver" and given(row.get(p["col"]))]
    years = [(p, year_of(row[p["col"]])[1]) for p in unnumbered if p["variable"] == "(years)" and given(row.get(p["col"]))]
    ordered = sorted(snaps.values(), key=lambda st: st["year_to"])
    for p, lab_ in labels_:
        if plain(lab_) in STABLE or survey.get("numbered_periods"):
            # 'no change' in the period, or a label for the whole period next to numbered changes (Kenya type_change_1990_2024)
            plot["info"].append((0, "period_label", lab_, p["col"], p["y1"], p["y2"]))
            continue
        if years and years[0][1]:
            y_from = y_to = years[0][1]
            before = [st for st in ordered if st["year_to"] < y_to]
            after = [st for st in ordered if st["year_to"] >= y_to]
        else:                                             # no year given: the change is somewhere in the period
            y_from, y_to = p["y1"], p["y2"]
            before = [st for st in ordered if y_from and st["year_to"] <= y_from]
            after = [st for st in ordered if y_to and st["year_to"] >= y_to]
        changes.append({"rows": [(4, "change_label", lab_, p["col"])] + [(4, "driver", d, dp["col"]) for dp, d in drivers],
                        "year_from": y_from, "year_to": y_to, "k": 0,
                        "from_state": before[-1] if before else None, "to_state": after[0] if after else None,
                        "link": "by date: the land reported before and after the year of the change"})
    # numbering: states and changes in time order
    states.sort(key=lambda st: (st["year_to"] if st["year_to"] is not None else -1, 1 if st["snap"] else 0))
    for i, st in enumerate(states):
        st["order"] = i
        plot["states"][i] = st
    index = {id(st): i for i, st in enumerate(states)}
    changes.sort(key=lambda ch: (ch["year_to"] or 0, ch["k"]))
    for i, ch in enumerate(changes, start=1):
        ch["order"] = i
        ch["from"] = index.get(id(ch.pop("from_state")))
        ch["to"] = index.get(id(ch.pop("to_state")))
        ch.pop("to_rows", None)
        ch.setdefault("link", "as asked: the land before this change, and after it")
        plot["changes"][i] = ch
    return plot


KINDS = {"degradation": r"deg", "loss": r"def|loss|clear", "gain": r"gain|refo|affo|enhanc|regrow|regen"}
KIND_NAMES = {"degradation": "Degradation", "loss": "Loss (deforestation)", "gain": "Gain (enhancement, reforestation)"}


def kind_of(text):
    """the kind of change a column or a type of change is about, from its words: degradation, loss, gain; None = not said."""
    for kind, words in KINDS.items():
        if re.search(words, str(text), re.I):
            return kind
    return None


def canopy_of(rows):
    """the canopy cover (a number) in the rows of a state or a change, or None."""
    v = next((x[2] for x in rows if x[1] == "canopy_cover"), None)
    try:
        return float(str(v).replace(",", "."))
    except (TypeError, ValueError):
        return None


def canopy_range_notes(row, columns, prefix):
    """a canopy cover is a percentage: a value outside 0-100 is a typing error."""
    out = []
    for c in columns:
        try:
            v = float(code(row, c).replace(",", "."))
        except ValueError:
            continue
        if not 0 <= v <= 100:
            out.append((4, "note", f"{prefix}{c} = {code(row, c)}: canopy cover outside 0–100", ""))
    return out


def canopy_direction_note(ch, before_state, y1, y2):
    """a degradation lowers the canopy, an enhancement raises it: compared with the canopy just before the change
    (the date that starts the period, or the change before it in the same period)."""
    label = next((x[2] for x in ch["rows"] if x[1] == "change_label"), "")
    kind = kind_of(label)
    if kind not in ("degradation", "gain") or before_state is None:
        return []
    before, after = canopy_of(before_state["rows"]), canopy_of(ch.get("to_rows") or [])
    if before is None or after is None or not after:
        return []
    if (kind == "degradation" and after >= before) or (kind == "gain" and after <= before):
        word = "not lower" if kind == "degradation" else "not higher"
        year = f" in {ch['year_to']}" if ch.get("year_to") == ch.get("year_from") else ""
        return [(4, "note", f"{y1}–{y2}: {label}{year}, canopy {before:g} → {after:g} ({word})", "")]
    return []


def is_land(x):
    """a row of a state that holds the land: the IPCC category or a subdivision."""
    return x[1] == "ipcc_category" or x[1] in ("subcategory_1", "subcategory_2", "subcategory_3")


def land_after_note(survey, after, at_date, y2):
    """the answer 'land after the change' and the land recorded at the date ending the period, with no change between them.
    In the classes of the dates, they must be the same; in other (broader) classes they cannot be compared: kept as a note."""
    if after in survey.get("date_classes", ()):
        return (4, "note", f"land after the change: {after}, but land in {y2}: {at_date}", "")
    return (4, "note", f"land after the change recorded as {after}; land in {y2}: {at_date}", "")


def check_plot(plot, end_year):
    """only what can be checked without interpreting: years in the right order."""
    out = []
    # the changes as the survey asked them (backward: newest first; dates: oldest first) must follow each other in time
    asked = [ch["year_to"] for k, ch in sorted(plot["changes"].items()) if ch.get("year_to")]
    by_time = [ch["year_to"] for ch in sorted(plot["changes"].values(), key=lambda c: c.get("order", 0)) if ch.get("year_to")]
    if by_time != sorted(by_time):
        out.append({"plot": plot["plot_id"], "finding": f"change years not in time order (as asked): {asked}"})
    out += [{"plot": plot["plot_id"], "finding": f"change {k}: year {ch['year_to']} is after the end year {end_year}"}
            for k, ch in plot["changes"].items() if end_year and ch.get("year_to") and ch["year_to"] > end_year]
    out += [{"plot": plot["plot_id"], "finding": f"change in {ch.get('year_to') or '?'}: no state "
             + ("before" if ch.get("from") is None else "after") + " it (the year is outside the dates reported)"}
            for k, ch in plot["changes"].items() if "from" in ch and (ch.get("from") is None or ch.get("to") is None)]
    return out


# ----------------------------------------------------------------------------------------------- Step 4: write


def quality_check(survey, advanced=False, set_aside=True):
    """Step 3: the quality check of the export, before anything is placed.
    basic (always): plot id missing or repeated, the end year known, value types (a text among numbers, a year that is
    not a year); advanced (optional): two plot ids at the same place, Collect Earth plots not saved.
    Plots with a problem to fix are set aside (not in the table) when set_aside is True; they are listed.
    Returns (report, set_aside_ids, stop_message)."""
    data, id_col = survey["data"], survey["id_col"]
    point = survey.get("point_col")
    out = []

    def note(rule, level, plot, detail):
        out.append({"check": rule, "level": level, "plot": plot, "detail": detail})

    stop = ""
    missing = data[~data[id_col].map(given)]
    for i in missing.index:
        note("1 plot id", "⛔ to fix", "", f"row {i + 2} of the export has no plot id: it cannot be placed")
    keys = [id_col] + ([point] if point else [])
    dup = data[data[id_col].map(given) & data.duplicated(keys, keep=False)]
    for pid, g in dup.groupby(id_col):
        note("1 plot id", "⛔ to fix", pid, f"the plot id appears {len(g)} times (rows {', '.join(str(i + 2) for i in g.index[:6])})")
    if survey.get("id_note"):
        note("1 plot id", "ℹ note", "", survey["id_note"])
    if not survey.get("end_year"):
        stop = "the end year is not known: write it in end_year (a Collect Earth export does not contain it)"
        note("2 end year", "⛔ to fix", "", stop)
    roles = survey["roles"]
    used = roles[(roles["record"] != "-") & ~roles["note"].astype(str).str.startswith("not known")]
    for r in used.to_dict("records"):
        c = r["export column"]
        vals = data[c][data[c].map(given)].astype(str).str.strip()
        if not len(vals):
            continue
        if r["variable"] == "(years)":
            bad = vals[~vals.str.fullmatch(r"((19|20)\d\d)(\.0)?|\d(\.0)?|(before|avant|antes de)\s+(19|20)\d\d", case=False)]
            for i, v in bad.head(50).items():
                note("3 value types", "⚠ to check", data.at[i, id_col], f"{c}: '{v}' is not a year")
            continue
        number = vals.str.fullmatch(r"-?\d+([.,]\d+)?")
        if number.mean() >= 0.9 and not number.all():
            for i, v in vals[~number].head(50).items():
                note("3 value types", "⚠ to check", data.at[i, id_col], f"{c}: '{v}' among numbers")
    if advanced:
        xy = [c for c, v in survey.get("P_cols", []) if v in ("x", "y")]
        if len(xy) == 2:
            loc = data[xy].apply(pd.to_numeric, errors="coerce").round(6)
            loc = loc[loc.notna().all(axis=1)]
            key = loc[xy[0]].astype(str) + " " + loc[xy[1]].astype(str)
            ids = data.loc[key.index, id_col]
            for k, g in ids.groupby(key):
                if g.nunique() > 1:
                    note("4 same location", "⚠ to check", " / ".join(map(str, g.unique()[:4])),
                         f"{g.nunique()} plot ids at the same place ({k})")
        if "actively_saved" in data.columns:
            for i in data.index[data["actively_saved"].astype(str).str.lower().isin(["false", "0", "no"])]:
                note("5 saved plots", "⚠ to check", data.at[i, id_col], "plot not saved by the interpreter (actively_saved = false)")
    report = pd.DataFrame(out, columns=["check", "level", "plot", "detail"])
    aside = set(report.loc[report["level"].str.startswith("⛔") & (report["plot"] != ""), "plot"]) if set_aside else set()
    return report, aside, stop


def quality_text(report, aside, advanced):
    lines = [f"Quality check ({'basic + advanced' if advanced else 'basic'}):"]
    if not len(report):
        return lines[0] + " no issue found."
    for (chk, lvl), g in report.groupby(["check", "level"], sort=True):
        lines.append(f"  {lvl:12} {chk:18} {len(g):6d}   e.g. {g['detail'].iloc[0]}")
    if aside:
        lines.append(f"Set aside (not in the table, listed in the workbook, sheet 'survey description'): {len(aside)} plot(s)")
    return "\n".join(lines)


# ----------------------------------------------------------------------------------------------- helpers for the notebook

def describe_text(survey):
    """Step 2 on screen: what the survey contains, and what to look at."""
    roles = survey["roles"]
    used = roles[roles["record"] != "-"]
    lines = [f"Tool: {survey['tool']}    Design: {survey['design_text']}",
             f"Plots: {survey['plots']}    End year: {survey['end_year']}"]
    lines.append(f"Plot id: {survey['id_col']}" + (f" - {survey['id_note']}" if survey.get("id_note") else ""))
    if survey.get("layout_note"):
        lines.append("Note: " + survey["layout_note"])
    kind = used["record"].astype(str).map(lambda r: "P" if r == "P" else "S0" if r == "S0" else "PERIOD" if r == "PERIOD"
                                           else "S@" if r.startswith("S@") else "Sk" if r.startswith("S") else "Ck")
    for rec, name in (("P", "plot information"), ("S0", "land in the end year"), ("Sk", "land before a change"),
                      ("Ck", "a change"), ("PERIOD", "a period (label, year, driver)")):
        n = (kind == rec).sum()
        if n:
            lines.append(f"  {n:4d} columns -> {name}")
    n_years = used["record"].astype(str).str.startswith("S@").sum()
    if n_years:
        lines.append(f"  {n_years:4d} columns -> the land at a date ({used['record'][used['record'].astype(str).str.startswith('S@')].nunique()} dates)")
    lines.append(f"  {(roles['record'] == '-').sum():4d} columns not used (identifiers, technical or calculated)")
    extra = roles[roles["note"].str.startswith("not known") & (roles["plots with an answer"] > 0)]
    if len(extra):
        lines.append("Columns the tool does not know (not in the table, they stay in raw_data): "
                     + ", ".join(extra["export column"].head(12)) + (" ..." if len(extra) > 12 else ""))
    tips = []
    if not survey["end_year"]:
        tips.append("the end year is not known: write it in end_year (a Collect Earth export does not contain it).")
    data = survey["data"]
    placeholders = [c for c in data.columns if data[c].astype(str).str.contains(r"<[^>]+>", regex=True).any()]
    if placeholders or any("<" in c for c in data.columns):
        tips.append("template placeholders found (e.g. <END YEAR>, <Forest type I>): the survey was not fully customised.")
    codes = [c for c in data.columns if re.match(r"^Year-", c) and data[c].dropna().astype(str).str.fullmatch(r"\d(\.0)?").any()]
    if codes:
        tips.append("change years are codes (1, 2, 3...), not years: they are kept as 'year_code' until the survey's list of years is given.")
    for t in tips:
        lines.append("Suggestion: " + t)
    return "\n".join(lines)


# ----------------------------------------------------------------------------------------------- v2: sheet choice

SHEET_NAME_HINT = re.compile(r"ce[_ ]?data|ceo|raw|data|export|collect", re.I)


def looks_like_export(df):
    """True when the columns are those of a survey export the tool can read."""
    try:
        d2, _ = normalise_layout(df)
        detect_design(d2)
        return True
    except Exception:
        return False


def read_sheet(raw, sheet):
    """one Excel sheet as text; a sheet with one line of CSV text per row is rebuilt."""
    df = pd.read_excel(io.BytesIO(raw), sheet_name=sheet, dtype=str, header=None)
    if df.empty:
        return df
    if df.shape[1] == 1 or df.iloc[0].notna().sum() == 1:
        lines = df.apply(lambda r: ",".join(x for x in r if isinstance(x, str)), axis=1)
        text = "\n".join(lines)
        first = text.split("\n", 1)[0]
        sep = ";" if first.count(";") > first.count(",") else ","
        out = pd.read_csv(io.StringIO(text), sep=sep, dtype=str, low_memory=False)
    else:
        out = df.iloc[1:].reset_index(drop=True)
        out.columns = [str(c).strip() for c in df.iloc[0]]
    out.columns = [fix_accents(c) for c in out.columns]
    return out.map(fix_accents)


def read_export(path_or_bytes, name="", sheet_name=""):
    """Read a CSV or Excel export as text. Excel: the sheet named in sheet_name; otherwise the first sheet if it is a
    survey export, else the sheet that looks like one (names such as CE_data, data, raw, CEO first).
    The sheet read is kept in read_export.sheet."""
    name = name or str(path_or_bytes)
    raw = path_or_bytes if isinstance(path_or_bytes, (bytes, bytearray)) else open(path_or_bytes, "rb").read()
    read_export.sheet = ""
    if not name.lower().endswith((".xlsx", ".xls", ".xlsm")):
        return _read_csv_bytes(raw)
    sheets = pd.ExcelFile(io.BytesIO(raw)).sheet_names
    if sheet_name:
        if sheet_name not in sheets:
            raise ValueError(f"No sheet '{sheet_name}' in the file. Sheets: {', '.join(sheets)}")
        read_export.sheet = sheet_name
        return read_sheet(raw, sheet_name)
    first = read_sheet(raw, sheets[0])
    if looks_like_export(first):
        read_export.sheet = sheets[0]
        return first
    candidates = sorted(sheets[1:], key=lambda s: 0 if SHEET_NAME_HINT.search(s) else 1)
    for s in candidates:
        df = read_sheet(raw, s)
        if looks_like_export(df):
            read_export.sheet = s
            return df
    raise ValueError("No sheet looks like a survey export. Sheets: " + ", ".join(sheets) +
                     ". Write the right one in sheet_name.")


def _read_csv_bytes(raw):
    for enc in ("utf-8-sig", "cp1252"):
        try:
            text = raw.decode(enc)
            break
        except UnicodeDecodeError:
            continue
    first = text.split("\n", 1)[0]
    sep = ";" if first.count(";") > first.count(",") else ","
    df = pd.read_csv(io.StringIO(text), sep=sep, dtype=str, low_memory=False)
    df.columns = [fix_accents(c).strip() if isinstance(c, str) else c for c in df.columns]
    return df.map(fix_accents)


# ----------------------------------------------------------------------------------------------- v2: the Collect Earth 'IPCC' form (e.g. Ghana)
# land_use_category / land_use_subdivision = the end year; land_use_subcategory 'F > C' with its year = change 1;
# second_lu_* = change 2; alu_2000_category = the land in 2000; *_disturbance with years = disturbances.

IPCC_FORM = {
    "land_use_category": ("S0", 1, "ipcc_category", "land use in the end year"),
    "land_use_subdivision": ("S0", 2, "subcategory_1", "subdivision in the end year"),
    "land_use_subcategory": ("C1", 4, "change_label", "category before > after (code); its first letter = the land before change 1"),
    "land_use_category_has_changed": ("C1", "", "(asked)", "a land-use change was recorded"),
    "land_use_subcategory_year_of_change": ("C1", "", "(years)", "year of change 1"),
    "land_use_subdivision_change": ("C1", "", "(asked)", "a subdivision change was recorded"),
    "land_use_subdivision_year_of_change": ("C1", "", "(years)", "year of the subdivision change"),
    "land_use_initial_subdivision": ("S1", 2, "subcategory_1", "subdivision before change 1"),
    "second_lu_change": ("C2", "", "(asked)", "a second change was recorded"),
    "second_lu_conversion": ("S2", 1, "ipcc_category", "land use before change 2"),
    "second_lu_conversion_year": ("C2", "", "(years)", "year of change 2"),
    "second_lu_subdivision": ("S2", 2, "subcategory_1", "subdivision before change 2"),
    "alu_2000_category": ("S@2000", 1, "ipcc_category", "land use in 2000"),
    "alu_2000_subdivision": ("S@2000", 2, "subcategory_1", "subdivision in 2000"),
    "primary_disturbance": ("CD", 4, "driver", "disturbance (primary)"),
    "secondary_disturbance": ("CD", 4, "driver", "disturbance (secondary)"),
    "tertiary_disturbance": ("CD", 4, "driver", "disturbance (tertiary)"),
    "pre_disturbance_treecover": ("SD-before", 3, "canopy_cover", "tree cover before the disturbance"),
    "post_disturbance_treecover": ("SD-after", 3, "canopy_cover", "tree cover after the disturbance"),
    "grassland_mgmt": ("S0", 3, "managed", "grassland management in the end year"),
    "land_use_assessment_confidence": ("P", 0, "confidence", "plot information"),
    "regions": ("P", 0, "province", "plot information"),
    "stratum": ("P", 0, "stratum", "plot information"),
    "comments": ("P", 0, "comment", "plot information"),
    "vhri_year": ("P", 0, "imagery_year", "plot information"),
}
IPCC_FORM_SKIP = re.compile(r"^(alu_2000_subcategory|land_use_subcategory_year_of_change_know|aoi_classification|"
                            r"initital_aoi_classification|oldest_aoi_classification|second_lu_conversion_label|"
                            r"primary_disturbance_year.*|.*_label(\.\d+\.)?)$")


NO_ANSWER = {"none", "aucune", "ninguna", "nenhuma", "no", "0", "none or regrowth"}


def is_ipcc_form(cols):
    return {"land_use_category", "land_use_subcategory", "land_use_category_has_changed"} <= set(cols)


def role_ipcc_form(c):
    if c in IPCC_FORM:
        return IPCC_FORM[c]
    if re.match(r"^primary_disturbance_year\.\d+\.$", c):
        return "CD", "", "(years)", "year of the disturbance"
    if IPCC_FORM_SKIP.match(c) or NOT_USED.match(c):
        return "-", "", "", "not used (label, technical or calculated)"
    return None


def lab(row, c):
    """the label of an answer when the export has it (…_label), else the answer."""
    for k in (c + "_label", c.replace("year.", "year_label.")):
        if given(row.get(k)):
            return str(row[k]).strip()
    return str(row[c]).strip() if given(row.get(c)) else None


def is_true(v):
    return yes_no(v) == "yes" if given(v) else False


def place_ce_ipcc_form(row, survey):
    """Collect Earth 'IPCC' form, backward from the end year. Disturbances without a year are placed in the period of
    the land use they were asked for: from the last land-use change (or open) to the end year."""
    plot = new_plot(row, survey, survey["id_col"])
    end = survey["end_year"]
    s0 = []
    add_rows(s0, row, [(c, lvl, v) for c, (rec, lvl, v, _) in IPCC_FORM.items() if rec == "S0" and c in row])
    events = []                                           # each: dict(year_from, year_to, rows, before_rows, kind)
    code = str(row.get("land_use_subcategory") or "").strip() if given(row.get("land_use_subcategory")) else ""
    before_cat = code[0] if len(code) == 2 else (code.split(">")[0].strip() if ">" in code else None)
    if is_true(row.get("land_use_category_has_changed")) or is_true(row.get("land_use_subdivision_change")):
        yc = "land_use_subcategory_year_of_change" if is_true(row.get("land_use_category_has_changed")) else "land_use_subdivision_year_of_change"
        y = year_of(lab(row, yc))
        before = []
        if before_cat:
            before.append((1, "ipcc_category", value_of("ipcc_category", before_cat), "land_use_subcategory (first letter)"))
        if given(row.get("land_use_initial_subdivision")):
            before.append((2, "subcategory_1", str(row["land_use_initial_subdivision"]).strip(), "land_use_initial_subdivision"))
        events.append({"year_from": y[0], "year_to": y[1], "rows": [(4, "change_label", code, "land_use_subcategory")],
                       "before": before, "kind": "lu"})
    if is_true(row.get("second_lu_change")):
        y = year_of(lab(row, "second_lu_conversion_year"))
        before = []
        if given(row.get("second_lu_conversion")):
            before.append((1, "ipcc_category", value_of("ipcc_category", row["second_lu_conversion"], lab(row, "second_lu_conversion")), "second_lu_conversion"))
        if given(row.get("second_lu_subdivision")):
            before.append((2, "subcategory_1", str(row["second_lu_subdivision"]).strip(), "second_lu_subdivision"))
        events.append({"year_from": y[0], "year_to": y[1], "rows": [], "before": before, "kind": "lu"})
    last_lu = max([e["year_to"] for e in events if e["year_to"]], default=None)
    drivers = [(4, "driver", str(row[c]).strip(), c) for c in ("primary_disturbance", "secondary_disturbance", "tertiary_disturbance")
               if given(row.get(c)) and plain(row[c]) not in NO_ANSWER]
    if drivers:
        years = sorted({year_of(lab(row, c))[1] for c in row.keys()
                        if re.match(r"^primary_disturbance_year\.\d+\.$", c) and given(row.get(c))} - {None})
        dist = [(y, y) for y in years] or [(last_lu, end)]   # no year: in the period of the current land use
        for y_from, y_to in dist:
            events.append({"year_from": y_from, "year_to": y_to, "kind": "disturbance", "before": [],
                           "rows": [(4, "change_label", "disturbance", "primary_disturbance")] + drivers})
    events.sort(key=lambda e: -(e["year_to"] or -1))      # newest first, as asked
    n = len(events)
    plot["states"][0] = {"rows": s0}
    dist_idx = [i for i, e in enumerate(events, start=1) if e["kind"] == "disturbance"]
    for k, e in enumerate(events, start=1):
        before = list(e["before"])
        if dist_idx and k == max(dist_idx) and given(row.get("pre_disturbance_treecover")):
            before.append((3, "canopy_cover", lab(row, "pre_disturbance_treecover"), "pre_disturbance_treecover"))  # label: '45-49 Points' 
        plot["states"][k] = {"rows": before}
        plot["changes"][k] = {"rows": e["rows"], "year_from": e["year_from"], "year_to": e["year_to"]}
    if dist_idx and given(row.get("post_disturbance_treecover")):
        plot["states"][min(dist_idx) - 1]["rows"].append(
            (3, "canopy_cover", lab(row, "post_disturbance_treecover"), "post_disturbance_treecover"))
    place_backward(plot, survey, n)
    # the land in 2000: a state in a reported year, not linked to a change
    a2000 = []
    add_rows(a2000, row, [(c, lvl, v) for c, (rec, lvl, v, _) in IPCC_FORM.items() if rec == "S@2000" and c in row])
    if a2000:
        newer = [k for k, st in plot["states"].items() if (st.get("year_to") or 9999) > 2000]
        for k, st in plot["states"].items():
            if k in newer:
                st["order"] += 1
        plot["states"][n + 1] = {"rows": a2000, "year_from": 2000, "year_to": 2000,
                                 "order": min([plot["states"][k]["order"] for k in newer], default=1) - 1}
        orders = sorted(plot["states"], key=lambda k: plot["states"][k]["order"])
        for i, k in enumerate(orders):
            plot["states"][k]["order"] = i
        for ch in plot["changes"].values():
            ch["order"] = sum(1 for c2 in plot["changes"].values() if (c2["year_to"] or -1) <= (ch["year_to"] or -1))
    return plot


# ----------------------------------------------------------------------------------------------- v4: the from/to table


def sheet_parts(name, df):
    """a table in sheets an Excel file can hold: '<name>' when it fits, otherwise '<name> (1)', '<name> (2)'..."""
    if len(df) <= EXCEL_ROWS:
        return [(name, df)]
    return [(f"{name} ({i // EXCEL_ROWS + 1})", df.iloc[i:i + EXCEL_ROWS]) for i in range(0, len(df), EXCEL_ROWS)]


YEAR_COLUMNS = ["survey_id", "plot_id", "year", "land_use", "subdivision", "subdivision_2", "subdivision_3", "source",
                 "change", "change_type", "disturbance", "regrowth", "driver_change", "driver_disturbance", "driver_regrowth", "driver_unclear"]
LAND = ["land_use", "subdivision", "subdivision_2", "subdivision_3"]
SUBDIVISIONS = {"subcategory_1": "subdivision", "subcategory_2": "subdivision_2", "subcategory_3": "subdivision_3"}
READY_DESIGNS = {"event", "change", "ipcc_form", "cards", "annual"}
BACKWARD = {"event", "change", "ipcc_form", "cards"}


# ----------------------------------------------------------------------------------------------- the rows

def code(row, column):
    """the answer as written in the export: '' when not answered."""
    if not given(row.get(column)):
        return ""
    v = str(row[column]).strip()
    return v[:-2] if re.match(r"^-?\d+\.0$", v) else v


def original(row, source, value, names=None):
    """the original code of an answer placed by the reader. A land written by the code, not read from a column, takes the
    name the survey itself uses for that land."""
    if "(first letter)" in source:
        return code(row, source.split()[0])[:1]
    return code(row, source) or (names or {}).get(value, value)


def survey_names(plots, records):
    """the name the survey itself uses for each land: the most common one in its columns."""
    seen = defaultdict(Counter)
    for plot in plots:
        row = records[str(plot["plot_id"])]
        for state in plot["states"].values():
            for level, _, value, source in state["rows"]:
                own = original(row, source, "")
                if level in (1, 2) and own:
                    seen[value][own] += 1
    return {value: names.most_common(1)[0][0] for value, names in seen.items()}


def add(rows, year, **values):
    """one row per year: what is already there stays, a note is added to the one before."""
    target = rows.setdefault(year, {})
    for k, v in values.items():
        if v in ("", None):
            continue
        if k == "note" and target.get(k):
            target[k] += "; " + v
        elif k.startswith("driver") and target.get(k) and v not in target[k]:
            target[k] += "; " + v
        else:
            target.setdefault(k, v)


def land_of(state, row, names):
    """the land and the attributes of a state, with the original codes."""
    land = {}
    for level, variable, value, source in state["rows"]:
        name = "land_use" if level == 1 else SUBDIVISIONS.get(variable, variable)
        if name == "note" and land.get("note"):
            land["note"] += "; " + value
            continue
        land[name] = original(row, source, value, names)
    return land


SEPARATE_DRIVER = re.compile(r"^(primary|secondary|tertiary)_disturbance$")
DRIVER_OF_CHANGE = re.compile(r"deforest|(^|_)def\d*(_|$)", re.I)
DRIVER_OF_DISTURBANCE = re.compile(r"degrad|(^|_)deg\d*(_|$)", re.I)


def driver_name(column):
    """where a driver goes, from its question. A question for the whole plot is decided later; a question naming its kind
    goes there; a driver inside a change follows what happened in that change."""
    if SEPARATE_DRIVER.match(column):
        return "driver_separate"
    if DRIVER_OF_CHANGE.search(column):
        return "driver_change"
    if DRIVER_OF_DISTURBANCE.search(column):
        return "driver_disturbance"
    return "driver"


def recorded_in(change, row):
    """what was recorded with a change: its kind, driver and attributes."""
    out = {"change": "yes"}
    for level, variable, value, source in change["rows"]:
        if variable == "change_label":
            text = value.lower()
            if value not in ("disturbance", "regrowth"):
                out["change_type"] = value
            if value == "disturbance" or "disturbance without change" in text or "degradation" in text:
                out["disturbance"] = "yes"
            elif value == "regrowth" or "enhancement" in text:
                out["regrowth"] = "yes"
        elif variable == "driver":
            name = driver_name(source)
            found = original(row, source, value)
            out[name] = f"{out[name]}; {found}" if out.get(name) and found not in out[name] else found
        elif variable == "note":
            out["note"] = f"{out['note']}; {value}" if out.get("note") else value
        elif variable != "year_code":
            out[variable] = original(row, source, value)
    return out


def rows_of_plot(plot, row, end, names):
    """the land of each state in its years, then each change in its year."""
    rows, copied = {}, []
    for k in sorted(plot["states"]):
        state = plot["states"][k]
        land = land_of(state, row, names)
        asked = any(r[0] in (1, 2) and not r[3].startswith("not asked again") for r in state["rows"])
        first, last = state.get("year_from"), state.get("year_to")
        given_for = first if state.get("snap") is False else last      # the year the input file gives this land for
        if asked and given_for is not None:
            add(rows, given_for, **land)
        copied += [(y, land) for y in (first, last) if y is not None and not (asked and y == given_for)]
    for y, land in copied:                                # the other end of the period: the same land, copied
        add(rows, y, **land, source="" if y in rows else "filled")
    for k in sorted(plot["changes"]):
        ch = plot["changes"][k]
        y = ch.get("year_to")
        if y is None:
            code_given = next((r[2] for r in ch["rows"] if r[1] == "year_code"), "")
            add(rows, end, note="change without a year" + (f" (year code {code_given})" if code_given else ""))
            continue
        if ch.get("year_from") != y:
            add(rows, y, note=f"change before {y + 1}: the exact year is not known")
        add(rows, y, **recorded_in(ch, row))
    return rows


def is_update_form(columns):
    return {"land_use_latest_category", "changes_from_previous_assesment"} <= set(columns)


def previous_year_found(survey):
    """the year of the earlier assessment, from the data: on the plots with a change, the land the survey calculates for each
    year (ipcc_YYYY_category) stops being the earlier land after that year. 0 when the data do not say it."""
    data = survey["data"]
    changed = data[data["changes_from_previous_assesment"].astype(str).eq("change") & data["land_use_latest_category"].notna()]
    years = sorted(int(m.group(1)) for c in data.columns for m in [re.match(r"^ipcc_(\d{4})_category$", c)] if m)
    if changed.empty or not years:
        return 0
    earlier = changed["land_use_latest_category"].astype(str).str[:1]
    same = {y: (changed[f"ipcc_{y}_category"].astype(str).str[:1] == earlier).mean() for y in years}
    best = max(same.values())
    found = max(y for y in years if same[y] == best)
    return found if found < years[-1] else 0


def rows_of_update(row, end, previous):
    """a survey updating an earlier assessment (Zambia): each plot comes with its land in the previous year; the
    interpreter says if it changed. 'no_change' hides the form: the land stays the land of the previous year."""
    rows = {}
    answer = code(row, "changes_from_previous_assesment")
    old_use, old_sub = code(row, "land_use_latest_category"), code(row, "land_use_latest_subdivision")
    use, sub = code(row, "land_use_category"), code(row, "land_use_subdivision")
    note = ""
    if answer == "no_change":
        use, sub, note = old_use, old_sub, f"not asked: no change since {previous}"
    elif answer == "change" and yes_no(code(row, "land_use_category_has_changed")) != "yes":
        note = f"change since {previous} answered, no year given"
    add(rows, end, land_use=use, subdivision=sub, note=note, source="filled" if answer == "no_change" else "",
        tree_cover=code(row, "land_covertrees_coverage") or code(row, "tree_cover_2024_degradation"))

    # the land changes, newest first: (year, land before)
    steps = []
    if yes_no(code(row, "land_use_category_has_changed")) == "yes":
        steps.append((code(row, "land_use_subcategory_year_of_change"), code(row, "land_use_subcategory")[:1],
                      code(row, "land_use_initial_subdivision")))
    elif yes_no(code(row, "land_use_subdivision_change")) == "yes":
        steps.append((code(row, "land_use_subdivision_year_of_change"), use, code(row, "land_use_initial_subdivision")))
    if yes_no(code(row, "second_lu_change")) == "yes":
        steps.append((code(row, "second_lu_conversion_year"), code(row, "second_lu_conversion"),
                      code(row, "second_lu_subdivision")))
    periods = [(None, use, sub)]                          # (first year, land): the newest first
    for y, before_use, before_sub in steps:
        y = year_of(y)[1]
        if y is None:
            add(rows, end, note="change without a year")
            continue
        periods[-1] = (y, *periods[-1][1:])
        periods.append((None, before_use, before_sub))
        add(rows, y, land_use=periods[-2][1], subdivision=periods[-2][2], change="yes", source="" if y in rows else "filled")
        add(rows, y - 1, land_use=before_use, subdivision=before_sub)

    def land_in(year):
        for first, u, s in periods:
            if first is None or year >= first:
                return u, s
        return periods[-1][1:]

    # the disturbance: the land is not asked again, the row has the land of that year
    driver = code(row, "primary_disturbance")
    if driver and plain(driver) != "none":
        y = year_of(code(row, "primary_disturbance_year"))[1]
        if y is None:
            add(rows, end, note=f"disturbance {driver} without a year")
        else:
            u, s = land_in(y)
            add(rows, y, land_use=u, subdivision=s, change="yes", disturbance="yes", driver_separate=driver, source="" if y in rows else "filled",
                tree_cover_before=code(row, "tree_cover_before_disturbance"),
                tree_cover_after=code(row, "tree_cover_after_disturbance"))

    # the land in the previous year, from the earlier assessment
    if old_use:
        wrong = answer == "misinterpretaion"
        if previous not in rows:                          # no land recorded by the interpreter for that year
            note = f"{previous} class from the earlier assessment: the interpreter says it was wrong" if wrong else ""
        elif rows[previous].get("land_use") != old_use:
            note = f"the earlier assessment says {old_use} in {previous}" + (": the interpreter says it was wrong" if wrong else "")
        else:
            note = ""
        add(rows, previous, land_use=old_use, subdivision=old_sub, note=note,
            wrong="yes" if wrong and previous not in rows else "",
            tree_cover=code(row, "land_covertrees_latest_coverage"))
    return rows


def fill_plot_years(rows, start, end, direction="backward"):
    """the years with no row, from the start year to the end year: the land of the rows around them, when they agree.
    A land the interpreter said was wrong does not count. Backward: before the oldest row, the land of the oldest row.
    Onward: after the newest row, the land of the newest row. The other side stays empty."""
    collected = sorted(rows)
    for y in range(start, end + 1):
        if y in collected:
            continue
        earlier = [c for c in collected if c < y]
        older = [c for c in earlier if not rows[c].get("wrong")]
        newer = [c for c in collected if c > y]
        if newer and older and rows[newer[0]].get("change") == "yes" and "exact year is not known" not in rows[newer[0]].get("note", ""):
            before = after = rows[older[-1]]                # a change with its year: the land before it lasts until then
        elif newer and older:
            before, after = rows[older[-1]], rows[newer[0]]
        elif newer and (earlier or direction == "backward"):
            before = after = rows[newer[0]]
        elif older and not newer and direction == "onward":
            before = after = rows[older[-1]]
        else:
            before = after = {}
        land = {k: after[k] for k in LAND if k in after and before.get(k) == after[k]}
        missing = [k for k in LAND if (k in after or k in before) and k not in land]
        rows[y] = {**land, "source": "filled" if land else "empty"}
        if missing:
            rows[y]["note"] = "the land before and after differs and no change gives the year: " + ", ".join(missing) + " left empty"


def settle_what_happened(rows):
    """change and disturbance are two different things. In a year with something recorded: a land use or subdivision that
    differs from the year before is a change; the same land is a disturbance (or a regrowth), never both."""
    for year, row in rows.items():
        if row.get("change") != "yes":
            continue
        before = rows.get(year - 1, {})
        if before.get("wrong"):                           # the interpreter said this land was wrong: it does not count
            before = {}
        same_kind = "disturbance" if row.get("disturbance") else "regrowth" if row.get("regrowth") else ""
        known = before.get("land_use") and row.get("land_use")
        if any(before.get(k) and row.get(k) and before[k] != row[k] for k in LAND):
            row.pop("disturbance", None)
            row.pop("regrowth", None)
            what = "change"
        else:
            what = same_kind or "unclear"
            if known or same_kind:
                del row["change"]
            if known and not same_kind:
                add(rows, year, note=f"{row.get('change_type') or 'a change'} recorded, but the land stays the same")
        if "driver" in row:                               # a driver inside the change: it belongs to what happened
            row[f"driver_{what}"] = row.pop("driver")


def place_separate_drivers(by_plot):
    """a driver question of its own for the whole plot, with its own year. Answered on plots where the land never changes,
    it cannot be about a land change: it is about a disturbance, on every plot. Otherwise it stays unclear."""
    with_driver = [rows for rows in by_plot.values() if any(r.get("driver_separate") for r in rows.values())]
    on_stable_land = any(not any(r.get("change") == "yes" for r in rows.values()) for rows in with_driver)
    what = "disturbance" if on_stable_land else "unclear"
    for rows in with_driver:
        for year, row in rows.items():
            if "driver_separate" in row:
                row[f"driver_{what}"] = row.pop("driver_separate")
                if row.get("change") == "yes" and what == "disturbance":
                    add(rows, year, note="a disturbance was also recorded in this year")


# ----------------------------------------------------------------------------------------------- the table

def survey_direction(survey):
    """backward when the survey starts from the end year and goes back from change to change. A survey with the land at
    each date has the direction of its questions: the oldest date asked first is onward. Empty = not found."""
    if survey["design"] == "annual":
        dates = list(survey["years"])
        return "onward" if dates[0] < dates[-1] else "backward" if dates[0] > dates[-1] else ""
    return "backward" if survey["design"] in BACKWARD else ""


def plot_year_table(survey, previous_year=0, start_year=0, direction=""):
    """the table: one row per plot and year. Start year 0 = the first year with information. Direction empty = what the survey says."""
    survey_id, end_year = survey["survey_id"], survey["end_year"]
    if survey["design"] not in READY_DESIGNS:
        raise ValueError(f"design '{survey['design']}': not ready yet")
    if survey.get("point_col"):
        raise ValueError("several points in each plot: not ready yet")
    direction = direction or survey_direction(survey)
    update = is_update_form(survey["data"].columns)
    if update:
        previous_year = survey["previous_year"] = previous_year or previous_year_found(survey)
        if not previous_year:
            raise ValueError("⛔ This survey updates an earlier assessment and its year could not be found: "
                             "run a new cell with previous_year = 2018 (your year), then this step again.")
    id_col = survey["id_col"]
    aside = set(map(str, survey.get("set_aside", ())))    # set aside by the quality check
    records = {str(r[id_col]): r for r in survey["data"].to_dict("records")
               if given(r.get(id_col)) and str(r[id_col]) not in aside}
    if update:
        by_plot = {p: rows_of_update(r, end_year, previous_year) for p, r in records.items()}
    else:
        plots, _ = reframe_plots(survey)
        names = survey_names(plots, records)
        by_plot = {}
        for plot in plots:
            fill_land_not_asked(plot)
            by_plot[str(plot["plot_id"])] = rows_of_plot(plot, records[str(plot["plot_id"])], end_year, names)
    start = start_year or min(min(years) for years in by_plot.values() if years)
    for years in by_plot.values():
        fill_plot_years(years, start, end_year, direction)
        settle_what_happened(years)
    place_separate_drivers(by_plot)
    rows = [{"survey_id": survey_id, "plot_id": p, "year": y, **{k: v for k, v in values.items() if k != "wrong"}}
            for p, years in by_plot.items() for y, values in sorted(years.items(), reverse=direction == "backward")]
    table = pd.DataFrame(rows).fillna("")
    if "source" not in table:                             # every year was collected: nothing to fill
        table["source"] = ""
    no_land = table[[k for k in LAND if k in table]].eq("").all(axis=1)
    table["source"] = table["source"].mask(no_land, "empty").replace("", "raw")
    table["change"] = table["change"].replace("", "no") if "change" in table else "no"
    table["disturbance"] = table["disturbance"].replace("", "no") if "disturbance" in table else "no"
    if "regrowth" in table:
        table["regrowth"] = table["regrowth"].replace("", "no")
    first = [c for c in YEAR_COLUMNS if c in table]
    last = ["note"] if "note" in table else []
    return table[first + [c for c in table if c not in first + last] + last]


def fill_land_not_asked(plot):
    """the land before a disturbance, when the survey does not ask it again, is the land after it."""
    for ch in plot["changes"].values():
        a, b = ch.get("from"), ch.get("to")
        if a is None or b is None or any(r[0] in (1, 2) for r in plot["states"][a]["rows"]):
            continue                                      # some land was recorded for it: nothing is copied
        plot["states"][a]["rows"] += [(lv, var, val, "not asked again")
                                      for lv, var, val, _ in plot["states"][b]["rows"] if lv in (1, 2)]


# ----------------------------------------------------------------------------------------------- the country

def country_name(code_found):
    """'CI' -> 'Cote_d_Ivoire': the country name, written so it can go in a file name."""
    try:
        import pycountry
        found = pycountry.countries.get(alpha_2=code_found)
        name = getattr(found, "common_name", found.name)
    except (ImportError, AttributeError):
        return code_found
    if ", " in name:                                      # 'Korea, Republic of' -> 'Republic of Korea'
        name = " ".join(reversed(name.split(", ", 1)))
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode()
    return re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")


def country_from_location(survey):
    """the country, from the location of the plots: the one with the majority of the plots. Otherwise, or when the
    location cannot be read, 'multiple'. The nearest town gives the country, so plots near a border can fall on the
    other side (about 10% in Benin): this is why it is the majority and not all the plots."""
    roles, data = survey["roles"], survey["data"]
    columns = [roles.loc[roles["variable"] == v, "export column"] for v in ("x", "y")]
    if not all(len(c) for c in columns):
        return "multiple"
    x, y = (pd.to_numeric(data[c.iloc[0]], errors="coerce") for c in columns)
    good = x.between(-180, 180) & y.between(-90, 90)
    if not good.any():
        return "multiple"
    try:
        import reverse_geocoder
        found = reverse_geocoder.search(list(zip(y[good], x[good])), mode=1, verbose=False)
    except Exception as problem:                          # the tool could not be installed or read
        print(f"⚠ The country could not be worked out ({problem}).")
        return "multiple"
    code_found, n = Counter(place["cc"] for place in found).most_common(1)[0]
    return country_name(code_found) if n / len(found) > 0.5 else "multiple"


# ----------------------------------------------------------------------------------------------- the land check

CANNOT_BE_PLACED = re.compile(r"without a year|no year given|the land stays the same|, but land in \d{4}:|= no, but a change is recorded"
                              r"|, outside the period|canopy cover outside 0–100")
CANOPY_CHECK = re.compile(r"^\d{4}–\d{4}: .*, canopy \S+ → \S+ \(not (lower|higher)\)$")
ANSWERS_ONLY = re.compile(r"^\d{4}–\d{4}: .*(, but \S+ = |with no type, year or land after|, but the change is )")


def land_problems(table):
    """what must be fixed in the export, read from the table, the same for any survey: the land differs between two years
    and no change is recorded; a year with no land; something recorded that cannot be placed or that the land contradicts."""
    land = [c for c in LAND if c in table]
    note = table["note"] if "note" in table else pd.Series("", index=table.index)
    out = []

    def problem(plot, detail):
        out.append({"check": "4 land", "level": "⛔ to fix", "plot": plot, "detail": detail})

    before = table[["plot_id", "year"] + land].assign(year=table["year"] + 1, note=note)
    pairs = table[["plot_id", "year", "change"] + land].assign(note=note).merge(before, on=["plot_id", "year"], suffixes=("", "_before"))
    differs = pd.concat([pairs[c].ne(pairs[f"{c}_before"]) & pairs[c].ne("") & pairs[f"{c}_before"].ne("") for c in land], axis=1).any(axis=1)
    for r in pairs[differs & pairs["change"].eq("no")].to_dict("records"):
        said = "; ".join(x for x in (r["note_before"], r["note"]) if x)
        was = " / ".join(r[f"{c}_before"] for c in land if r[f"{c}_before"])
        now = " / ".join(r[c] for c in land if r[c])
        problem(r["plot_id"], f"{r['year'] - 1}: {was}, {r['year']}: {now}, no change recorded")
    empty = table[table["source"] == "empty"].groupby("plot_id", sort=False)["year"].agg(["min", "max", "size"])
    rows_of = table.groupby("plot_id", sort=False).size()
    for plot, e in empty.iterrows():
        problem(plot, "no land in any year" if e["size"] == rows_of[plot] else
                f"{e['min']}–{e['max']}: the land before and after differs, no change year recorded")
    for r in table.assign(note=note)[note.ne("")].to_dict("records"):
        for piece in str(r["note"]).split("; "):         # only the part of the note that is a problem
            if CANOPY_CHECK.match(piece):
                out.append({"check": "4 canopy", "level": "⚠ to check", "plot": r["plot_id"], "detail": piece})
            elif ANSWERS_ONLY.match(piece):
                out.append({"check": "4 answers", "level": "⚠ to check", "plot": r["plot_id"], "detail": piece})
            elif CANNOT_BE_PLACED.search(piece):
                problem(r["plot_id"], f"{r['year']}: {piece}")
    recorded = table[(table["change"] == "yes") | (table["disturbance"] == "yes")]
    if len(recorded) >= 5 and recorded["year"].nunique() == 1 and table["year"].nunique() > 2:
        out.append({"check": "5 years", "level": "⚠ to check", "plot": "",
                    "detail": f"all the {len(recorded)} changes and disturbances are in {recorded['year'].iloc[0]}: please check it"})
    if "driver_unclear" in table:
        n = int(table["driver_unclear"].ne("").sum())
        if n:
            out.append({"check": "6 drivers", "level": "⚠ to check", "plot": "", "detail": f"{n} drivers: the data do not say what they are about (driver_unclear)"})
    return pd.DataFrame(out, columns=["check", "level", "plot", "detail"])


# ----------------------------------------------------------------------------------------------- the survey description

def what_it_holds(record):
    """the role of a column, in words."""
    record = str(record)
    if record == "-":
        return "not used"
    if record == "P":
        return "plot information"
    if record == "S0":
        return "land in the end year"
    if record.startswith("S@"):
        return f"land in {record[2:]}"
    if record == "PERIOD":
        return "a period between two dates"
    if record == "CD":
        return "a disturbance"
    return "a change" if record.startswith("C") else "land before a change"


def type_of_answers(answers, coded=False):
    """the type of the answers of a question. coded: the export gives a label with each answer."""
    v = answers.astype(str).str.strip()
    if not len(v):
        return "(no answer)"
    if v.str.fullmatch(r"-?\d+").all() and pd.to_numeric(v).between(1900, 2100).all():
        return "year"
    if coded:
        return "code"
    if v.str.fullmatch(r"-?\d+").all():
        return "yes / no (1 / 0)" if set(v) == {"0", "1"} else "integer"
    if v.str.fullmatch(r"-?\d+([.,]\d+)?").all():
        return "decimal"
    if v.str.fullmatch(r"\d{4}-\d{2}-\d{2}.*").all():
        return "date"
    if v.map(lambda x: yes_no(x) is not None).all():
        return "yes / no"
    return "text"


def answers_offered(raw):
    """the answers a form offered, when the export has one column per answer ('question:answer')."""
    offered = defaultdict(list)
    for c in raw.columns:
        if ":" in str(c):
            question, answer = str(c).split(":", 1)
            offered[question.strip()].append(answer.strip())
    return offered


def possible_answers(data, column, answers, offered, most=40):
    """each answer with its label, when the export has one, and how many plots gave it. Too many different answers
    (e.g. coordinates): only the number."""
    counts = answers.astype(str).str.strip().value_counts()
    label_column = f"{column}_label"
    labels_found = {}
    if label_column in data.columns:
        pairs = data.loc[answers.index, [column, label_column]].dropna().astype(str)
        labels_found = dict(zip(pairs[column].str.strip(), pairs[label_column].str.strip()))
    if len(counts) > most:
        return f"{len(counts):,} different answers"
    items = [f"{a} = {labels_found[a]} ({n:,})" if labels_found.get(a, a) != a else f"{a} ({n:,})" for a, n in counts.items()]
    items += [f"{a} (0)" for a in offered if a not in counts]
    return "; ".join(items)


BLOCK_ORDER = ["plot information", "land in the end year", "a change", "land before a change", "a disturbance",
               "a period between two dates", "not used"]


def survey_questions(survey):
    """the structure of the survey, question by question: what each one holds, its type, its possible answers."""
    data, roles = survey["data"], survey["roles"]
    offered = answers_offered(survey["raw"])
    plots = len(data)
    rows = []
    for r in roles.to_dict("records"):
        c = r["export column"]
        if str(c).endswith("_label") and str(c)[:-6] in data.columns:
            continue
        answers = data[c][data[c].map(given)]
        rows.append({"question": c, "what it holds": what_it_holds(r["record"]), "type": type_of_answers(answers, f"{c}_label" in data.columns),
                     "plots answered": len(answers), "% of plots": round(100 * len(answers) / plots, 1) if plots else 0,
                     "different answers": answers.astype(str).str.strip().nunique(),
                     "possible answers (plots)": possible_answers(data, c, answers, offered.get(str(c), [])),
                     "detail": r["note"]})
    out = pd.DataFrame(rows)
    rank = out["what it holds"].map(lambda h: BLOCK_ORDER.index(h) if h in BLOCK_ORDER else 2.5 if h.startswith("land in") else 9)
    return out.assign(rank=rank).sort_values("rank", kind="stable").drop(columns="rank").reset_index(drop=True)


def survey_facts(survey, table, direction):
    """the main facts of the survey."""
    x_y = survey["roles"].loc[survey["roles"]["variable"].isin(["x", "y"]), "export column"]
    facts = [("file", survey.get("file", "")), ("sheet read", survey.get("sheet", "")),
             ("made on", date.today().isoformat()),
             ("tool", {"CE": "Collect Earth", "CEO": "Collect Earth Online"}[survey["tool"]]),
             ("design", survey["design_text"]), ("country (from the location of the plots)", survey["survey_id"]),
             ("location columns", ", ".join(x_y)), ("plot id column", survey["id_col"]),
             ("plots", survey["plots"]), ("end year", survey["end_year"]),
             ("years of the table", f"{table['year'].min()}-{table['year'].max()}"), ("direction", direction),
             ("year of the earlier assessment (found from the data: please check it)", survey.get("previous_year", ""))]
    return pd.DataFrame([f for f in facts if f[1] != ""], columns=["survey", ""])


def write_workbook(path, survey, table, direction, report):
    """three parts: the raw data, the survey description, the table (filled rows in blue)."""
    raw_parts, table_parts = sheet_parts("raw_data", survey["raw"]), sheet_parts("unified_format", table)
    with pd.ExcelWriter(path, engine="xlsxwriter") as xw:
        for name, part in raw_parts:
            part.to_excel(xw, sheet_name=name, index=False)
            xw.sheets[name].freeze_panes(1, 0)
        row = 0
        for part in [survey_facts(survey, table, direction)] + ([report] if len(report) else []) + [survey_questions(survey)]:
            part.to_excel(xw, sheet_name="survey description", index=False, startrow=row)
            row += len(part) + 2
        ws = xw.sheets["survey description"]
        ws.set_column(0, 0, 44)
        ws.set_column(1, 5, 20)
        ws.set_column(6, 6, 90)
        ws.set_column(7, 7, 50)
        blue = xw.book.add_format({"font_color": "#1F5FA8", "bold": True})
        col = chr(ord("A") + list(table.columns).index("source"))
        for name, part in table_parts:
            part.to_excel(xw, sheet_name=name, index=False)
            ws = xw.sheets[name]
            ws.freeze_panes(1, 0)
            ws.autofilter(0, 0, len(part), len(part.columns) - 1)
            ws.conditional_format(1, 0, len(part), len(part.columns) - 1,
                                  {"type": "formula", "criteria": f'=${col}2="filled"', "format": blue})


# ----------------------------------------------------------------------------------------------- the pairs

def pairs_table(table):
    """one row per plot and pair of years following each other: the land in the two years, and what was recorded in the
    second one. Made from the plot-by-year table only."""
    land = [c for c in LAND if c in table]
    before = table[["plot_id", "year"] + land].rename(columns={"year": "from_year", **{c: f"from_{c}" for c in land}})
    before["year"] = before["from_year"] + 1
    pairs = table.drop(columns=["source"] + (["note"] if "note" in table else [])).merge(before, on=["plot_id", "year"])
    pairs = pairs.rename(columns={"year": "to_year", **{c: f"to_{c}" for c in land}})
    first = ["survey_id", "plot_id", "from_year", "to_year"] + [f"from_{c}" for c in land] + [f"to_{c}" for c in land] + ["change", "disturbance"]
    return pairs[first + [c for c in pairs if c not in first]]


# ----------------------------------------------------------------------------------------------- the picture of the survey

PAGE_STYLE = """
:root { --ink:#1d2733; --soft:#5b6b7b; --line:#c9d3dc; --page:#ffffff; --box:#f4f7f9; --land:#e3f1e5; --landline:#3f8f55;
        --chg:#fdf0dc; --chgline:#c98a1b; --raw:#3f8f55; --filled:#bcdcc3; --empty:#e6e9ec; }
@media (prefers-color-scheme: dark) {
  :root { --ink:#e8edf2; --soft:#a4b1bd; --line:#3c4a57; --page:#161c22; --box:#1f2831; --land:#1e3325; --landline:#5fb878;
          --chg:#3a2d15; --chgline:#e0a93f; --raw:#5fb878; --filled:#2f5a3d; --empty:#2b343d; } }
* { box-sizing: border-box; }
body { margin: 0; background: var(--page); color: var(--ink); font: 15px/1.5 system-ui, "Segoe UI", Arial, sans-serif; }
main { max-width: 1040px; margin: 0 auto; padding: 24px 16px 48px; }
h1 { font-size: 24px; margin: 0 0 4px; } h2 { font-size: 17px; margin: 32px 0 10px; }
.sub, .hint { color: var(--soft); } .hint { font-size: 13px; margin: 6px 0 0; }
.facts { display: flex; flex-wrap: wrap; gap: 10px; margin-top: 16px; }
.fact { background: var(--box); border-radius: 6px; padding: 8px 12px; min-width: 96px; }
.fact b { display: block; font-size: 18px; font-variant-numeric: tabular-nums; } .fact span { color: var(--soft); font-size: 12px; }
.flow { display: flex; flex-wrap: wrap; align-items: stretch; gap: 8px; }
.arrow { align-self: center; color: var(--soft); font-size: 20px; }
.repeat { display: flex; flex-wrap: wrap; align-items: stretch; gap: 8px; border: 1.5px dashed var(--line); border-radius: 8px; padding: 8px; flex: 3 1 400px; }
.repeat .box { flex: 1 1 130px; min-width: 130px; }
.repeat > .tag { flex-basis: 100%; color: var(--soft); font-size: 13px; }
.box { flex: 1 1 170px; min-width: 150px; border-radius: 6px; padding: 10px 12px; border-left: 4px solid; }
.box.land { background: var(--land); border-color: var(--landline); } .box.chg { background: var(--chg); border-color: var(--chgline); }
.box h3 { margin: 0 0 2px; font-size: 15px; } .box p { margin: 0; font-size: 13px; color: var(--soft); }
details { margin-top: 6px; font-size: 12px; } summary { cursor: pointer; color: var(--soft); }
details ul { margin: 4px 0 0; padding-left: 16px; overflow-wrap: anywhere; }
ul.rules { margin: 0; padding-left: 20px; }
.strip { display: flex; flex-wrap: wrap; gap: 3px; }
.year { width: 80px; text-align: center; font-size: 12px; font-variant-numeric: tabular-nums; }
.cell { border-radius: 4px; padding: 7px 2px; min-height: 34px; overflow: hidden; text-overflow: ellipsis; white-space: nowrap; }
.raw .cell { background: var(--raw); color: #fff; font-weight: 600; } .filled .cell { background: var(--filled); } .empty .cell { background: var(--empty); }
.strip .cell { white-space: normal; font-size: 11px; line-height: 1.2; min-height: 40px; padding: 4px 2px; }
.strip .mark { height: 26px; line-height: 1.1; white-space: normal; font-size: 10px; display: flex; align-items: flex-end; justify-content: center; }
.cov { color: var(--soft); font-size: 11px; min-height: 15px; }
h3.ex { font-size: 15px; margin: 22px 0 4px; }
.answers { display: flex; flex-wrap: wrap; gap: 8px; margin-bottom: 4px; }
.ans { background: var(--box); border-radius: 6px; padding: 6px 10px; font-size: 12px; flex: 1 1 200px; max-width: 340px; }
.ans ul { margin: 2px 0 0; padding-left: 14px; overflow-wrap: anywhere; } .ans li.z { list-style: none; margin-left: -14px; }
.ans code { font-size: 11px; color: var(--soft); }
.mark { height: 16px; color: var(--chgline); font-size: 11px; font-weight: 700; }
.legend { display: flex; flex-wrap: wrap; gap: 14px; margin-top: 8px; font-size: 13px; color: var(--soft); }
details.block { margin-top: 10px; font-size: 14px; border: 1px solid var(--line); border-radius: 6px; padding: 6px 10px; }
details.block > summary { color: var(--ink); }
.tw { overflow-x: auto; }
table.q { border-collapse: collapse; width: 100%; margin-top: 8px; font-size: 13px; }
table.q th { text-align: left; color: var(--soft); font-weight: 600; border-bottom: 1px solid var(--line); padding: 4px 6px; vertical-align: bottom; }
table.q td { border-bottom: 1px solid var(--line); padding: 5px 6px; vertical-align: top; }
table.q td.n { text-align: right; font-variant-numeric: tabular-nums; white-space: nowrap; }
table.q tr.none td { color: var(--soft); }
table.q code { font-size: 12px; overflow-wrap: anywhere; }
.det { color: var(--soft); font-size: 12px; }
.tag1 { display: inline-block; font-size: 11px; color: var(--chgline); }
ul.ans { margin: 0; padding-left: 14px; overflow-wrap: anywhere; } ul.ans details { margin: 0; }
.schema { align-items: center; margin-top: 6px; }
.tree { display: grid; grid-template-columns: 170px 1fr; gap: 8px 14px; font-size: 13px; line-height: 1.45; margin: 8px 0 16px; }
.tree .lu { border-left: 3px solid var(--landline); padding: 3px 8px; font-weight: 600; }
.tree .lu.chg { border-left-color: var(--chgline); }
.tree .lu small { font-weight: 400; color: var(--soft); }
.tree .subs { display: flex; flex-wrap: wrap; gap: 4px 6px; align-items: flex-start; }
.tree .s { border: 1px solid var(--line); border-radius: 6px; padding: 1px 7px; }
.tree .s small, .tree .at small { color: var(--soft); font-size: 11.5px; }
.tree .s.odd { border: 1px dashed #c0392b; color: #c0392b; }
.tree .at { width: 100%; font-size: 12px; color: var(--soft); }
.sd { background: var(--land); border-left: 4px solid var(--landline); border-radius: 6px; padding: 8px 14px; font-variant-numeric: tabular-nums; }
.sp { color: var(--chgline); font-size: 12px; padding: 0 6px; }
.sp::before, .sp::after { content: " ⟷ "; color: var(--soft); }
.ans i { display: block; margin-top: 4px; font-style: normal; font-weight: 600; }
.dot { display: inline-block; width: 12px; height: 12px; border-radius: 3px; margin-right: 5px; vertical-align: -1px; }
"""


def picture_box(title, text, columns, kind):
    """one block of the survey, with the export columns it comes from."""
    cols = "".join(f"<li>{escape(str(c))}</li>" for c in columns)
    more = f"<details><summary>{len(columns)} column{'s' if len(columns) != 1 else ''}</summary><ul>{cols}</ul></details>" if len(columns) else ""
    return f'<div class="box {kind}"><h3>{escape(title)}</h3><p>{escape(text)}</p>{more}</div>'


def picture_flow(survey, table):
    """the blocks of the survey in the order of its questions, read from what each column holds."""
    roles = survey["roles"]
    roles = roles[~roles["note"].astype(str).str.startswith("not known")]
    held = roles["record"].map(what_it_holds)

    def columns(name):
        return list(roles.loc[held == name, "export column"])

    arrow = '<span class="arrow">→</span>'
    dates = sorted({h for h in held if re.fullmatch(r"land in \d{4}", h)}, reverse=survey_direction(survey) == "backward")
    if survey["design"] == "annual":
        boxes = [picture_box(d.capitalize(), "asked at this date", columns(d), "land") for d in dates]
        flow = f'<div class="flow">{arrow.join(boxes)}</div>'
        if columns("a period between two dates"):
            by_kind = defaultdict(list)
            for c in columns("a period between two dates"):
                by_kind[kind_of(period_of(c)[3])].append(c)
            boxes = [picture_box("The change", "its type, year and the land after it", by_kind.pop(None, []), "chg")]
            boxes += [picture_box(KIND_NAMES[k], "what is recorded for this kind of change", by_kind[k], "chg") for k in KIND_NAMES if by_kind.get(k)]
            flow += ('<div class="repeat" style="margin-top:8px"><div class="tag">Between two dates: repeats for each change of each period '
                     '(1, 2 ... in the column names)</div>' + "".join(boxes) + "</div>")
        return flow
    recorded = table[(table["change"] == "yes") | (table["disturbance"] == "yes")]
    most = int(recorded.groupby("plot_id").size().max()) if len(recorded) else 0
    parts = [picture_box(f"Land in {survey['end_year']}", "the end year: where the survey starts", columns("land in the end year"), "land")]
    title = "A change" if columns("a disturbance") else "A change or disturbance"
    group = picture_box(title, "its year, its kind, what is recorded with it", columns("a change"), "chg")
    group += arrow + picture_box("Land before the change", "the land in the year before it", columns("land before a change"), "land")
    parts.append(f'<div class="repeat"><div class="tag">Repeats going back in time. Most in one plot in this file: {most}</div>{group}</div>')
    if columns("a disturbance"):
        parts.append(picture_box("A disturbance", "its year and what is recorded with it", columns("a disturbance"), "chg"))
    if survey.get("previous_year"):
        earlier = [c for c in survey["data"].columns if c.startswith("land_use_latest_")]
        parts.append(picture_box(f"Land in {survey['previous_year']}", "given with the plot: the earlier assessment", earlier, "land"))
    return f'<div class="flow">{arrow.join(parts)}</div>'


def period_years(survey):
    """the periods of the survey and the years of the changes recorded in each: which period a change in a date year belongs to."""
    data = survey["data"]
    found = defaultdict(list)
    for p in survey.get("period_cols", []):
        if p["variable"] == "(years)" and p["y1"]:
            found[(p["y1"], p["y2"])] += [y for y in (year_of(v)[1] for v in data[p["col"]] if given(v)) if y]
    return {k: found[k] for k in sorted(found)}


def picture_periods(survey):
    """the periods in words: the first and last year of each, as the change years recorded show it."""
    periods = period_years(survey)
    if not periods:
        return []
    at_start = sum(1 for (a, b), ys in periods.items() for y in ys if y == a)
    at_end = sum(1 for (a, b), ys in periods.items() for y in ys if y == b)
    names = ", ".join(f"{a}–{b}" for a, b in periods)
    (a, b) = list(periods)[-1]
    if at_end and not at_start:
        return [f"Periods: {names}. The changes recorded go from the year after the start to the end year "
                f"({', '.join(f'{x + 1}–{y}' for x, y in periods)}): a change in {a} belongs to the period that ends in {a}, "
                f"and the land in {a} is the land after it."]
    if at_start and not at_end:
        return [f"Periods: {names}. The changes recorded go from the start year to the year before the end: "
                f"a change in {a} belongs to the period that starts in {a}."]
    if at_start and at_end:
        return [f"Periods: {names}. Changes are recorded both in the first and in the last year of the periods "
                f"({at_start} and {at_end}): the period of a change in a date year is the one in the name of its column."]
    return [f"Periods: {names}."]


def picture_rules(direction, design):
    """how the table is filled, in this survey."""
    if design == "annual":
        return ["The land of each date comes directly from the file (raw).",
                "A change with its year: the land before it lasts until that year, the land after it from that year (filled).",
                "Years between two dates with no change take the land of the dates (filled)."]
    rules = ["The land in the end year comes directly from the file (raw), when the survey asks it.",
             "The year before a change has the land before the change (raw).",
             "The year of a change has the land after the change (filled).",
             "Other years take the land of the years around them, when it is the same (filled)."]
    rules.append("Years before the oldest land take that land." if direction == "backward" else "Years after the newest land take that land.")
    rules.append("change = yes: the land use or subdivision differs from the year before. disturbance = yes: the land stays the same. Never both.")
    return rules


def picture_example(table):
    """one plot with a change and at least one observed (raw) year, year by year, with its notes."""
    changed = set(table.loc[(table["change"] == "yes") | (table["disturbance"] == "yes"), "plot_id"])
    observed = set(table.loc[table["source"] == "raw", "plot_id"])
    best = [p for p in table["plot_id"].drop_duplicates() if p in changed and p in observed]
    plot_id = best[0] if best else (next(iter(changed)) if changed else table["plot_id"].iloc[0])
    plot = table[table["plot_id"] == plot_id]
    cells = []
    for r in plot.to_dict("records"):
        mark = "disturbance" if r["disturbance"] == "yes" else "change" if r["change"] == "yes" else ""
        title = " / ".join(str(r.get(c, "")) for c in LAND if r.get(c, ""))
        cells.append(f'<div class="year {r["source"]}" title="{escape(title)}"><div class="mark">{mark}</div>'
                     f'<div class="cell">{escape(str(r["land_use"]))}</div><div>{r["year"]}</div></div>')
    notes = "".join(f"<li>{r['year']}: {escape(str(r['note']))}</li>" for r in plot.to_dict("records") if given(r.get("note")))
    return plot_id, f'<div class="strip">{"".join(cells)}</div>' + (f"<ul>{notes}</ul>" if notes else "")


def picture_questions(survey):
    """the questions of the export, block by block: type, plots answered, different answers, the answers."""
    q = survey_questions(survey)
    other = q["detail"].astype(str).str.startswith("not known") | (q["what it holds"] == "not used")
    q.loc[other, "what it holds"] = "not used by the tool (kept in the export)"
    q.loc[q["detail"].astype(str).str.startswith("not known"), "detail"] = ""
    q = pd.concat([q[~other], q[other]], ignore_index=True)
    out = []
    for block, part in q.groupby("what it holds", sort=False):
        rows = []
        for r in part.to_dict("records"):
            if not r["plots answered"]:
                continue
            answers = str(r["possible answers (plots)"])
            items = [a for a in answers.split("; ") if a] if r["plots answered"] else []
            shown = "".join(f"<li>{escape(a)}</li>" for a in items[:6])
            if len(items) > 6:
                shown += f'<li><details><summary>{len(items) - 6} more</summary><ul>' + "".join(f"<li>{escape(a)}</li>" for a in items[6:]) + "</ul></details></li>"
            detail = str(r["detail"]) if given(r["detail"]) and str(r["detail"]) != block else ""
            single = ' <span class="tag1">one value only</span>' if r["different answers"] == 1 and r["plots answered"] > 1 else ""
            rows.append(f'<tr class="{"none" if not r["plots answered"] else ""}"><td><code>{escape(str(r["question"]))}</code>'
                        + (f'<div class="det">{escape(detail)}</div>' if detail else "") + f'</td><td>{escape(r["type"])}{single}</td>'
                        f'<td class="n">{r["plots answered"]:,}</td><td class="n">{r["% of plots"]:g}%</td><td class="n">{r["different answers"]:,}</td>'
                        f'<td><ul class="ans">{shown}</ul></td></tr>')
        answered = int((part["plots answered"] > 0).sum())
        empty = [str(x) for x in part.loc[part["plots answered"] == 0, "question"]]
        if empty:
            rows.append(f'<tr class="none"><td colspan="6"><details><summary>{len(empty)} question{"s" if len(empty) != 1 else ""} with no answer</summary>'
                        f'<ul>{"".join(f"<li><code>{escape(x)}</code></li>" for x in empty)}</ul></details></td></tr>')
        out.append(f'<details class="block"{" open" if not block.startswith("not used") else ""}><summary><b>{escape(block.capitalize())}</b> · '
                   f'{len(part)} question{"s" if len(part) != 1 else ""}, {answered} answered</summary>'
                   '<div class="tw"><table class="q"><thead><tr><th>Question</th><th>Type</th><th>Plots answered</th><th>% of plots</th>'
                   f'<th>Different answers</th><th>Answers (plots)</th></tr></thead><tbody>{"".join(rows)}</tbody></table></div></details>')
    return "".join(out)


def pick_examples(table):
    """a plot with a deforestation and a plot with a degradation (a change or disturbance within forest land), with one event
    and recorded years when there is one; otherwise any plot with a change."""
    o = table.sort_values(["plot_id", "year"])
    prev = o.groupby("plot_id", sort=False)["land_use"].shift().fillna("")
    event = o["change"].eq("yes") | o["disturbance"].eq("yes")
    n_events = event.groupby(o["plot_id"]).sum()
    raw = o["source"].eq("raw").groupby(o["plot_id"]).sum()
    noted = o["note"].ne("").groupby(o["plot_id"]).any()           # a clean plot first: nothing noted
    defo = o[event & prev.eq(FOREST_LAND) & o["land_use"].ne(FOREST_LAND) & o["land_use"].ne("")]
    degr = o[event & prev.eq(FOREST_LAND) & o["land_use"].eq(FOREST_LAND)]
    degr = degr.assign(said=degr["change_type"].map(lambda t: kind_of(t) == "degradation")).sort_values("said", ascending=False, kind="stable")
    out = []
    for name, cand in (("A deforestation", defo), ("A degradation", degr)):
        ids = [p for p in cand["plot_id"].drop_duplicates()]
        said = set(cand.loc[cand["change_type"].map(lambda t: kind_of(t) == ("loss" if name == "A deforestation" else "degradation")), "plot_id"])
        ids.sort(key=lambda p: (p not in said, n_events[p] != 1, noted[p], -raw[p]))   # its type says it first
        if ids:
            out.append((name, ids[0]))
    if not out:
        ids = list(o.loc[event, "plot_id"].drop_duplicates()) or list(o["plot_id"][:1])
        out.append(("An example", ids[0]))
    return out


def picture_answers(survey, plot_id):
    """the answers of the file for one plot, grouped by date and period: what the years below are built from."""
    data, id_col = survey["data"], survey["id_col"]
    row = data[data[id_col].astype(str) == str(plot_id)]
    if row.empty:
        return ""
    row = row.iloc[0]
    groups, zeros = {}, defaultdict(list)
    periods = set(period_years(survey))
    for r in survey["roles"].to_dict("records"):
        c, rec = r["export column"], str(r["record"])
        if rec in ("-", "P") or str(r["note"]).startswith("not known") or not given(row.get(c)):
            continue
        if rec.startswith("S@"):
            key = (int(rec[2:]), 0, rec[2:])
        elif rec.startswith("PERIOD"):
            y1, y2, k, _ = period_of(c)
            key = (y1 or 0, 1, f"{y1}–{y2}") if (y1, y2) in periods else (9998, 1, f"{y1}–{y2}")   # the whole time: last
        else:
            key = (9999, 2, what_it_holds(rec))
        v = code(row, c)
        if v in ("0", "0.0"):
            zeros[key].append(c)
        else:
            groups.setdefault(key, []).append((c, v))
    for key in zeros:
        groups.setdefault(key, [])
    cards = []
    for key in sorted(groups):
        items = "".join(f"<li><code>{escape(c)}</code> {escape(v)}</li>" for c, v in groups[key])
        if zeros[key]:
            items += f'<li class="z"><details><summary>{len(zeros[key])} answered 0</summary>{escape(", ".join(zeros[key]))}</details></li>'
        cards.append(f'<div class="ans"><b>{escape(key[2])}</b><ul>{items}</ul></div>')
    return f'<div class="answers">{"".join(cards)}</div>'


def picture_strip(table, plot_id):
    """the years of one plot as built: land, raw or filled, what happened, and the canopy."""
    plot = table[table["plot_id"] == plot_id].sort_values("year")
    cells = []
    for r in plot.to_dict("records"):
        mark = "disturbance" if r["disturbance"] == "yes" else (r.get("change_type") or "change") if r["change"] == "yes" else ""
        title = " / ".join(str(r.get(c, "")) for c in LAND if r.get(c, ""))
        cover = r.get("canopy_cover") or r.get("tree_cover") or ""
        cells.append(f'<div class="year {r["source"]}" title="{escape(title)}"><div class="mark">{escape(str(mark))}</div>'
                     f'<div class="cell">{escape(str(r["subdivision"] or r["land_use"]))}</div><div>{r["year"]}</div>'
                     f'<div class="cov">{escape(str(cover)) + "%" if cover != "" else ""}</div></div>')
    notes = "".join(f"<li>{r['year']}: {escape(str(r['note']))}</li>" for r in plot.to_dict("records") if given(r.get("note")))
    return f'<div class="strip">{"".join(cells)}</div>' + (f"<ul>{notes}</ul>" if notes else "")


def picture_examples(survey, table):
    out = []
    for name, plot_id in pick_examples(table):
        out.append(f'<h3 class="ex">{escape(name)}: plot {escape(str(plot_id))}</h3>'
                   f'<p class="hint">The answers in the file:</p>{picture_answers(survey, plot_id)}'
                   f'<p class="hint">The years built from them (the land, what happened, the canopy):</p>{picture_strip(table, plot_id)}')
    return "".join(out)


def answer_stem(column):
    """the question without its number and period: 'fcc_after_deg1_2016_2024' -> 'fcc_after_deg'."""
    return PERIOD_RE.sub("", str(column)).rstrip("_- ")


def picture_schema(survey):
    """the survey as a schema: the land at the dates, the periods, the changes of each period with the answers of each type."""
    if not survey.get("numbered_periods"):
        return ""
    dates = sorted(survey["years"])
    periods = list(period_years(survey)) or sorted({(p["y1"], p["y2"]) for p in survey["period_cols"] if p["k"]})
    at_date = list(dict.fromkeys(re.sub(r"_?(19|20)\d\d$", "", c) for y in dates for c, lvl, var in survey["years"][y]))
    data, numbers = survey["data"], set()
    group = lambda lab_: kind_of(lab_) or lab_            # the kind of a type of change (or the type itself)
    with_type, labels = defaultdict(Counter), Counter()
    by_key = defaultdict(list)
    for p in survey["period_cols"]:
        if p["k"]:
            numbers.add(p["k"])
            by_key[(p["y1"], p["y2"], p["k"])].append(p)
    for key, cols in by_key.items():                      # which type of change each answer is given with, in the data
        lab_col = next((p["col"] for p in cols if p["variable"] == "change_label"), None)
        if lab_col is None:
            continue
        labs = data[lab_col].map(lambda v: str(v).strip() if given(v) else "")
        labels.update(l for l in labs if l)
        for p in cols:
            v = data[p["col"]]
            ok = v.map(given) & ~v.astype(str).str.strip().isin(["0", "0.0"]) & labs.ne("")
            with_type[answer_stem(p["col"])].update(labs[ok].map(group))
    base, own = [], defaultdict(list)
    tag_kind = {answer_stem(p["col"]): kind_of(p["tag"]) for p in survey["period_cols"] if p["k"] and kind_of(p["tag"])}
    for stem, n in with_type.items():
        if stem in tag_kind:                              # the name says its kind (deg, def, gain ...): the checks use the same
            own[tag_kind[stem]].append(stem)
            continue
        top, m = n.most_common(1)[0] if n else (None, 0)
        if top is not None and m >= 0.9 * sum(n.values()) and len(set(map(group, labels))) > 1 and \
                not any(p["variable"] in ("change_label", "(years)") for p in survey["period_cols"] if answer_stem(p["col"]) == stem):
            own[top].append(stem)
        else:
            base.append(stem)
    whole = [p["col"] for p in survey["period_cols"] if not p["k"] and p["variable"] == "change_label"]
    flags = sorted({answer_stem(c) for c, *_ in survey.get("period_flags", [])})
    base = [b for b in base if b not in flags]
    boxes = "".join(f'<div class="sd"><b>{y}</b></div>' + (f'<div class="sp">{y}–{z}</div>' if (y, z) in periods else '<div class="sp"></div>')
                    for y, z in zip(dates, dates[1:])) + (f'<div class="sd"><b>{dates[-1]}</b></div>' if dates else "")
    rows = []
    order = [k for k in KIND_NAMES if k in own or any(group(t) == k for t in labels)] + \
            [g for g in dict.fromkeys(group(t) for t, _ in labels.most_common()) if g not in KIND_NAMES]
    for g in order:
        types = ", ".join(f"{t} ({n:,})" for t, n in labels.most_common() if group(t) == g)
        rows.append(f'<tr><td><b>{escape(KIND_NAMES.get(g, g))}</b><div class="det">{escape(types)}</div></td>'
                    f'<td>{escape(", ".join(sorted(own.get(g, []))) or "only the answers of every change")}</td></tr>')
    base_txt = ", ".join(sorted(base))
    nums = " or ".join(str(n) for n in sorted(numbers))
    return (f'<div class="strip schema">{boxes}</div>'
            f'<p class="hint">At each date: {escape(", ".join(at_date))}.</p>'
            f'<div class="box chg" style="max-width:none"><h3>In each period: change {nums}</h3>'
            f'<p>{("yes / no question: " + escape(", ".join(flags)) + ". ") if flags else ""}Every change: {escape(base_txt)}. '
            f'The type of the change says which other answers belong to it (same number, same period).</p></div>'
            f'<div class="tw"><table class="q"><thead><tr><th>Type of change</th><th>Its own answers</th></tr></thead><tbody>{"".join(rows)}</tbody></table></div>'
            + (f'<p class="hint">Not used: {escape(", ".join(whole))}, one label for the whole period (a summary, not a change).</p>' if whole else ""))


def sample_answers(survey, row):
    """the answers of one plot as the survey asks them: the dates; then each period, change by change, with the answers of
    its type, and the answers given where there is no change of their type (not used)."""
    roles = {r["export column"]: r for r in survey["roles"].to_dict("records")}
    zero = lambda c: code(row, c) in ("0", "0.0")
    li = lambda c, extra="": f"<li><code>{escape(c)}</code> {escape(code(row, c))}{extra}</li>"
    cards = []
    for y in sorted(survey["years"]):
        cols = [c for c, lvl, var in survey["years"][y] if given(row.get(c))]
        if cols:
            cards.append(f'<div class="ans"><b>{y}</b><ul>{"".join(li(c) for c in cols)}</ul></div>')
    numbered = defaultdict(list)
    for p in survey["period_cols"]:
        if p["k"]:
            numbered[(p["y1"], p["y2"])].append(p)
    flags = {(f1, f2, k): col for col, f1, f2, k in survey.get("period_flags", [])}
    for (y1, y2) in sorted(numbered):
        parts = []
        for k in sorted({p["k"] for p in numbered[(y1, y2)]}):
            cols = [p for p in numbered[(y1, y2)] if p["k"] == k and given(row.get(p["col"]))]
            flag = flags.get((y1, y2, k))
            label = next((code(row, p["col"]) for p in cols if p["variable"] == "change_label"), "")
            base = [p for p in cols if not kind_of(p["tag"])]
            own = [p for p in cols if kind_of(p["tag"]) and label and kind_of(p["tag"]) == kind_of(label)]
            rest = [p for p in cols if kind_of(p["tag"]) and p not in own]
            rest_kinds = defaultdict(list)
            for p in rest:
                rest_kinds[kind_of(p["tag"])].append(p)
            rest = [p for ps in rest_kinds.values() if not all(zero(q["col"]) for q in ps) for p in ps]   # all 0 = not answered
            if not (base or own or rest) and not (flag and yes_no(row.get(flag)) == "yes"):
                continue
            head = f"Change {k}" + (f": {label}" if label else "")
            items = (li(flag) if flag and given(row.get(flag)) else "") + "".join(li(p["col"]) for p in base + own)
            if rest:
                items += ('<li class="z"><span class="tag1">not used: no change of this type here</span></li>'
                          + "".join(li(p["col"]) for p in rest))
            parts.append(f"<i>{escape(head)}</i><ul>{items}</ul>")
        if parts:
            cards.append(f'<div class="ans"><b>{y1}–{y2}</b>{"".join(parts)}</div>')
    whole = [p["col"] for p in survey["period_cols"] if not p["k"] and given(row.get(p["col"]))]
    if whole:
        cards.append(f'<div class="ans"><b>The whole period (not used)</b><ul>{"".join(li(c) for c in whole)}</ul></div>')
    if not survey.get("numbered_periods"):
        return None
    return f'<div class="answers">{"".join(cards)}</div>'


def cover_of(r):
    """the canopy cover of a row (forest) or its tree cover (other land): '' when none."""
    return str(r.get("canopy_cover") or r.get("tree_cover") or "")


def cover_spans(recs):
    """the canopy over a stretch of the same land: ', canopy 92' or ', canopy 100 (1990–2007), 92 (2008–2017)'."""
    spans = []
    for r in recs:
        c = cover_of(r)
        if spans and spans[-1][0] == c:
            spans[-1][2] = r["year"]
        else:
            spans.append([c, r["year"], r["year"]])
    spans = [x for x in spans if x[0]]
    if not spans:
        return ""
    if len(spans) == 1:
        return f", canopy {spans[0][0]}"
    return ", canopy " + ", ".join(f"{c} ({a}–{b})" if a != b else f"{c} ({a})" for c, a, b in spans)


def sample_runs(plot):
    """how the years of one plot were built, in a few lines: each stretch of the same land, each change."""
    out, recs = [], plot.to_dict("records")
    land = lambda r: " / ".join(str(r[c]) for c in LAND if r.get(c)) or "no land"
    i = 0
    while i < len(recs):
        j = i
        while j + 1 < len(recs) and land(recs[j + 1]) == land(recs[i]) and recs[j + 1]["change"] != "yes" and recs[j + 1]["disturbance"] != "yes":
            j += 1
        r0, r1 = recs[i], recs[j]
        if r0["change"] == "yes" or r0["disturbance"] == "yes":
            what = r0.get("change_type") or ("disturbance" if r0["disturbance"] == "yes" else "change")
            drivers = "; ".join(f"{c.replace('_', ' ')}: {r0[c]}" for c in r0 if (c.startswith("driver") or c in ("gain_origin", "temporary_loss")) and r0[c])
            prev = recs[i - 1] if i else {}
            canopy = (f", canopy {cover_of(prev)} → {cover_of(r0)}" if cover_of(prev) and cover_of(r0) and cover_of(prev) != cover_of(r0)
                      else f", canopy {cover_of(r0)}" if cover_of(r0) else "")
            out.append(f"{r0['year']}: {what}, land after: {land(r0)}{canopy}" + (f" ({drivers})" if drivers else ""))
            if j > i:
                out.append(f"{recs[i + 1]['year']}–{r1['year']}: {land(r1)}{cover_spans(recs[i + 1:j + 1])} " + sample_sources(recs[i + 1:j + 1]))
        else:
            span = f"{r0['year']}–{r1['year']}" if j > i else f"{r0['year']}"
            out.append(f"{span}: {land(r0)}{cover_spans(recs[i:j + 1])} " + sample_sources(recs[i:j + 1]))
        i = j + 1
    return out


def sample_sources(recs):
    raw = [str(r["year"]) for r in recs if r["source"] == "raw"]
    filled = sum(r["source"] == "filled" for r in recs)
    empty = sum(r["source"] == "empty" for r in recs)
    parts = ([f"raw in {', '.join(raw)}"] if raw else []) + ([f"{filled} year{'s' if filled != 1 else ''} filled"] if filled else []) \
        + ([f"{empty} empty"] if empty else [])
    return f"({'; '.join(parts)})" if parts else ""


def write_sample(path, survey, table, plot_id):
    """the page of one sample: its answers in the file, the years built from them, how, and the messages."""
    data, id_col = survey["data"], survey["id_col"]
    found = data[data[id_col].astype(str).str.strip() == str(plot_id).strip()]
    plot = table[table["plot_id"].astype(str) == str(plot_id).strip()].sort_values("year")
    if found.empty or plot.empty:
        return False
    row = found.iloc[0].to_dict()
    answers = sample_answers(survey, row) or picture_answers(survey, plot["plot_id"].iloc[0])
    probs = land_problems(plot)
    msgs = "".join(f"<li>{escape(r['level'][:1])} {escape(str(r['detail']))}</li>" for r in probs.to_dict("records")) or "<li>no message</li>"
    runs = "".join(f"<li>{escape(x)}</li>" for x in sample_runs(plot))
    name = survey["survey_id"].replace("_", " ")
    xy = plot[["x", "y"]].iloc[0].tolist() if {"x", "y"} <= set(plot.columns) else ["", ""]
    page = f"""<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Sample {escape(str(plot_id))}</title><style>{PAGE_STYLE}</style></head>
<body><main>
<h1>{escape(name)}: sample {escape(str(plot_id))}</h1>
<div class="sub">File: {escape(str(survey.get('file', '')))} · {escape(str(id_col))} = {escape(str(plot_id))} · location {escape(str(xy[0]))}, {escape(str(xy[1]))}</div>
<h2>1 · The answers in the file</h2>
<p class="hint">Only the questions answered. In each period, every change with the answers of its type; an answer of a type with no such change in that period is not used.</p>
{answers}
<h2>2 · The years built from them</h2>
{picture_strip(table, plot["plot_id"].iloc[0])}
<div class="legend"><span><i class="dot" style="background:var(--raw)"></i>raw (a date of the survey)</span><span><i class="dot" style="background:var(--filled)"></i>filled</span>
<span><i class="dot" style="background:var(--empty)"></i>empty</span></div>
<h2>3 · How</h2>
<ul class="rules">{runs}</ul>
<h2>4 · Messages for this sample</h2>
<ul class="rules">{msgs}</ul>
</main></body></html>
"""
    with open(path, "w", encoding="utf-8") as f:
        f.write(page)
    return True


def write_picture(path, survey, table, direction):
    """an auxiliary page: the structure of this survey, how the table is filled, one plot as an example."""
    name = survey["survey_id"].replace("_", " ")
    tool = {"CE": "Collect Earth", "CEO": "Collect Earth Online"}[survey["tool"]]
    counts = table["source"].value_counts()
    facts = [("plots", table["plot_id"].nunique()), ("rows", len(table)), ("raw", counts.get("raw", 0)), ("filled", counts.get("filled", 0)),
             ("empty", counts.get("empty", 0)), ("change = yes", (table["change"] == "yes").sum()), ("disturbance = yes", (table["disturbance"] == "yes").sum())]
    facts = "".join(f'<div class="fact"><b>{int(v):,}</b><span>{k}</span></div>' for k, v in facts)
    rules = "".join(f"<li>{escape(r)}</li>" for r in picture_periods(survey) + picture_rules(direction, survey["design"]))
    page = f"""<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>{escape(name)} survey structure</title><style>{PAGE_STYLE}</style></head>
<body><main>
<h1>{escape(name)}: the structure of the survey</h1>
<div class="sub">{tool} · {escape(survey['design_text'])} · years {table['year'].min()}-{table['year'].max()} · direction: {direction}</div>
<div class="sub">File: {escape(str(survey.get('file', '')))}{' · sheet ' + escape(str(survey['sheet'])) if survey.get('sheet') else ''} · plot id: {escape(str(survey['id_col']))} · location: {escape(', '.join(map(str, survey['roles'].loc[survey['roles']['variable'].isin(['x', 'y']), 'export column'])))}</div>
<div class="facts">{facts}</div>
<h2>How the survey asks</h2>
<h3 class="ex">The classes and their attributes</h3>
{picture_classes(survey, table)}
{('<h3 class="ex">The schema</h3>' + picture_schema(survey) + '<h3 class="ex">The blocks</h3>') if survey.get("numbered_periods") else ""}
{picture_flow(survey, table)}
<p class="hint">Open "columns" in a box to see where it comes from in the export.</p>
<h2>How the table is filled</h2>
<ul class="rules">{rules}</ul>
<h2>Examples</h2>
{picture_examples(survey, table)}
<div class="legend"><span><i class="dot" style="background:var(--raw)"></i>raw</span><span><i class="dot" style="background:var(--filled)"></i>filled</span>
<span><i class="dot" style="background:var(--empty)"></i>empty</span></div>
<h2>The questions, one by one</h2>
<p class="hint">Every question of the export: what it holds, its type, how many plots answered it, and the answers given (with the number of plots). The questions no plot answered are listed at the end of each block.</p>
{picture_questions(survey)}
</main></body></html>
"""
    with open(path, "w", encoding="utf-8") as f:
        f.write(page)


import datetime, os, subprocess, sys
for tool, needed in (("xlsxwriter", "xlsxwriter"), ("reverse_geocoder", "reverse_geocoder"), ("pycountry", "pycountry")):
    try:
        __import__(tool)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", needed])
previous_year = 0   # only if Step 3 asks for it
print('✅ Setup complete.')



# ----------------------------------------------------------------------------------------------- v6: errors, lookup, classes, matrices, IPCC

import unicodedata as _ud

IPCC_ORDER = ["Forest land", "Cropland", "Grassland", "Wetlands", "Settlements", "Other land"]
IPCC_SHORT = dict(zip(IPCC_ORDER, ["FL", "CL", "GL", "WL", "SL", "OL"]))
IPCC_WORDS = {
    "Forest land": ["forest", "forest land", "forestland", "forests", "foret", "forets", "terres forestieres", "terre forestiere", "f"],
    "Cropland": ["cropland", "crop land", "croplands", "agriculture", "agricultural land", "terres cultivees", "terre cultivee", "terres agricoles", "cultures", "c"],
    "Grassland": ["grassland", "grass land", "grasslands", "prairie", "prairies", "paturage", "paturages", "parcours", "terres gramineennes", "g"],
    "Wetlands": ["wetland", "wetlands", "wet land", "terres humides", "terre humide", "zones humides", "zone humide", "w"],
    "Settlements": ["settlement", "settlements", "etablissement", "etablissements", "etablissements humains", "etablissement humain", "zones baties", "bati", "s"],
    "Other land": ["other land", "other lands", "otherland", "autres terres", "autre terre", "autres", "o"],
}
IPCC_SUBCATEGORIES = {
    "Forest land": ["Managed Forest Land", "Unmanaged Forest Land"], "Cropland": ["Cropland Annual Crops", "Cropland Perennial Crops"],
    "Grassland": ["Managed Grassland", "Unmanaged Grassland"], "Wetlands": ["Managed Wetlands", "Unmanaged Wetlands"],
    "Settlements": ["Settlements (Treed)", "Settlements (Other)"], "Other land": ["Managed Other Land", "Unmanaged Other Land"],
}
LOOKUP_CHOICES = ["Degradation", "Enhancement", "Stable forest", "Not possible"]
LOOKUP_WORDS = {"degradation": "Degradation", "enhancement": "Enhancement", "amelioration": "Enhancement", "restauration": "Enhancement",
                "restoration": "Enhancement", "regrowth": "Enhancement", "regeneration": "Enhancement", "no change": "Stable forest", "stable": "Stable forest",
                "stable forest": "Stable forest", "foret stable": "Stable forest", "pas de changement": "Stable forest", "not possible": "Not possible", "pas possible": "Not possible", "impossible": "Not possible"}
FOREST_LAND = "Forest land"
LOOKUP_SHOWN = {}          # (from_sub, to_sub) -> the word written in the lookup file (shown in the rule column)


def simple_words(value):
    text = _ud.normalize("NFKD", str(value).strip().lower())
    text = "".join(c for c in text if not _ud.combining(c))
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9]+", " ", text)).strip()


def ipcc_category_of(value):
    """the IPCC category of a land use, read from its name or Collect Earth letter; '' if not recognised."""
    name = simple_words(value)
    found = [cat for cat, names in IPCC_WORDS.items() if name in names or name == simple_words(cat)]
    return found[0] if len(found) == 1 else ""


def same_id(value):
    return re.sub(r"\.0$", "", str(value).strip())


def write_error_files(survey, report, country, stamp, folder):
    """Step 3: the list of errors with a description, and the original rows of the plots to correct. Returns the paths."""
    errors = report.rename(columns={"check": "type of error", "detail": "description"})[["type of error", "level", "plot", "description"]]
    errors_csv = os.path.join(folder, f"{country}_step3_errors_{stamp}.csv")
    errors.to_csv(errors_csv, index=False, encoding="utf-8-sig")
    stops = errors[errors["level"].str.startswith("⛔")]
    if not len(stops):
        return errors_csv, None
    data = survey["data"]
    ids = data[survey["id_col"]].map(same_id)
    why = stops.groupby(stops["plot"].map(same_id))["description"].agg(" | ".join)
    keep = ids.isin(set(why.index) - {""})
    no_id = {int(m.group(1)) - 2: d for d in stops.loc[stops["plot"].astype(str) == "", "description"]
             for m in [re.match(r"row (\d+) of the export", d)] if m}
    rows = data.loc[keep | data.index.isin(no_id)].copy()
    rows.insert(0, "errors", [no_id.get(i, why.get(ids[i], "")) for i in rows.index])
    rows.insert(1, "row_in_export", rows.index + 2)
    correct_csv = os.path.join(folder, f"{country}_step3_to_correct_{stamp}.csv")
    rows.to_csv(correct_csv, index=False, encoding="utf-8-sig")
    return errors_csv, correct_csv


def land_classes(table):
    """every land use and subdivision found: its IPCC category and the plots in the first and last year."""
    t = table[table["land_use"].map(given)].copy()
    t["sub"] = t["subdivision"].where(t["subdivision"].map(given), t["land_use"]) if "subdivision" in t else t["land_use"]
    first, last = t["year"].min(), t["year"].max()
    out = t.groupby(["land_use", "sub"]).agg(**{f"plots in {first}": ("year", lambda y: (y == first).sum()), f"plots in {last}": ("year", lambda y: (y == last).sum())}).reset_index()
    out = out.rename(columns={"sub": "subdivision"})
    out.insert(2, "IPCC category", out["land_use"].map(ipcc_category_of))
    out.insert(3, "IPCC subcategory", "")
    out["_o"] = out["IPCC category"].map({c: i for i, c in enumerate(IPCC_ORDER)}).fillna(9)
    return out.sort_values(["_o", "subdivision"]).drop(columns="_o").reset_index(drop=True)


def land_pairs(pairs, classes):
    """the pairs of years with the IPCC category on both sides; pairs with land missing on one side are left out."""
    p = pairs.copy()
    for side in ("from", "to"):
        sub = p[f"{side}_subdivision"] if f"{side}_subdivision" in p else p[f"{side}_land_use"]
        p[f"{side}_sub"] = sub.where(sub.map(given), p[f"{side}_land_use"])
    p = p[p["from_land_use"].map(given) & p["to_land_use"].map(given)].copy()
    cat = dict(zip(zip(classes["land_use"], classes["subdivision"]), classes["IPCC category"]))
    p["from_ipcc"] = [cat.get(k, "") for k in zip(p["from_land_use"], p["from_sub"])]
    p["to_ipcc"] = [cat.get(k, "") for k in zip(p["to_land_use"], p["to_sub"])]
    return p.reset_index(drop=True)


def complete_years(lp):
    """the matrices need the same plots in every year: only pairs of years with (nearly) all the plots, and only plots in all of them."""
    n = lp.groupby("from_year")["plot_id"].nunique()
    years = sorted(n.index[n >= 0.9 * n.max()])
    kept = lp[lp["from_year"].isin(years)]
    full = kept.groupby("plot_id")["from_year"].nunique()
    plots = set(full.index[full == len(years)])
    out = kept[kept["plot_id"].isin(plots)].reset_index(drop=True)
    text = f"Years used: {years[0]}-{max(out['to_year'])}, {len(plots):,} plots."
    if len(out) < len(lp):
        text += f" Left out: {len(lp) - len(out):,} pairs of years that do not have all the plots."
    return out, text


def forest_changes(lp):
    a, b = ("from_m", "to_m") if "from_m" in lp else ("from_sub", "to_sub")
    f = lp[(lp["from_ipcc"] == FOREST_LAND) & (lp["to_ipcc"] == FOREST_LAND) & (lp[a] != lp[b])]
    return f.groupby([a, b]).size().rename_axis(["from_sub", "to_sub"])


def make_lookup_template(path, classes, lp, prefill=None):
    """the Excel file of the country's rules: one row per change between two forest types. A sheet 'classes' is added only
    when a land name is not recognised as an IPCC category, to write it there."""
    found = forest_changes(lp)
    types = list(classes.loc[classes["IPCC category"] == FOREST_LAND, "subdivision"])
    rows = [{"from_sub": a, "to_sub": b, "found in your data": "yes" if (a, b) in found.index else "no", "plots with this change": int(found.get((a, b), 0))}
            for a in types for b in types if a != b]
    rows.sort(key=lambda r: -r["plots with this change"])
    unknown = classes[classes["IPCC category"].map(lambda v: not given(v))][["subdivision", "land_use", "IPCC category"]]
    guide = [("What it is", "The country's rules for the changes between two forest types. The notebook does not guess them."),
             ("Sheet lookup", "One row per change between two forest types. Fill the yellow column at least where 'found in your data' = yes."),
             ("Degradation", "The forest becomes poorer, e.g. dense forest to open forest."), ("Enhancement", "The forest becomes richer, e.g. open forest to dense forest."),
             ("Stable forest", "The forest stays forest, no REDD+ activity: the types recorded each year are kept as they are."),
             ("Not possible", "The change cannot happen: the types recorded are kept (the most recent class is taken as right), no REDD+ activity (stable forest), the plot is listed to be reviewed."),
             ("Your own file", "A lookup the country already has works too: it needs a sheet 'lookup' with the columns from_sub, to_sub and Transition within forest."),
             ("Then", "Save the file. In Step 6 choose 'use my filled file'.")]
    if len(unknown):
        guide.insert(2, ("Sheet classes", "Land names the notebook does not recognise: write their IPCC category in the yellow column."))
    with pd.ExcelWriter(path, engine="xlsxwriter") as w:
        book = w.book
        head = book.add_format({"bold": True, "bg_color": "#DCE6F1", "border": 1, "text_wrap": True, "valign": "top"})
        yellow = book.add_format({"bg_color": "#FFF2CC", "border": 1})
        pd.DataFrame(guide, columns=["", "how to fill it"]).to_excel(w, sheet_name="README", index=False)
        w.sheets["README"].set_column(0, 0, 18); w.sheets["README"].set_column(1, 1, 120)
        look = pd.DataFrame(rows, columns=["from_sub", "to_sub", "found in your data", "plots with this change"])
        look["Transition within forest"] = [(prefill or {}).get((a, b), "") for a, b in zip(look["from_sub"], look["to_sub"])]
        look["comment"] = ""
        look.to_excel(w, sheet_name="lookup", index=False)
        s = w.sheets["lookup"]
        for c, name in enumerate(look.columns):
            s.write(0, c, name, head)
        s.set_column(0, 5, 26)
        for r in range(1, len(look) + 1):
            s.write(r, 4, look.iat[r - 1, 4], yellow)
        if len(look):
            s.data_validation(1, 4, len(look), 4, {"validate": "list", "source": LOOKUP_CHOICES})
        if len(unknown):
            unknown.to_excel(w, sheet_name="classes", index=False)
            s = w.sheets["classes"]
            for c, name in enumerate(unknown.columns):
                s.write(0, c, name, head)
            s.set_column(0, 2, 24)
            for r in range(1, len(unknown) + 1):
                s.write_blank(r, 2, None, yellow)
            s.data_validation(1, 2, len(unknown), 2, {"validate": "list", "source": IPCC_ORDER})


def lookup_key(land, subdivision):
    """a class of a filled file: a file filled earlier may still have the survey's code for the land (F, Forestland, ...)."""
    land, subdivision = str(land).strip(), str(subdivision).strip()
    name = ipcc_category(land) or land
    return name, name if subdivision == land else subdivision


def read_lookup(source, classes):
    """the filled file: (rules, classes). It stops on a word it does not know or on two rules for the same change."""
    book = excel_of(source)
    if "lookup" not in book.sheet_names:
        raise ValueError("⛔ The file has no sheet 'lookup'. Use the template made by this step.")
    look = book.parse("lookup", dtype=str)
    col = next((c for c in look.columns if simple_words(c).startswith("transition")), None)
    col = col or next((c for c in look.columns if simple_words(c) == "classification"), None)   # the Extract Tool's lookup
    if col is None or not {"from_sub", "to_sub"} <= set(look.columns):
        raise ValueError("⛔ The sheet 'lookup' needs the columns from_sub, to_sub and Transition within forest (or classification).")
    rules, bad = {}, []
    LOOKUP_SHOWN.clear()
    for r in look.to_dict("records"):
        if not given(r[col]):
            continue
        word = LOOKUP_WORDS.get(simple_words(r[col]))
        key = (str(r["from_sub"]).strip(), str(r["to_sub"]).strip())
        if word is None:
            bad.append(f"{key[0]} > {key[1]}: '{r[col]}' is not one of {LOOKUP_CHOICES}")
        elif rules.get(key, word) != word:
            bad.append(f"{key[0]} > {key[1]}: two different rules")
        else:
            rules[key] = word
            LOOKUP_SHOWN[key] = str(r[col]).strip()
    if bad:
        raise ValueError("⛔ Correct the lookup file:\n   " + "\n   ".join(bad[:20]))
    out = classes.copy()
    if "classes" in book.sheet_names:
        filled = book.parse("classes", dtype=str)
        for c in ("IPCC category",):
            if c in filled and {"land_use", "subdivision"} <= set(filled.columns):
                m = {lookup_key(a, b): v for a, b, v in zip(filled["land_use"], filled["subdivision"], filled[c]) if given(v)}
                out[c] = [m.get((a, b), v) for a, b, v in zip(out["land_use"], out["subdivision"], out[c])]
    return rules, out


def apply_rules(lp, rules):
    """the rule of each change between two forest types (the lookup); the forest types recorded are kept in every case
    (v19: 'Stable forest' and 'Not possible' no longer keep the earlier type)."""
    lp = lp.sort_values(["plot_id", "from_year"]).reset_index(drop=True)
    fs, ts = ("from_m", "to_m") if "from_m" in lp else ("from_sub", "to_sub")      # merged classes (Step 6 merge), if any
    lp["from_adj"], lp["to_adj"] = lp[fs], lp[ts]
    lp["rule"], lp["rule_word"] = "", ""
    if not rules:
        return lp
    hit = (lp["from_ipcc"] == FOREST_LAND) & (lp["to_ipcc"] == FOREST_LAND) & (lp[fs] != lp[ts])
    groups = lp[lp["plot_id"].isin(set(lp.loc[hit, "plot_id"]))].groupby("plot_id", sort=False)
    from_adj, to_adj, rule, shown = lp["from_adj"].to_dict(), lp["to_adj"].to_dict(), lp["rule"].to_dict(), lp["rule_word"].to_dict()
    for _, g in groups:
        now, year = None, None
        for i, r in zip(g.index, g.to_dict("records")):
            start = now if now is not None and year == r["from_year"] else r[fs]
            from_adj[i], end = start, r[ts]
            if r["from_ipcc"] == FOREST_LAND and r["to_ipcc"] == FOREST_LAND and start != end:
                rule[i] = rules.get((start, end), "no rule")
                shown[i] = LOOKUP_SHOWN.get((start, end), rule[i])
            to_adj[i], now, year = end, end, r["to_year"]
    lp["from_adj"], lp["to_adj"], lp["rule"], lp["rule_word"] = pd.Series(from_adj), pd.Series(to_adj), pd.Series(rule), pd.Series(shown)
    return lp



def history_reported(history, lp):
    """the history of Step 5 after the country's rules (reclassify, merge; the lookup does not change the forest types
    recorded): the same columns as the Step 5 history, lu_ipcc_ and lu_sub_ of every year replaced by the
    classes reported (Step 6), cc_ and the other columns as recorded."""
    out = history.copy()
    lp = lp.sort_values(["plot_id", "from_year"])
    first = lp.groupby("plot_id", sort=False).first()
    ipcc = lp.pivot(index="plot_id", columns="to_year", values="to_ipcc")
    sub = lp.pivot(index="plot_id", columns="to_year", values="to_adj")
    y0 = int(lp["from_year"].min())
    ipcc[y0], sub[y0] = first["from_ipcc"], first["from_adj"]
    key = out["plot_id"]
    for y in sorted(ipcc.columns):
        if f"lu_ipcc_{y}" in out:
            out[f"lu_ipcc_{y}"] = key.map(ipcc[y]).values
            out[f"lu_sub_{y}"] = key.map(sub[y]).values
    return out


def classify_pairs(lp, has_rules):
    """level 1, 2, 3 of each plot and pair of years, from the land and from what was recorded."""
    def word(v):
        return str(v).strip().replace(" ", "_")

    def driver_of(r, *names):
        return next((word(r[n]) for n in names if n in r and given(r[n])), "")

    out = []
    for r in lp.to_dict("records"):
        fi, ti, fs, ts = r["from_ipcc"], r["to_ipcc"], word(r["from_adj"]), word(r["to_adj"])
        how, d = "", driver_of(r, "driver_change", "driver_unclear")
        if fi == FOREST_LAND and ti != FOREST_LAND:
            cls, l2, l3 = "Deforestation", f"Deforestation_in_{fs}", f"Deforestation_in_{fs}_to_{ts}"
        elif fi != FOREST_LAND and ti == FOREST_LAND:
            cls, l2, l3 = "Afforestation", f"Afforestation_{ts}", f"Afforestation_{ts}_from_{fs}"
        elif fi != FOREST_LAND:
            same = fi == ti and fs == ts
            cls, l2, l3 = ("Stable non-forest", "Stable_nonforest", "Stable_nonforest") if same else ("Non-forest change", f"{word(fi)}_to_{word(ti)}", f"{fs}_to_{ts}")
        elif r["rule"] == "no rule" and has_rules:
            cls, l2, l3 = "To review", f"To_review_in_{word(r['from_adj'])}", f"To_review_{word(r['from_adj'])}_to_{word(r['to_sub'])}"
        elif fs != ts and r["rule"] not in ("Stable forest", "Not possible"):
            cls = r["rule"] if r["rule"] in ("Degradation", "Enhancement") else "Forest subclass change"
            l2, l3 = f"{word(cls)}_in_{fs}", f"{word(cls)}_in_{fs}_to_{ts}"
        elif r.get("disturbance") == "yes":
            d = driver_of(r, "driver_disturbance", "driver_unclear")
            cls, l2, l3 = "Degradation", f"Degradation_in_{fs}", f"Degradation_in_{fs}"
            how = "disturbance"
        elif r.get("regrowth") == "yes":
            d = driver_of(r, "driver_regrowth", "driver_unclear")
            cls, l2, l3 = "Enhancement", f"Enhancement_in_{fs}", f"Enhancement_in_{fs}"
            how = "regrowth"
        else:
            cls, l2, l3 = "Stable forest", f"Stable_forest_in_{fs}", f"Stable_forest_in_{fs}"
        if d and cls not in ("Stable forest", "Stable non-forest"):
            l3 += f"_due_{d}"
        out.append((cls, l2, l3, how or r["rule"], d))
    result = lp[["survey_id", "plot_id", "from_year", "to_year", "from_ipcc", "to_ipcc", "from_sub", "to_sub", "from_adj", "to_adj"]].copy()
    result[["level1", "level2", "level3", "based_on", "driver"]] = pd.DataFrame(out, index=lp.index)
    if "rule_word" in lp:
        result["rule_word"] = lp["rule_word"].values
    return result


CLASS_ORDER = ["Stable forest", "Deforestation", "Degradation", "Afforestation", "Enhancement", "Forest subclass change", "To review", "Non-forest change", "Stable non-forest"]


def class_table(classified):
    t = pd.crosstab(classified["level1"], classified["from_year"].astype(str) + "-" + classified["to_year"].astype(str))
    t = t.reindex([c for c in CLASS_ORDER if c in t.index])
    t.loc["All plots"] = t.sum()
    t.index.name, t.columns.name = "class", None
    return t


REDD_ACTIVITY = {"Deforestation": "Reducing deforestation", "Degradation": "Reducing forest degradation",
                 "Afforestation": "Enhancement of forest carbon stocks", "Enhancement": "Enhancement of forest carbon stocks"}


def class_areas(classified, per_plot=0):
    """plots and area of each class, per pair of years: REDD+ activity, what decided the class, forest type, driver."""
    c = classified.copy()
    how = c["based_on"].fillna("").astype(str).str.strip()
    c["decided_by"] = how.map(lambda v: "land use" if not v or v == "nan" else ("canopy" if v.startswith("canopy") else v if v == "disturbance" else "rule"))
    c["driver"] = c["driver"].fillna("").astype(str).str.strip().replace("nan", "")
    c["forest_type"] = ""
    gone = c["level1"].isin(["Stable forest", "Deforestation", "Degradation", "Enhancement", "Forest subclass change", "To review"])
    c.loc[gone, "forest_type"] = c.loc[gone, "from_adj"]
    c.loc[c["level1"] == "Afforestation", "forest_type"] = c.loc[c["level1"] == "Afforestation", "to_adj"]
    g = c.groupby(["from_year", "to_year", "level1", "decided_by", "forest_type", "driver"]).size().reset_index(name="plots")
    g.insert(2, "REDD+ activity", g["level1"].map(REDD_ACTIVITY).fillna(""))
    g = g.rename(columns={"level1": "class"})
    g["_k"] = g["class"].map({v: i for i, v in enumerate(CLASS_ORDER)})
    g = g.sort_values(["from_year", "_k", "decided_by", "forest_type", "driver"]).drop(columns="_k").reset_index(drop=True)
    if per_plot:
        g["area_ha"] = (g["plots"] * per_plot).round(2)
    return g


def subdivision_order(classes):
    return [s for c in IPCC_ORDER for s in classes.loc[classes["IPCC category"] == c, "subdivision"].drop_duplicates()]


def transitions_long(lp, classes, per_plot):
    """from > to with the plots and the area, per pair of years; the base of the matrices and of the units of land."""
    g = lp.groupby(["from_year", "to_year", "from_ipcc", "from_adj", "to_ipcc", "to_adj"]).size().reset_index(name="plots")
    g = g.rename(columns={"from_adj": "from_subdivision", "to_adj": "to_subdivision"})
    g["area_ha"] = (g["plots"] * per_plot).round(2) if per_plot else ""
    return g


def whole_period(lp):
    """each plot: its land in the first year and in the last one (not the sum of the pairs)."""
    o = lp.sort_values(["plot_id", "from_year"])
    a, b = o.drop_duplicates("plot_id", keep="first"), o.drop_duplicates("plot_id", keep="last")
    keep = ["plot_id", "from_ipcc", "from_adj"] + (["w"] if "w" in o else [])
    return a[keep].merge(b[["plot_id", "to_ipcc", "to_adj"]], on="plot_id")


def write_matrix(w, sheet, title, a, b, order, per_plot, formats, wt=None):
    """rows = initial, columns = final, as in the example of the IPCC guide; plots, then hectares when the area is known
    (v17: wt = the hectares of each plot; the hectares are the sum of the weights)."""
    t = pd.crosstab(a, b).reindex(index=order, columns=order, fill_value=0)
    h = pd.crosstab(a, b, values=wt, aggfunc="sum").reindex(index=order, columns=order).fillna(0) if wt is not None else t * per_plot
    keep = [x for x in order if t.loc[x].sum() or t[x].sum()]
    t = t.loc[keep, keep]
    s = w.book.add_worksheet(sheet[:31])
    w.sheets[sheet[:31]] = s
    s.set_column(0, 0, 30); s.set_column(1, len(keep) + 1, 13)
    row = 0
    h = h.loc[keep, keep]
    for unit, m, fmt in [("plots", t, formats["int"])] + ([("hectares", h, formats["ha"])] if per_plot else []):
        s.write(row, 0, f"{title} - {unit}", formats["title"])
        row += 1
        s.write(row, 0, "Initial (rows) / Final (columns)", formats["head"])
        for c, name in enumerate(keep, 1):
            s.write(row, c, name, formats["head"])
        s.write(row, len(keep) + 1, "Total initial", formats["head"])
        for i, name in enumerate(keep):
            row += 1
            s.write(row, 0, name, formats["head"])
            for j, other in enumerate(keep):
                s.write_number(row, j + 1, float(m.at[name, other]), formats["int_d" if unit == "plots" else "ha_d"] if i == j else fmt)
            s.write_number(row, len(keep) + 1, float(m.loc[name].sum()), fmt)
        row += 1
        s.write(row, 0, "Total final", formats["head"])
        for j, other in enumerate(keep):
            s.write_number(row, j + 1, float(m[other].sum()), fmt)
        s.write_number(row, len(keep) + 1, float(m.values.sum()), fmt)
        row += 2


def write_matrices(path, lp, classes, per_plot, total_area):
    subs = subdivision_order(classes)
    with pd.ExcelWriter(path, engine="xlsxwriter") as w:
        book = w.book
        fm = {"title": book.add_format({"bold": True}), "head": book.add_format({"bold": True, "bg_color": "#DCE6F1", "border": 1, "text_wrap": True}),
              "int": book.add_format({"num_format": "#,##0", "border": 1}), "ha": book.add_format({"num_format": "#,##0.0", "border": 1}),
              "int_d": book.add_format({"num_format": "#,##0", "border": 1, "bg_color": "#D9D9D9"}), "ha_d": book.add_format({"num_format": "#,##0.0", "border": 1, "bg_color": "#D9D9D9"})}
        info = [("What it is", "Land-use conversion matrices. Rows = land in the first year, columns = land in the second year. Grey = no change."),
                ("Land", "As reported (reclassify, merge); the lookup does not change the forest types recorded."),
                ("Area", (f"Total area used: {total_area:,.0f} ha; the hectares are the sum of the weights of the plots (sampling design: see Step 7)." if "w" in lp else f"Total area used: {total_area:,.0f} ha; each plot = {per_plot:,.2f} ha.") if per_plot else "No total area given: plots only."),
                ("Note", "In the IPCC software's matrix tab, rows are final and columns initial: the opposite of this layout and of the guide's Excel example.")]
        pd.DataFrame(info, columns=["", "note"]).to_excel(w, sheet_name="README", index=False)
        w.sheets["README"].set_column(0, 0, 14); w.sheets["README"].set_column(1, 1, 130)
        whole = whole_period(lp)
        first, last = int(lp["from_year"].min()), int(lp["to_year"].max())
        wt = whole["w"] if "w" in whole else None
        write_matrix(w, "IPCC whole period", f"IPCC categories {first}-{last}", whole["from_ipcc"], whole["to_ipcc"], IPCC_ORDER, per_plot, fm, wt)
        write_matrix(w, "Subdivisions whole period", f"Subdivisions {first}-{last}", whole["from_adj"], whole["to_adj"], subs, per_plot, fm, wt)
        for (y0, y1), g in lp.groupby(["from_year", "to_year"]):
            write_matrix(w, f"{y0}-{y1}", f"Subdivisions {y0}-{y1}", g["from_adj"], g["to_adj"], subs, per_plot, fm, g["w"] if "w" in g else None)


def units_of_land(periods, info, transition=20):
    """the table of the guide's Step 3 for Approach 2. periods = [(first year, second year, {(from, to): area})], in order;
    info = {subdivision: (category, subcategory)}. Returns (table, year columns, area check per year)."""
    years = [periods[0][0]] + [p[1] for p in periods]
    before = dict(zip(years[1:], years[:-1]))
    initial = defaultdict(float)
    for (a, b), v in periods[0][2].items():
        initial[a] += v
    conv = []
    for y0, y1, cells in periods:
        for (a, b), v in cells.items():
            if a != b and v > 0:
                conv.append({"from": a, "to": b, "year": y1, "area": v})
    level = dict(initial)
    remaining = {s: {years[0]: initial.get(s, 0.0)} for s in info}
    for y in years[1:]:
        p = before[y]
        for s in info:
            out = sum(c["area"] for c in conv if c["year"] == y and c["from"] == s)
            back = sum(c["area"] for c in conv if c["to"] == s and (before.get(p, float("-inf")) < c["year"] + transition <= p))
            level[s] = level.get(s, 0.0) - out + back
            remaining[s][y] = level[s]

    def cell(c, y):
        end = c["year"] + transition
        if y < c["year"]:
            return "-"
        if y < end:
            return c["area"]
        return "merge" if before.get(y, float("-inf")) < end else "-"

    rows, n = [], defaultdict(int)
    cat_rank = {c: i for i, c in enumerate(IPCC_ORDER)}
    subs = list(info)
    for s in subs:
        if not any(abs(v) > 1e-9 for v in remaining[s].values()):
            continue
        c, sc = info[s]
        n[c] += 1
        rows.append({"_k": (cat_rank.get(c, 9), 0, subs.index(s), 0, 0), "ID": f"{IPCC_SHORT.get(c, c)}-{IPCC_SHORT.get(c, c)}_{n[c]}", "Previous category": c, "Previous subcategory": sc, "Previous subdivision": s,
                     "Current category": c, "Current subcategory": sc, "Current subdivision": s, "Transition period": "", "Year of conversion": "",
                     **{str(y): round(remaining[s][y], 2) for y in years}})
    for c in sorted(conv, key=lambda c: (c["year"], cat_rank.get(info[c["from"]][0], 9), subs.index(c["from"]), subs.index(c["to"]))):
        fc, fs = info[c["from"]]
        tc, ts = info[c["to"]]
        head = f"{IPCC_SHORT.get(fc, fc)}-{IPCC_SHORT.get(tc, tc)}_{c['year']}"
        n[head] += 1
        rows.append({"_k": (cat_rank.get(tc, 9), 1, c["year"], 0 if fc == tc else 1 + cat_rank.get(fc, 9), n[head]), "ID": f"{head}-{n[head]}", "Previous category": fc, "Previous subcategory": fs, "Previous subdivision": c["from"],
                     "Current category": tc, "Current subcategory": ts, "Current subdivision": c["to"], "Transition period": transition, "Year of conversion": c["year"],
                     **{str(y): (round(cell(c, y), 2) if isinstance(cell(c, y), float) else cell(c, y)) for y in years}})
    rows.sort(key=lambda r: r["_k"])
    table = pd.DataFrame(rows).drop(columns="_k")
    check = {}
    for y in years:
        conv_area = sum(c["area"] for c in conv if cell(c, y) != "-")
        check[y] = round(sum(remaining[s][y] for s in info) + conv_area, 2)
    return table, [str(y) for y in years], check


def write_ipcc_table(path, units, years, total_area, transition):
    """the Excel for the IPCC software: first sheet = what to do, second sheet = the land table. Area only."""
    table = units.drop(columns=["ID", "Transition period"])
    table = table[[c for c in table.columns if c != "Year of conversion"] + ["Year of conversion"]]
    table["Year of conversion"] = table["Year of conversion"].replace("", None)
    table.insert(0, "No.", range(1, len(table) + 1))
    lines = [("What this is", f"Activity data (area, ha) for the IPCC Inventory Software, Land Representation Manager, Approach 2. Years {years[0]}-{years[-1]}. Area only."),
             ("1. Subdivisions", "In the Land Use Manager, create every subdivision of the sheet Land table, with the same name, before entering any area."),
             ("2. Region", f"In the Land Representation Manager, tab Regions: one Region, total area {total_area:,.0f} ha, Approach 2. The survey has no soil or climate."),
             ("3. Enter the rows", "Tab Land representation table: one row of the sheet Land table = one entry. Start from the first year and go forward (the software does not accept the reverse order)."),
             ("Columns", ""),
             ("No.", "A row number only, to find the row. The software gives each unit of land its own code when it is saved (guide p. 55); the land unit code you can type there is optional and free (p. 54)."),
             ("Subcategory", "Previous and current: chosen from the dropdown list of the software. The names here are those of that list."),
             ("Subdivision", "Previous and current: chosen from the dropdown list, filled by the names created in step 1."),
             ("Area per year", "Hectares. - = no area in that year (the change has not happened yet, or the land has already joined the row with no change)."),
             ("merge", "Appears only if the series is longer than the transition period: the land joins the row with no change (button M in the software)."),
             ("Year of conversion", "Required by the software for every row where the land changed: the year the change was seen. Empty = no change."),
             ("Transition period", f"Not a column: the software proposes 20 years. Used here: {transition}. Change it in the software only if the country uses another value."),
             ("Where to find each result", ""),
             ("Deforestation", "Previous category = Forest land, current category = another one."),
             ("Afforestation", "Current category = Forest land, previous category = another one."),
             ("Change of forest type", "Both categories = Forest land, different subdivisions. The rules of Step 6 say which ones are degradation or enhancement."),
             ("Other land changes", "Neither category is Forest land."),
             ("No change", "Previous = current; Year of conversion empty."),
             ("Disturbance", "A disturbance with no change of forest type is not in this table: see the class_areas file of Step 7."),
             ("Limits", ""),
             ("Repeated changes", "Approach 2 does not link them: a plot that changes twice gives two rows, and the first stays counted."),
             ("First year", "All land in the first year is taken as not in conversion."),
             ("Hectares", "Plots / all plots x total area: every plot represents the same area."),
             ("Source", "IPCC Inventory Software, Guide to Land Representation v1.2: pp. 5, 6, 44, 50, 52, 54, 55, 57, 58, 105, 108.")]
    with pd.ExcelWriter(path, engine="xlsxwriter") as w:
        book = w.book
        head = book.add_format({"bold": True, "font_color": "white", "bg_color": "#486A47"})
        bold = book.add_format({"bold": True, "valign": "top"})
        wrap = book.add_format({"text_wrap": True, "valign": "top"})
        s = book.add_worksheet("What to do")
        w.sheets["What to do"] = s
        s.set_column(0, 0, 26); s.set_column(1, 1, 110)
        s.write(0, 0, "Topic", head); s.write(0, 1, "What to do", head)
        for i, (a, b) in enumerate(lines, 1):
            if b:
                s.write(i, 0, a, bold); s.write(i, 1, b, wrap)
            else:
                s.write(i, 0, a, head); s.write(i, 1, "", head)
        table.to_excel(w, sheet_name="Land table", index=False)
        s2 = w.sheets["Land table"]
        s2.freeze_panes(1, 0)
        s2.set_column(0, 0, 6)
        s2.set_column(1, 6, 22)
        s2.set_column(7, 7 + len(years), 13)
        s2.set_column(7 + len(years), 7 + len(years), 18)
    return table


def ipcc_periods(lp, per_plot):
    out = []
    for (y0, y1), g in lp.groupby(["from_year", "to_year"]):
        cells = g.groupby(["from_adj", "to_adj"])["w"].sum() if "w" in g else g.groupby(["from_adj", "to_adj"]).size() * per_plot
        out.append((int(y0), int(y1), {k: float(v) for k, v in cells.items()}))
    return out


# ----------------------------------------------------------------------------------------------- the same columns and land names in every file

STANDARD_COLUMNS = YEAR_COLUMNS + ["driver_origin",
                                   "canopy_cover", "tree_cover", "patch_size", "trees_present", "native_species", "commercial_use", "managed", "cyclical_system",
                                   "temporarily_unstocked", "peat", "canals",
                                   "change_size", "tree_cover_before", "tree_cover_after", "tree_cover_change",
                                   "extends_previous_change", "temporary_loss", "within_buffer", "nonforest_duration_before", "forest_before_nonforest",
                                   "gain_origin", "note"]
STATE_COLUMNS = ["canopy_cover", "tree_cover", "patch_size", "trees_present", "native_species", "commercial_use", "managed", "cyclical_system",
                 "temporarily_unstocked", "peat", "canals"]     # they describe the land in a year; the other attributes describe a change
NO_INFORMATION = {"na", "n a"}
STATE_ATTRIBUTES.update({"canopy cover percent": "canopy_cover", "presence of peat": "peat", "presence of canals": "canals"})
YES_NO_VARIABLES.update({"peat", "canals", "tree_cover_decrease", "tree_cover_increase"})
NOT_USED = re.compile(NOT_USED.pattern.replace("canopy_cover_percent.*|", ""), re.I)
_stem_v5, _plot_year_table_v5 = stem, plot_year_table


def stem(column):
    """also 'commercial_t2' -> 'commercial', as lu_t2 is the land: for the forms that name the end year _t2."""
    return re.sub(r" t2$", "", _stem_v5(column))


def standard_table(table):
    """every survey gets the same columns in the same order (empty when not asked), the land in the names of the IPCC software
    and the same yes / no. Subdivisions, classes of canopy or patch size and drivers stay as the survey wrote them."""
    t = table.rename(columns={"canopy_cover_before": "tree_cover_before", "canopy_cover_after": "tree_cover_after",
                              "canopy_cover_decrease": "tree_cover_decrease", "canopy_cover_increase": "tree_cover_increase"})
    for c in STANDARD_COLUMNS:
        if c not in t:
            t[c] = ""
    if "loss_type" in t:                              # v14: the kind of loss (true loss / cyclical) = temporary_loss (no / yes)
        t["temporary_loss"] = t["temporary_loss"].where(t["temporary_loss"].map(given), t["loss_type"])
    t["gain_origin"] = t["gain_origin"].map(lambda v: GAIN_ORIGIN.get(plain(v), v) if given(v) else "")   # v15
    t["land_use"] = t["land_use"].map(lambda v: ipcc_category(v) or v)
    first = STANDARD_COLUMNS.index("driver_origin")
    for c in [x for x in STANDARD_COLUMNS[first:-1] + ["canopy_class"] if x in t]:
        t[c] = t[c].map(lambda v: "" if plain(v) in NO_INFORMATION else v)
    cover = t["canopy_cover"].where(t["canopy_cover"].map(given), t["tree_cover"])     # the land says which one it is
    if "canopy_class" in t:
        cover = cover.where(cover.map(given), t["canopy_class"])
    forest = t["land_use"].eq(FOREST_LAND)
    t["canopy_cover"], t["tree_cover"] = cover.where(forest, ""), cover.where(~forest, "")
    for c in YES_NO_VARIABLES & set(t):
        t[c] = t[c].map(lambda v: yes_no(v) or EXTRA_YES_NO.get(plain(v)) or v)
    dec = t["tree_cover_decrease"] if "tree_cover_decrease" in t else pd.Series("", index=t.index)
    inc = t["tree_cover_increase"] if "tree_cover_increase" in t else pd.Series("", index=t.index)
    t["tree_cover_change"] = ["decrease" if d == "yes" else "increase" if i == "yes" else "none" if "no" in (d, i) else ""
                              for d, i in zip(dec, inc)]                 # one column: decrease / increase / none
    t["driver_origin"] = t["driver_origin"].map(lambda v: {"yes": "anthropogenic", "no": "natural"}.get(yes_no(v), v))
    t["note"] = t["note"].astype(str).str.replace(r"(says )(\S+)( in \d{4})", lambda m: m[1] + (ipcc_category(m[2]) or m[2]) + m[3], regex=True)
    extra_columns = [c for c in t if c not in STANDARD_COLUMNS + ["canopy_class"] and t[c].map(given).any()]
    plot_year_table.left_out = extra_columns
    return repeat_state(t[STANDARD_COLUMNS])


def repeat_state(table):
    """the attributes of the land are filled like the land: over the years of the same land, up to any change, disturbance or
    regrowth, and only in the filled years. A year recorded by the survey (source = raw) keeps its own answers, empty when
    not answered: nothing is taken from another year. A canopy that differs from the year recorded before it, with the same
    land and nothing recorded in between, is not a change: a note says it (the year is not known between two dates)."""
    o = table.sort_values(["plot_id", "year"])
    new_land = (o[LAND] != o[LAND].shift()).any(axis=1) | (o["plot_id"] != o["plot_id"].shift())
    event = o["change"].eq("yes") | o["disturbance"].eq("yes") | o["regrowth"].eq("yes")
    same_state = (new_land | event).cumsum()
    filled = o["source"].ne("raw")
    for c in STATE_COLUMNS:
        v = o[c].where(o[c] != "")
        if v.notna().any():
            o[c] = o[c].where(~filled, v.groupby(same_state).ffill().groupby(same_state).bfill().fillna(""))
    o["note"] = canopy_between_dates(o, same_state)
    return o.loc[table.index]


def canopy_between_dates(o, same_state):
    """the note of each row, plus: 'canopy 100 → 92 between 2000 and 2008 (year not known), no change recorded' on a recorded
    year whose canopy (or tree cover) differs from the year recorded before it, with the same land and no change in between."""
    note = o["note"].astype(str).replace("nan", "")
    for c, word in (("canopy_cover", "canopy"), ("tree_cover", "tree cover")):
        r = o[o["source"].eq("raw") & o[c].map(given)]
        if not len(r):
            continue
        prev = r.groupby("plot_id", sort=False)[[c, "year"]].shift()
        state_before = same_state[r.index].groupby(r["plot_id"]).shift()
        m = prev[c].notna() & state_before.eq(same_state[r.index]) & (r[c].astype(str) != prev[c].astype(str))
        for i in r.index[m]:
            y0, y1 = int(prev.at[i, "year"]), int(r.at[i, "year"])
            when = f"in {y1}" if y1 - y0 == 1 else f"between {y0} and {y1} (year not known)"
            text = f"{word} {prev.at[i, c]} → {r.at[i, c]} {when}, no change recorded"
            note[i] = f"{note[i]}; {text}" if note[i] else text
    return note


def plot_year_table(survey, previous_year=0, start_year=0, direction=""):
    """only the years selected: the answers about earlier years stay in raw_data."""
    table = standard_table(_plot_year_table_v5(survey, previous_year, start_year, direction))
    keep = table["year"].le(survey["end_year"])
    if start_year:
        keep &= table["year"].ge(start_year)
    return table[keep].reset_index(drop=True)


_pairs_table_v5 = pairs_table


def pairs_table(table):
    """the pairs, with the attributes of the land in both years: from_ = the first year, to_ = the second year."""
    pairs = _pairs_table_v5(table).rename(columns={c: f"to_{c}" for c in STATE_COLUMNS})
    before = table[["plot_id", "year"] + STATE_COLUMNS].assign(year=table["year"] + 1).rename(columns={"year": "to_year", **{c: f"from_{c}" for c in STATE_COLUMNS}})
    pairs = pairs.merge(before, on=["plot_id", "to_year"], how="left")
    attributes = [f"{side}_{c}" for c in STATE_COLUMNS for side in ("from", "to")]
    return pairs[[c for c in pairs if c not in attributes] + attributes]


# ----------------------------------------------------------------------------------------------- v7: Step 7, the IPCC subcategories

DEFAULT_SUBCATEGORY = {"Forest land": "Managed Forest Land", "Cropland": "Cropland Annual Crops", "Grassland": "Managed Grassland",
                       "Wetlands": "Managed Wetlands", "Settlements": "Settlements (Other)", "Other land": "Managed Other Land"}


def make_subcategory_template(path, classes):
    """the Excel file for the IPCC software: each national subdivision, its IPCC category and the IPCC subcategory, proposed."""
    plots = [c for c in classes.columns if c.startswith("plots in")]
    t = classes[["subdivision", "IPCC category"] + plots].copy()
    t.insert(2, "IPCC subcategory", t["IPCC category"].map(DEFAULT_SUBCATEGORY).fillna(""))
    guide = [("What it is", "Only for the IPCC Inventory Software: it places each national subdivision under one of its 12 subcategories (guide pp. 5-6)."),
             ("Proposed", "Managed for every category, Cropland Annual Crops, Settlements (Other). Change it where the country decides otherwise."),
             ("Unmanaged", "Land with no human management (e.g. some natural forest, wetlands or other land): emissions are not reported for it."),
             ("Perennial crops", "Cropland with woody crops, e.g. oil palm, orchards, cashew."),
             ("Settlements (Treed)", "Settlements with trees, e.g. parks, gardens, tree-lined streets."),
             ("How", "Change only the yellow column, from the list (the words of the software). Then save the file and in Step 8 choose 'use my filled file'.")]
    with pd.ExcelWriter(path, engine="xlsxwriter") as w:
        book = w.book
        head = book.add_format({"bold": True, "bg_color": "#DCE6F1", "border": 1, "text_wrap": True, "valign": "top"})
        yellow = book.add_format({"bg_color": "#FFF2CC", "border": 1})
        pd.DataFrame(guide, columns=["", "how to fill it"]).to_excel(w, sheet_name="README", index=False)
        w.sheets["README"].set_column(0, 0, 20); w.sheets["README"].set_column(1, 1, 120)
        t.to_excel(w, sheet_name="subcategories", index=False)
        s = w.sheets["subcategories"]
        for c, name in enumerate(t.columns):
            s.write(0, c, name, head)
        s.set_column(0, len(t.columns), 26)
        for r, v in enumerate(t["IPCC subcategory"], 1):
            s.write(r, 2, v, yellow)
        if len(t):
            s.data_validation(1, 2, len(t), 2, {"validate": "list", "source": [x for v in IPCC_SUBCATEGORIES.values() for x in v]})


def read_subcategories(source, classes):
    """the filled file of Step 7: the IPCC subcategory of each subdivision, in the words of the software and of its category.
    It stops on a class with no subcategory or with a name the software does not have."""
    book = pd.ExcelFile(source)
    sheet = next((s for s in ("subcategories", "classes") if s in book.sheet_names), None)
    if sheet is None:
        raise ValueError("⛔ The file has no sheet 'subcategories'. Use the template made by this step.")
    f = book.parse(sheet, dtype=str)
    if not {"subdivision", "IPCC subcategory"} <= set(f.columns):
        raise ValueError("⛔ The sheet needs the columns subdivision and IPCC subcategory.")
    cat = f["IPCC category"] if "IPCC category" in f else pd.Series("", index=f.index)
    by_both = {(str(c).strip(), str(s).strip()): v for c, s, v in zip(cat, f["subdivision"], f["IPCC subcategory"]) if given(v)}
    by_sub = {str(s).strip(): v for s, v in zip(f["subdivision"], f["IPCC subcategory"]) if given(v)}
    allowed = {simple_words(x): (c, x) for c, xs in IPCC_SUBCATEGORIES.items() for x in xs}
    out = classes.copy()
    missing, bad = [], []
    for i, r in out.iterrows():
        v = by_both.get((r["IPCC category"], r["subdivision"]), by_sub.get(r["subdivision"]))
        if not given(v):
            missing.append(r["subdivision"])
            continue
        hit = allowed.get(simple_words(v))
        if hit is None or hit[0] != r["IPCC category"]:
            bad.append(f"{r['subdivision']} ({r['IPCC category']}): '{v}' - choose one of {IPCC_SUBCATEGORIES.get(r['IPCC category'], [])}")
        else:
            out.at[i, "IPCC subcategory"] = hit[1]
    if bad:
        raise ValueError("⛔ Not a subcategory of the software for that category:\n   " + "\n   ".join(bad[:20]))
    if missing:
        raise ValueError(f"⛔ {len(missing)} subdivisions have no IPCC subcategory: {missing[:12]}. Fill them in the file.")
    return out


# ----------------------------------------------------------------------------------------------- v8: the unified format, always the same

UNIFIED_COLUMNS = ["survey_id", "plot_id", "x", "y"] + STANDARD_COLUMNS[2:]   # 39 standard columns + x, y; nothing else


def plot_xy(survey):
    """x and y of each plot, from the columns the description found for them."""
    roles, data, id_col = survey["roles"], survey["data"], survey["id_col"]
    out = {}
    for v in ("x", "y"):
        col = roles.loc[roles["variable"] == v, "export column"]
        if len(col):
            out[v] = {str(p): value_of(v, x) for p, x in zip(data[id_col], data[col.iloc[0]]) if given(p) and given(x)}
    return out


_plot_year_table_v7 = plot_year_table


def plot_year_table(survey, previous_year=0, start_year=0, direction=""):
    """v8: the unified format: the land, change, drivers and attributes, plus x and y. Every other answer stays in the input."""
    table = _plot_year_table_v7(survey, previous_year, start_year, direction)
    xy = plot_xy(survey)
    for v in ("x", "y"):
        table[v] = table["plot_id"].astype(str).map(xy.get(v, {})).fillna("")
    return table[UNIFIED_COLUMNS]          # v14: tree_cover_before / _after / _change only when the survey asks them itself


def canopy_before_after(table):              # not used since v14: the canopy before and after a change are the rows around it
    """a change or disturbance within forest land, with the canopy recorded before and after it: tree_cover_before,
    tree_cover_after and tree_cover_change (decrease / increase / none), when the survey did not ask them itself."""
    o = table.sort_values(["plot_id", "year"])
    g = o.groupby("plot_id", sort=False)
    before, land_before = g["canopy_cover"].shift().fillna(""), g["land_use"].shift().fillna("")
    event = o["change"].eq("yes") | o["disturbance"].eq("yes") | o["regrowth"].eq("yes")
    m = (event & o["land_use"].eq(FOREST_LAND) & land_before.eq(FOREST_LAND) & o["canopy_cover"].map(given) & before.map(given)
         & o["tree_cover_before"].eq("") & o["tree_cover_after"].eq(""))
    if m.any():
        a, b = pd.to_numeric(before[m], errors="coerce"), pd.to_numeric(o.loc[m, "canopy_cover"], errors="coerce")
        o.loc[m, "tree_cover_before"], o.loc[m, "tree_cover_after"] = before[m], o.loc[m, "canopy_cover"]
        change = pd.Series(["decrease" if y < x else "increase" if y > x else "none" if y == x else "" for x, y in zip(a, b)], index=a.index)
        o.loc[m & o["tree_cover_change"].eq(""), "tree_cover_change"] = change
    return o.loc[table.index]


# ----------------------------------------------------------------------------------------------- v7_v4: the classes, one row per plot

def classes_wide(classified):
    """as in the Extract Tool: one row per plot; per pair of years (2019_2020 = the changes during 2020)
    _ipcc and _sub (from > to, as recorded), _level1, _level2, _level3, _driver and _rule (why the class)."""
    c = classified.copy()
    c["pair"] = c["from_year"].astype(int).astype(str) + "_" + c["to_year"].astype(int).astype(str)

    def why(r):
        b = r["based_on"]
        if b in ("disturbance", "regrowth"):
            return f"{b} recorded"
        if b in ("Degradation", "Enhancement", "Stable forest", "Not possible"):
            w = r.get("rule_word", "")
            return f"lookup: {w if isinstance(w, str) and w.strip() else b}"
        if b == "no rule":
            return "lookup: no rule"
        if r["from_ipcc"] != r["to_ipcc"]:
            return "IPCC category change"
        return "subdivision change" if r["from_sub"] != r["to_sub"] else "no change"

    c["ipcc"] = c["from_ipcc"] + " > " + c["to_ipcc"]
    c["sub"] = c["from_sub"].astype(str) + " > " + c["to_sub"].astype(str)
    c["rule"] = c.apply(why, axis=1)
    fields = ["ipcc", "sub", "level1", "level2", "level3", "driver", "rule"]
    w = c.pivot_table(index=["survey_id", "plot_id"], columns="pair", values=fields, aggfunc="first")
    pairs = sorted(c["pair"].unique())
    w = w.reindex(columns=[(f, p) for p in pairs for f in fields])
    w.columns = [f"{p}_{'level' + f[-1] if f.startswith('level') else f}" for f, p in w.columns]
    return w.fillna("").reset_index()

# ----------------------------------------------------------------------------------------------- v14: Step 5, the history of the samples

YEARS_EXPLAINED = ("How to read the years: the land of a year is the land at the end of that year (31 December). "
                   "A pair 2012-2013 = the land at the end of 2012 and at the end of 2013: a change in the pair happened at any time "
                   "in 2013, from 1 January to 31 December (29 December 2013 is still a change of 2013), and the 2013 row already "
                   "shows the land after it. The type and the drivers of the change are in the second year of the pair (2013).")


def history_wide(table):
    """the history of each plot in one row, before any rule of the country: the IPCC category, the subdivision and the canopy
    of every year side by side (lu_ipcc_1990, lu_sub_1990, cc_1990, lu_ipcc_1991, ...). cc = canopy_cover on forest land,
    tree_cover on other land, as in the unified format; temporary_loss_<year> (yes / no) only for the years where a plot has
    one. Made from the unified format only."""
    t = table.drop_duplicates(["plot_id", "year"]).copy()
    t["cc"] = t["canopy_cover"].where(t["canopy_cover"].map(given), t["tree_cover"])
    order = list(dict.fromkeys(t["plot_id"]))
    ipcc = t.pivot(index="plot_id", columns="year", values="land_use").reindex(order)
    sub = t.pivot(index="plot_id", columns="year", values="subdivision").reindex(order)
    cc = t.pivot(index="plot_id", columns="year", values="cc").reindex(order)
    loss = t.pivot(index="plot_id", columns="year", values="temporary_loss").reindex(order).fillna("")
    gain = t.pivot(index="plot_id", columns="year", values="gain_origin").reindex(order).fillna("")
    cols = {}
    for y in sorted(t["year"].unique()):
        cols[f"lu_ipcc_{y}"], cols[f"lu_sub_{y}"], cols[f"cc_{y}"] = ipcc[y], sub[y], cc[y]
        if loss[y].map(given).any():                  # an attribute of a change: only the years where a plot has one
            cols[f"temporary_loss_{y}"] = loss[y]
        if gain[y].map(given).any():
            cols[f"gain_origin_{y}"] = gain[y]
    xy = t.groupby("plot_id", sort=False)[["x", "y"]].first().reindex(order)
    out = pd.concat([xy, pd.DataFrame(cols, index=ipcc.index)], axis=1).fillna("")
    return out.rename_axis("plot_id").reset_index()


def history_line(table, plot_id):
    """the history of one plot in a few words: '1990–2017 Forest land / Montane - Dense, 2018–2024 Forest land / Montane - Open'."""
    p = table[table["plot_id"].astype(str) == str(plot_id)].sort_values("year")
    land = (p["land_use"].astype(str) + " / " + p["subdivision"].astype(str)).tolist()
    years, out, i = p["year"].tolist(), [], 0
    while i < len(years):
        j = i
        while j + 1 < len(years) and land[j + 1] == land[i]:
            j += 1
        out.append(f"{years[i]}–{years[j]} {land[i]}" if j > i else f"{years[i]} {land[i]}")
        i = j + 1
    return ", ".join(out)


def flag_to_correct(table, to_fix):
    """the errors left uncorrected (Step 3, skip_correction): nothing is changed or removed, the values stay as in the export;
    the note of their row says 'to correct: ...' (the row of the year the error names, else the last year of the plot)."""
    t = table.copy()
    note = t["note"].astype(str).replace("nan", "")
    for r in to_fix.to_dict("records"):
        rows = t.index[t["plot_id"].astype(str) == str(r["plot"])]
        if not len(rows):
            continue
        detail = str(r["detail"])
        m = re.match(r"^(\d{4}): ", detail)
        what = detail[m.end():] if m else detail
        at = [i for i in rows if m and int(t.at[i, "year"]) == int(m[1])] or [t.loc[rows, "year"].astype(int).idxmax()]
        text = f"to correct: {what}"
        for i in at:
            if text in note[i]:
                continue
            note[i] = note[i].replace(what, text) if what in note[i] else (f"{note[i]}; {text}" if note[i] else text)
    t["note"] = note
    return t


def land_name_class(name, classes):
    """a land name that is not a class of the dates but the start of exactly one of them: that class ('Annual' -> 'Annual cropland')."""
    n = plain(name)
    found = [c for c in classes if plain(c) == n or plain(c).startswith(n + " ")]
    return found[0] if len(found) == 1 else None


def land_names_report(survey):
    """the answers 'land after a change' that are not a class of the dates, per question: what the tool does with them."""
    classes, seen = survey.get("date_classes", set()), Counter()
    for p in survey.get("period_cols", []):
        if p["variable"] != "subcategory_1":
            continue
        for v in survey["data"][p["col"]]:
            if given(v) and str(v).strip() not in classes and str(v).strip() != "0":
                seen[(answer_stem(p["col"]), str(v).strip())] += 1
    out = []
    for (q, v), n in sorted(seen.items()):
        k = land_name_class(v, classes)
        out.append(f"{q}: '{v}' ({n} answers) is not a class of the dates: " +
                   (f"read as '{k}' where no date gives the land" if k else "the land recorded at the date is used; where no date gives it, it stays as written"))
    return out


def read_land_names(table, survey):
    """a land name left in the table that is not a class of the dates but the start of exactly one: that class and its land use,
    with a note on the year it was recorded."""
    classes = survey.get("date_classes", set())
    names = {v: land_name_class(v, classes) for v in set(table["subdivision"]) - set(classes) - {""}}
    names = {v: k for v, k in names.items() if k}
    if not names:
        return table
    t = table.copy()
    known = t[t["subdivision"].isin(classes) & t["land_use"].ne("")]
    land_use_of = known.groupby("subdivision")["land_use"].agg(lambda x: x.mode().iat[0])
    m = t["subdivision"].isin(names)
    old = t.loc[m, "subdivision"]
    t.loc[m, "subdivision"] = old.map(names)
    t.loc[m, "land_use"] = t.loc[m, "subdivision"].map(land_use_of).fillna(t.loc[m, "land_use"])
    raw = m & t["source"].eq("raw")
    for i in t.index[raw]:
        text = f"land recorded as '{old[i]}', read as '{t.at[i, 'subdivision']}'"
        t.at[i, "note"] = f"{t.at[i, 'note']}; {text}" if t.at[i, "note"] else text
    return t


_plot_year_table_v14 = plot_year_table


def plot_year_table(survey, previous_year=0, start_year=0, direction=""):
    """v14: + the land names read as a class of the dates."""
    return read_land_names(_plot_year_table_v14(survey, previous_year, start_year, direction), survey)


# ----------------------------------------------------------------------------------------------- v15

GAIN_ORIGIN = {"planting": "planted", "planted": "planted", "plantation": "planted", "plant": "planted",
               "natural": "natural", "natural regeneration": "natural", "naturally": "natural", "natural regrowth": "natural"}
CHANGE_COLUMNS_SHOWN = ["driver_change", "driver_disturbance", "driver_regrowth", "driver_unclear", "driver_origin", "gain_origin",
                     "temporary_loss", "nonforest_duration_before", "forest_before_nonforest", "change_size", "extends_previous_change",
                     "within_buffer", "tree_cover_before", "tree_cover_after", "tree_cover_change", "canopy_cover", "tree_cover"]


def gain_problems(table):
    """how a gain happened (gain_origin, the interpreter's answer) against the class after it: in each class reached by gains,
    the answers that differ from what most gains to that class say (e.g. planted, but a natural forest type after)."""
    out = []
    if "gain_origin" in table:
        g = table[table["change"].eq("yes") & table["gain_origin"].map(given)]
        for sub, part in g.groupby("subdivision"):
            counts = part["gain_origin"].value_counts()
            if len(counts) < 2 or counts.iloc[0] == counts.iloc[1]:
                continue
            usual = counts.index[0]
            for r in part[part["gain_origin"] != usual].to_dict("records"):
                out.append({"check": "4 gain", "level": "⚠ to check", "plot": r["plot_id"],
                            "detail": f"{r['year']}: gain_origin = {r['gain_origin']}, but land after = {sub} "
                                      f"(most gains to {sub}: {usual}, {counts.iloc[0]} of {counts.sum()})"})
    return pd.DataFrame(out, columns=["check", "level", "plot", "detail"])


_land_problems_v14 = land_problems


def land_problems(table):
    """v15: + how a gain happened against the class after it."""
    return pd.concat([_land_problems_v14(table), gain_problems(table)], ignore_index=True)


def land_names_found(survey):
    """the answers 'land after a change' that are not a class of the dates: (question, name, answers, the class it is read as)."""
    classes, seen = survey.get("date_classes", set()), Counter()
    for p in survey.get("period_cols", []):
        if p["variable"] != "subcategory_1":
            continue
        for v in survey["data"][p["col"]]:
            if given(v) and str(v).strip() not in classes and str(v).strip() != "0":
                seen[(answer_stem(p["col"]), str(v).strip())] += 1
    return [(q, v, n, land_name_class(v, classes)) for (q, v), n in sorted(seen.items())]


def land_names_report(survey):
    """the same, in words."""
    return [f"{q}: '{v}' ({n} answers) is not a class of the dates: " +
            (f"read as '{k}' where no date gives the land" if k else "the land recorded at the date is used; where no date gives it, it stays as written")
            for q, v, n, k in land_names_found(survey)]


def values_in_words(values):
    """a column's values in a few words: a range for numbers, the most common answers otherwise."""
    v = values[values.map(given)].astype(str)
    num = pd.to_numeric(v, errors="coerce")
    if len(v) and num.notna().all():
        return f"{num.min():g}–{num.max():g}"
    top = v.value_counts()
    return ", ".join(f"{k} {n:,}" for k, n in top.head(4).items()) + (f", +{len(top) - 4} more" if len(top) > 4 else "")


def picture_classes(survey, table):
    """the classification as the survey uses it, drawn from the table: each land use with its subdivisions (plots) and the
    attributes of the land; each type of change with what is recorded with it and its column in the unified format. A land
    name used only in the answers about a change, not at the dates, is marked."""
    t = table[table["land_use"].ne("")]
    plots = t.groupby(["land_use", "subdivision"])["plot_id"].nunique()
    odd = defaultdict(list)
    land_use_of = t.groupby("subdivision")["land_use"].agg(lambda x: x.mode().iat[0])
    for q, v, n, k in land_names_found(survey):
        odd[land_use_of.get(k, "") if k else ""].append((q, v, n, k))
    rows = []
    for lu in [x for x in IPCC_ORDER if x in plots.index.get_level_values(0)] + [x for x in plots.index.get_level_values(0).unique() if x not in IPCC_ORDER]:
        subs = plots[lu].sort_values(ascending=False)
        chips = "".join(f'<span class="s">{escape(str(k))} <small>{v:,}</small></span>' for k, v in subs.items() if k)
        chips += "".join(f'<span class="s odd">{escape(v)} <small>only in {escape(q)} · {n:,}, read as {escape(k)}</small></span>' for q, v, n, k in odd.get(lu, []))
        part = t[t["land_use"] == lu]
        attrs = [f"<code>{c}</code> <small>{escape(values_in_words(part[c]))}</small>" for c in STATE_COLUMNS if c in part and part[c].map(given).any()]
        if attrs:
            chips += f'<div class="at">attributes of the land: {" · ".join(attrs)}</div>'
        rows.append(f'<div class="lu">{escape(lu)} <small>{part["plot_id"].nunique():,} plots</small></div><div class="subs">{chips}</div>')
    if odd.get(""):
        chips = "".join(f'<span class="s odd">{escape(v)} <small>only in {escape(q)} · {n:,}</small></span>' for q, v, n, k in odd[""])
        rows.append(f'<div class="lu">Other names <small>in the answers</small></div><div class="subs">{chips}'
                    '<div class="at">not a class of the dates: the land recorded at the date is used</div></div>')
    states = f'<div class="tree">{"".join(rows)}</div>'
    ch = table[table["change"].eq("yes") | table["disturbance"].eq("yes")]
    rows = []
    kinds = ch["change_type"].replace("", "(no type given)")
    for kind in kinds.value_counts().index:
        g = ch[kinds == kind]
        chips = '<span class="s">land after <small>→ land_use, subdivision</small></span>'
        chips += "".join(f'<span class="s"><code>{c}</code> <small>{escape(values_in_words(g[c]))}</small></span>'
                         for c in CHANGE_COLUMNS_SHOWN if c in g and g[c].map(given).any())
        rows.append(f'<div class="lu chg">{escape(kind)} <small>{len(g):,}</small></div><div class="subs">{chips}</div>')
    changes = f'<div class="tree">{"".join(rows)}</div>'
    return ('<p class="hint">The land: each land use with the subdivisions used (number of plots) and the attributes of the land. '
            'Dashed red: a name used only in the answers about a change, not at the dates.</p>' + states +
            '<p class="hint">The changes: each type with what is recorded with it, on the year of the change (column of the unified format, values).</p>' + changes)


# ----------------------------------------------------------------------------------------------- v16: the country's reading of the classes (Step 6)

def class_names(survey):
    """{code: label} of the classes, from the export (a column and its _label column), to find a class by its name too."""
    data, out = survey["data"], {}
    for c in data.columns:
        if f"{c}_label" in data.columns:
            for a, b in zip(data[c], data[f"{c}_label"]):
                if given(a) and given(b):
                    out.setdefault(str(a).strip(), str(b).strip())
    return out


def find_class(text, classes, names):
    """the subdivision written by its code or its name (capitals, accents and spaces do not matter); None if not found."""
    key = simple_words(text)
    for sub in classes["subdivision"]:
        if key in (simple_words(sub), simple_words(names.get(sub, ""))):
            return sub
    return None


def excel_of(source):
    """an Excel file from a path or from the bytes of an upload (read again each time)."""
    return pd.ExcelFile(io.BytesIO(source) if isinstance(source, (bytes, bytearray)) else source)


def read_adjustments(source):
    """the country's reading of the classes in a lookup file: the sheet 'crosswalk' (sub, reported_sub, reported_ipcc: one row
    per recorded class that is reported differently), or the sheets of the Extract Tool 'category changes' (sub, category in
    the adjusted version) and 'subclass merges' (sub, merged into). ({sub: IPCC category}, {sub: reported class})."""
    book = excel_of(source)
    changes, merges = {}, {}
    if "category changes" in book.sheet_names:
        t = book.parse("category changes", dtype=str)
        col = next((c for c in t.columns if "adjusted" in simple_words(c) or simple_words(c).startswith("new category")), None)
        if col and "sub" in t:
            changes = {str(a).strip(): ipcc_category(b) or str(b).strip() for a, b in zip(t["sub"], t[col]) if given(a) and given(b)}
    if "subclass merges" in book.sheet_names:
        t = book.parse("subclass merges", dtype=str)
        col = next((c for c in t.columns if simple_words(c).startswith("merged")), None)
        if col and "sub" in t:
            merges = {str(a).strip(): str(b).strip() for a, b in zip(t["sub"], t[col]) if given(a) and given(b)}
    if "crosswalk" in book.sheet_names:
        t = book.parse("crosswalk", dtype=str)
        if "sub" in t:
            for r in t.to_dict("records"):
                sub = str(r["sub"]).strip() if given(r["sub"]) else ""
                if not sub:
                    continue
                if given(r.get("reported_ipcc")):
                    changes[sub] = ipcc_category(r["reported_ipcc"]) or str(r["reported_ipcc"]).strip()
                if given(r.get("reported_sub")) and str(r["reported_sub"]).strip() != sub:
                    merges[sub] = str(r["reported_sub"]).strip()
    return changes, merges


def read_fields(reclassify, merge, classes, names):
    """the form fields of Step 6: 'class > IPCC category; ...' and 'class + class > new name; ...'. ({}, {}, problems)."""
    changes, merges, problems = {}, {}, []
    for entry in [e.strip() for e in str(reclassify).split(";") if e.strip()]:
        if ">" not in entry:
            problems.append(f"reclassify '{entry}': write  class > IPCC category")
            continue
        name, category = [p.strip() for p in entry.rsplit(">", 1)]
        sub, cat = find_class(name, classes, names), ipcc_category(category)
        if sub is None:
            problems.append(f"reclassify: class '{name}' not found")
        elif cat not in IPCC_ORDER:
            problems.append(f"reclassify: '{category}' is not one of {IPCC_ORDER}")
        else:
            changes[sub] = cat
    for entry in [e.strip() for e in str(merge).split(";") if e.strip()]:
        if ">" not in entry or "+" not in entry:
            problems.append(f"merge '{entry}': write  class + class > new name")
            continue
        members, new = entry.rsplit(">", 1)
        new = new.strip()
        for name in [m.strip() for m in members.split("+") if m.strip()]:
            sub = find_class(name, classes, names)
            if sub is None:
                problems.append(f"merge: class '{name}' not found")
            else:
                merges[sub] = new
    return changes, merges, problems


def with_category_changes(classes, changes):
    """the classes with the IPCC category of the reclassified subdivisions changed."""
    out = classes.copy()
    hit = out["subdivision"].isin(changes)
    out.loc[hit, "IPCC category"] = out.loc[hit, "subdivision"].map(changes)
    return out


def merged_classes(classes, merges):
    """the classes after the merges: the merged classes replace their members (one row, the plots added up)."""
    if not merges:
        return classes
    out = classes[~classes["subdivision"].isin(merges)].copy()
    rows = []
    for new in dict.fromkeys(merges.values()):
        part = classes[classes["subdivision"].isin([k for k, v in merges.items() if v == new])]
        if not len(part):
            continue
        cats = part["IPCC category"].unique()
        if len(cats) > 1:
            print(f"⚠ merge {new}: its classes are in different IPCC categories ({', '.join(cats)}): {cats[0]} is used.")
        row = part.iloc[0].copy()
        row["subdivision"] = new
        for c in [c for c in part.columns if str(c).startswith("plots in")]:
            row[c] = part[c].sum()
        rows.append(row)
    out = pd.concat([out, pd.DataFrame(rows)], ignore_index=True)
    out["_o"] = out["IPCC category"].map({c: i for i, c in enumerate(IPCC_ORDER)}).fillna(9)
    return out.sort_values(["_o", "subdivision"]).drop(columns="_o").reset_index(drop=True)


def with_merges(lp, merges):
    """the pairs of years with the merged classes (from_m, to_m); from_sub / to_sub keep the classes as recorded."""
    lp = lp.copy()
    lp["from_m"] = lp["from_sub"].map(lambda v: merges.get(v, v))
    lp["to_m"] = lp["to_sub"].map(lambda v: merges.get(v, v))
    return lp


def compare_levels(recorded, reported):
    """level 1, all pairs: recorded / reported / difference."""
    a, b = recorded["level1"].value_counts(), reported["level1"].value_counts()
    order = [c for c in CLASS_ORDER if c in a.index or c in b.index]
    t = pd.DataFrame({"recorded": a.reindex(order, fill_value=0), "reported": b.reindex(order, fill_value=0)})
    t["difference"] = t["reported"] - t["recorded"]
    t.loc["All plot-pairs"] = t.sum()
    return t



# ----------------------------------------------------------------------------------------------- v17: density order (Step 6)

def density_rules(classes, order):
    """the rules of a density order ('Dense > Moderate > Open', densest first): two forest subdivisions of the same type
    ('Montane - Dense', 'Montane - Open') = the same name once the density word is taken away. A change to a lower density =
    Degradation, to a higher one = Enhancement. {(from, to): rule}."""
    words = [w.strip() for w in str(order).split(">") if w.strip()]
    if len(words) < 2:
        return {}
    rank = {simple_words(w): i for i, w in enumerate(words)}
    parts = {}
    for sub in classes.loc[classes["IPCC category"] == FOREST_LAND, "subdivision"]:
        for w in sorted(words, key=len, reverse=True):
            low = str(sub).lower()
            k = low.rfind(w.lower())
            if k >= 0 and low[k:].strip() == w.lower():
                kind = str(sub)[:k].strip(" -_/()")
                parts[sub] = (simple_words(kind), rank[simple_words(w)])
                break
    rules = {}
    for a, (ka, ra) in parts.items():
        for b, (kb, rb) in parts.items():
            if a != b and ka == kb and ra != rb:
                rules[(a, b)] = "Degradation" if rb > ra else "Enhancement"
    return rules


# ----------------------------------------------------------------------------------------------- v17: sampling design (Step 7)

def find_strata(data):
    """the strata of a stratified sample in the export: a column of numbers > 0 with 2-50 values (the hectares of a plot) and
    another column that has one value for each of them (the stratum). (stratum column, weight column) or (None, None).
    Several: the weight column with the largest total."""
    nums = {}
    for c in data.columns:
        v = pd.to_numeric(data[c], errors="coerce")
        if v.notna().all() and (v > 0).all() and 2 <= v.nunique() <= 50:
            nums[c] = v
    found = []
    for w, v in nums.items():
        for c in data.columns:
            if c == w or c.endswith("_label"):
                continue
            k = data[c].astype(str).str.strip()
            if k.nunique() == v.nunique() and (v.groupby(k).nunique() == 1).all():
                found.append((v.sum(), c, w))
                break
    if not found:
        return None, None
    _, c, w = max(found)
    return c, w


def read_stratum_areas(text):
    """'0: 14310000; 1: 2,000,000' -> {'0': 14310000.0, '1': 2000000.0}; problems listed."""
    out, bad = {}, []
    for e in [e.strip() for e in str(text).split(";") if e.strip()]:
        if ":" not in e:
            bad.append(f"'{e}': write  stratum: area")
            continue
        k, v = [p.strip() for p in e.rsplit(":", 1)]
        try:
            out[k] = float(v.replace(",", "").replace(" ", ""))
        except ValueError:
            bad.append(f"'{e}': the area is not a number")
    return out, bad


def plot_strata(survey, plots, design, total_area, stratum_column="", stratum_areas=""):
    """the stratum and the weight (ha) of each plot used. Returns (DataFrame plot_id, stratum, w), table of the strata, text."""
    plots = pd.Index(sorted(set(plots)))
    if design != "stratified":
        n = len(plots)
        w = float(total_area) / n if total_area else 0.0
        t = pd.DataFrame({"stratum": ["all"], "plots": [n], "ha per plot": [w], "area_ha": [float(total_area)]})
        return pd.DataFrame({"plot_id": plots, "stratum": "all", "w": w}), t, "Systematic sample: every plot has the same weight."
    data = survey["data"]
    ids = data[survey["id_col"]].astype(str).str.strip()
    col, wcol = (stratum_column.strip(), None) if stratum_column.strip() else find_strata(data)
    if col and col not in data:
        raise ValueError(f"⛔ The column '{col}' is not in the export.")
    if not col:
        raise ValueError("⛔ No column of strata and weights found in the export: write stratum_column and stratum_areas.")
    stratum = pd.Series(data[col].astype(str).str.strip().values, index=ids)
    stratum = stratum[~stratum.index.duplicated()].reindex(plots)
    if stratum.isna().any():
        raise ValueError(f"⛔ {int(stratum.isna().sum())} plots have no stratum in '{col}'.")
    n = stratum.value_counts()
    areas, bad = read_stratum_areas(stratum_areas)
    if bad:
        raise ValueError("⛔ Correct stratum_areas:\n   " + "\n   ".join(bad))
    if areas:
        missing = sorted(set(n.index) - set(areas))
        if missing:
            raise ValueError(f"⛔ stratum_areas has no area for the strata {missing}.")
        A = pd.Series({h: areas[h] for h in n.index})
        how = f"Strata: column '{col}'; areas: stratum_areas."
    else:
        if wcol is None:                                   # stratum column written, no areas: a column of hectares per plot
            for c in data.columns:
                x = pd.to_numeric(data[c], errors="coerce")
                if c != col and x.notna().all() and (x > 0).all() and (x.groupby(data[col].astype(str).str.strip()).nunique() == 1).all() and x.nunique() > 1:
                    wcol = c
                    break
            if wcol is None:
                raise ValueError(f"⛔ No column gives the hectares of a plot in each stratum of '{col}': write stratum_areas.")
        wv = pd.Series(pd.to_numeric(data[wcol], errors="coerce").values, index=ids)
        wv = wv[~wv.index.duplicated()].reindex(plots)
        hw = wv.groupby(stratum).first()
        A = hw * n
        how = f"Strata: column '{col}'; hectares per plot: column '{wcol}' (area of a stratum = hectares per plot x its plots)."
    w = (A / n).reindex(stratum.values).values
    t = pd.DataFrame({"stratum": n.index, "plots": n.values, "ha per plot": (A / n).reindex(n.index).values, "area_ha": A.reindex(n.index).values})
    t = t.sort_values("stratum", key=lambda x: pd.to_numeric(x, errors="coerce").fillna(1e18)).reset_index(drop=True)
    if total_area and abs(A.sum() - total_area) > 0.005 * total_area:
        how += f"\n⚠ The strata add up to {A.sum():,.0f} ha, not {total_area:,.0f} ha (total_area_ha): the strata are used."
    return pd.DataFrame({"plot_id": plots, "stratum": stratum.values, "w": w}), t, how


def estimate(rows, keys, strata):
    """area and standard error of each group of plot-pairs (stratified estimator; one stratum = systematic):
    area = sum_h A_h p_h, var = sum_h A_h^2 p_h (1 - p_h) / (n_h - 1), p_h = share of the plots of stratum h in the group,
    n_h = plots of stratum h in that pair of years. rows: one row per plot and pair of years, with stratum."""
    A = strata.set_index("stratum")["area_ha"]
    nh = rows.groupby(["from_year", "stratum"])["plot_id"].nunique().rename("n_h")
    c = rows.groupby(keys + ["stratum"]).size().rename("k").reset_index()
    c = c.merge(nh.reset_index(), on=["from_year", "stratum"])
    c["A"] = c["stratum"].map(A)
    c["p"] = c["k"] / c["n_h"]
    c["area"] = c["A"] * c["p"]
    c["var"] = (c["A"] ** 2) * c["p"] * (1 - c["p"]) / (c["n_h"] - 1).clip(lower=1)
    g = c.groupby(keys).agg(plots=("k", "sum"), area_ha=("area", "sum"), var=("var", "sum")).reset_index()
    g["se_ha"] = g.pop("var") ** 0.5
    return g


MOCAREDD_ACTIVITY = {"Deforestation": "DF", "Degradation": "DG", "Afforestation": "EN", "Enhancement": "EN"}


def planted_of(classes, planted_text):
    """the forest subdivisions that are planted: the field (names separated by ;), else the names with 'plantation'."""
    forest = list(classes.loc[classes["IPCC category"] == FOREST_LAND, "subdivision"])
    if str(planted_text).strip():
        wanted = {simple_words(x) for x in str(planted_text).split(";") if x.strip()}
        return {f for f in forest if simple_words(f) in wanted}
    return {f for f in forest if "plantation" in simple_words(f)}


def mocaredd_transitions(rows, strata, planted):
    """one row per change of class and year: area and standard error, in the columns of the activity data of mocaredd.
    Only the REDD+ activities with a change of class (from > to differ); trans_period = the year of the change."""
    r = rows[rows["level1"].isin(list(MOCAREDD_ACTIVITY)) & (rows["from_adj"] != rows["to_adj"])].copy()
    if not len(r):
        return pd.DataFrame()
    est = estimate(rows.assign(_x=rows["level1"] + "|" + rows["from_adj"] + "|" + rows["to_adj"]), ["from_year", "to_year", "_x"], strata)
    est = est[est["_x"].isin(set(r["level1"] + "|" + r["from_adj"] + "|" + r["to_adj"]))]
    est[["level1", "lu_initial", "lu_final"]] = est["_x"].str.split("|", expand=True)
    est["redd_activity"] = est["level1"].map(MOCAREDD_ACTIVITY)

    def detail(x):
        if x["level1"] == "Deforestation":
            return "Deforestation (planted)" if x["lu_initial"] in planted else "Deforestation (natural)"
        if x["level1"] == "Afforestation":
            return "Afforestation (planted)" if x["lu_final"] in planted else "Afforestation (natural)"
        return x["level1"]
    est["redd_activity_details"] = est.apply(detail, axis=1)
    order = {"DF": 0, "DG": 1, "EN": 2}
    est = est.sort_values(["redd_activity", "to_year", "lu_initial", "lu_final"], key=lambda x: x.map(order) if x.name == "redd_activity" else x)
    out = pd.DataFrame({"trans_no": range(1, len(est) + 1), "trans_period": est["to_year"].astype(int).values,
                        "lu_initial": est["lu_initial"].values, "lu_final": est["lu_final"].values,
                        "trans_area": est["area_ha"].values, "trans_se": est["se_ha"].values, "trans_pdf": "Normal",
                        "trans_pdf_a": "", "trans_pdf_b": "", "trans_pdf_c": "",
                        "redd_activity": est["redd_activity"].values, "redd_activity_details": est["redd_activity_details"].values,
                        "plots": est["plots"].values})
    return out



# ----------------------------------------------------------------------------------------------- 2027: several files (Step 1)

EXPORT_TYPES = (".csv", ".xlsx", ".xls", ".xlsm")


def export_paths(text):
    """the files written in file_path: one file, several separated by ; or a folder (its csv / Excel files, in name order).
    [(path, from a folder)]: in a folder, a file that is not a survey export (e.g. a lookup) is left out in Step 1."""
    out = []
    for part in [p.strip().strip('"') for p in str(text).split(";") if p.strip()]:
        if os.path.isdir(part):
            found = sorted(f for f in os.listdir(part) if f.lower().endswith(EXPORT_TYPES) and not f.startswith("~$"))
            if not found:
                raise ValueError(f"⛔ No csv or Excel file in the folder {part}.")
            out += [(os.path.join(part, f), True) for f in found]
        elif os.path.isfile(part):
            out.append((part, False))
        else:
            raise ValueError(f"⛔ Not found: {part}")
    return out


def combine_exports(parts):
    """parts = [(name, table)]: one table, the column source_file says the file of each row. The files must be the same
    form: a column missing in a file is listed (left empty); files that share less than half of their columns stop."""
    if len(parts) == 1:
        name, t = parts[0]
        return t, []
    notes, allcols = [], []
    for _, t in parts:
        allcols += [c for c in t.columns if c not in allcols]
    for name, t in parts:
        shared = len(set(t.columns) & set(parts[0][1].columns)) / max(len(set(t.columns) | set(parts[0][1].columns)), 1)
        if shared < 0.5:
            raise ValueError(f"⛔ {name} is not the same form as {parts[0][0]} (only {shared:.0%} of the columns in common): "
                             "read different surveys separately; their unified formats can be put together after Step 4.")
        missing = [c for c in allcols if c not in t.columns]
        if missing:
            notes.append(f"{name}: {len(missing)} column(s) of the other files are not in it (left empty), e.g. {', '.join(missing[:5])}")
    table = pd.concat([t.assign(source_file=name) for name, t in parts], ignore_index=True)
    return table[[c for c in allcols] + ["source_file"]].fillna(""), notes


def duplicate_ids(data, id_col, point_col=None):
    """the plot ids that are in the export more than once (with the file of each row when there are several files)."""
    keys = [id_col] + ([point_col] if point_col and point_col in data else [])
    d = data[data[id_col].map(given) & data.duplicated(keys, keep=False)]
    out = []
    for pid, g in d.groupby(id_col):
        where = (", ".join(f"{f} row {i + 2}" for f, i in zip(g["source_file"], g.index))
                 if "source_file" in g else ", ".join(f"row {i + 2}" for i in g.index))
        out.append(f"{pid}: {len(g)} times ({where[:200]})")
    return out


# ----------------------------------------------------------------------------------------------- 2027: gaps (Step 3)

def year_gaps(table):
    """years without land between two years with land, per plot: (report rows, plots). Nothing is filled or changed."""
    t = table[["plot_id", "year", "land_use"]].copy()
    t["has"] = t["land_use"].map(given)
    rows = []
    for pid, g in t.sort_values("year").groupby("plot_id", sort=False):
        have = g.loc[g["has"], "year"]
        if not len(have):
            rows.append({"check": "6 gaps", "level": "⚠ to check", "plot": pid, "detail": "no land in any year"})
            continue
        gap = g[(~g["has"]) & (g["year"] > have.min()) & (g["year"] < have.max())]["year"].tolist()
        if gap:
            rows.append({"check": "6 gaps", "level": "⚠ to check", "plot": pid,
                         "detail": f"no land in {', '.join(map(str, gap[:8]))}{' ...' if len(gap) > 8 else ''} (between {have.min()} and {have.max()})"})
    return pd.DataFrame(rows, columns=["check", "level", "plot", "detail"])


# ----------------------------------------------------------------------------------------------- 2027: canopy before / after (Step 4)

def canopy_value(v):
    """a canopy as the survey wrote it: a number (45, '45%', '45,5') or a class ('40-70', '>70', '<10') -> (low, high)
    for a class, (x, x) for a number; None when it is not a canopy."""
    t = str(v).strip().replace(",", ".").replace("%", "").replace(" ", "")
    if not given(v) or not t:
        return None
    m = re.fullmatch(r"(-?\d+(?:\.\d+)?)", t)
    if m:
        x = float(m[1])
        return (x, x)
    m = re.fullmatch(r"(?:>=|>|<=|<)?(\d+(?:\.\d+)?)[-–_](\d+(?:\.\d+)?)", t)      # '40-70', '> 30-50 %'

    if m:
        return (float(m[1]), float(m[2]))
    m = re.fullmatch(r"(>|>=|<|<=)(\d+(?:\.\d+)?)", t)
    if m:
        x = float(m[2])
        return (x, 100.0) if m[1].startswith(">") else (0.0, x)
    return None


def survey_canopy_classes(values):
    """the canopy classes the survey wrote itself (e.g. '> 30-50 %', '> 50 %'), lowest first: [(low, high)]. Empty when the
    survey writes numbers only (e.g. 45, 72)."""
    found = {canopy_value(v) for v in values if given(v)}
    return sorted(c for c in found if c and c[0] != c[1])


def canopy_class(v, classes):
    """the position of a canopy among the survey's classes: a class = its own position; a number = the class it falls in,
    -1 below the lowest class, len(classes) above the highest."""
    if v in classes:
        return classes.index(v)
    x = (v[0] + v[1]) / 2
    for i, (lo, hi) in enumerate(classes):
        if lo <= x < hi or (i == len(classes) - 1 and x == hi):
            return i
    return -1 if x < classes[0][0] else len(classes)


def canopy_word(before, after, threshold, classes):
    """stable / decrease / increase between two canopies: by the survey's classes when it has them (a change of class), else
    by the threshold in points ('' when there is no threshold)."""
    if classes:
        a, b = canopy_class(before, classes), canopy_class(after, classes)
        return "stable" if a == b else ("decrease" if b < a else "increase")
    if not threshold:
        return ""
    d = (after[0] + after[1]) / 2 - (before[0] + before[1]) / 2
    return "stable" if abs(d) < threshold else ("decrease" if d < 0 else "increase")


CANOPY_WORDS = {"none": "stable", "no change": "stable", "no": "stable", "unchanged": "stable", "stable": "stable",
                "decrease": "decrease", "decreased": "decrease", "loss": "decrease", "increase": "increase",
                "increased": "increase", "gain": "increase"}


def canopy_changes(table, threshold=0):
    """where the canopy recorded for a plot differs from the year before (canopy cover on forest land, tree cover on other
    land): tree_cover_before, tree_cover_after (as the survey wrote them) and tree_cover_change (stable / decrease /
    increase) in that year. The classes are those the survey wrote (found by themselves); a survey that writes numbers
    needs threshold (points). Values the survey gave itself are kept. Nothing is filled or interpolated.
    Returns (table, counts, classes)."""
    t = table.copy()
    t["tree_cover_change"] = t["tree_cover_change"].map(lambda v: CANOPY_WORDS.get(plain(v), v) if given(v) else v)   # the survey's own words
    cc = t["canopy_cover"].where(t["canopy_cover"].map(given), t["tree_cover"])
    classes = survey_canopy_classes(cc.unique())
    if not (classes or threshold):
        return t, {}, classes
    order = t.sort_values(["plot_id", "year"]).index
    counts, prev, prev_plot = Counter(), None, None
    for i in order:
        pid, v = t.at[i, "plot_id"], cc[i]
        if pid != prev_plot:
            prev, prev_plot = None, pid
        if not given(v):
            continue
        if prev is not None and str(v).strip() != str(prev).strip():
            a, b = canopy_value(prev), canopy_value(v)
            if a and b and not any(given(t.at[i, c]) for c in ("tree_cover_before", "tree_cover_after", "tree_cover_change")):
                w = canopy_word(a, b, float(threshold or 0), classes)
                if w:
                    t.at[i, "tree_cover_before"], t.at[i, "tree_cover_after"], t.at[i, "tree_cover_change"] = str(prev), str(v), w
                    counts[w] += 1
        prev = v
    return t, dict(counts), classes


# ----------------------------------------------------------------------------------------------- 2027: canopy rule (Step 6)

CANOPY_RULES = ["not used", "decrease = degradation", "decrease = degradation, increase = enhancement"]


def apply_canopy_rule(classified, lp, rule):
    """in stable forest (same forest type in the two years), a canopy decrease recorded in the second year = Degradation;
    with the second rule, an increase = Enhancement. based_on = 'canopy'. Returns (classified, counts)."""
    if rule == CANOPY_RULES[0] or "tree_cover_change" not in lp:
        return classified, {}
    c = classified.copy()
    tc = lp["tree_cover_change"].reindex(c.index).astype(str)
    counts = {}
    todo = [("decrease", "Degradation")] + ([("increase", "Enhancement")] if rule == CANOPY_RULES[2] else [])
    for word, cls in todo:
        m = (c["level1"] == "Stable forest") & (tc == word)
        if m.any():
            fs = c.loc[m, "from_adj"].astype(str).str.strip().str.replace(" ", "_")
            c.loc[m, "level1"] = cls
            c.loc[m, "level2"] = cls + "_in_" + fs
            d = c.loc[m, "driver"].fillna("").astype(str)
            c.loc[m, "level3"] = cls + "_in_" + fs + d.map(lambda x: f"_due_{x}" if x else "")
            c.loc[m, "based_on"] = "canopy"
            counts[cls] = int(m.sum())
    return c, counts


# ----------------------------------------------------------------------------------------------- 2027: review of samples (Step 9)

def complexity(lp, classified):
    """per plot: changes of class recorded, decisions made by a rule, the canopy or 'Not possible', disturbances."""
    k = pd.DataFrame({"plot_id": lp["plot_id"],
                      "changes": (lp["from_sub"] != lp["to_sub"]).astype(int),
                      "rules": lp["rule"].astype(str).str.strip().ne("").astype(int),
                      "not_possible": lp["rule"].astype(str).eq("Not possible").astype(int),
                      "canopy": classified["based_on"].astype(str).str.startswith("canopy").astype(int).values,
                      "disturbance": lp.get("disturbance", pd.Series("", index=lp.index)).astype(str).eq("yes").astype(int)})
    g = k.groupby("plot_id").sum()
    g["score"] = g["changes"] + 2 * g["rules"] + 3 * g["not_possible"] + 2 * g["canopy"] + g["disturbance"]
    return g


def pick_samples(score, n=5, seed=1):
    """n plots at random among the complex ones (score >= 3; else the highest scores)."""
    pool = score[score["score"] >= 3]
    if len(pool) < n:
        pool = score.sort_values("score", ascending=False).head(max(n, 1))
    return list(pool.sample(min(n, len(pool)), random_state=int(seed)).index)


def review_rows(table, classified, pid):
    """the plot year by year: recorded land and canopy, what was recorded, the class reported for the pair ending that year."""
    t = table[table["plot_id"].astype(str) == str(pid)].sort_values("year")
    c = classified[classified["plot_id"].astype(str) == str(pid)].set_index("to_year")
    rows = []
    for r in t.to_dict("records"):
        y = r["year"]
        k = c.loc[y] if y in c.index else None
        cc = r["canopy_cover"] if given(r["canopy_cover"]) else r["tree_cover"]
        rows.append({"year": y, "source": r["source"], "recorded": f"{r['land_use']} / {r['subdivision']}",
                     "canopy": cc, "canopy change": r.get("tree_cover_change", ""),
                     "recorded change": ", ".join(x for x in (r.get("change_type", ""), "disturbance" if r.get("disturbance") == "yes" else "") if given(x)),
                     "reported": "" if k is None else f"{k['to_ipcc']} / {k['to_adj']}",
                     "class (pair ending this year)": "" if k is None else k["level1"],
                     "decided by": "" if k is None else (k["based_on"] or "land use"),
                     "driver": "" if k is None else k["driver"], "note": r.get("note", "")})
    return pd.DataFrame(rows)


def write_review(path, table, classified, picks, score, survey):
    """one page: each sample year by year, with what was decided automatically, to be checked on the images."""
    parts = [f"<h1>Review of {len(picks)} samples</h1><p class='sub'>{escape(str(survey.get('file', '')))} - "
             f"{date.today().isoformat()}. Chosen at random among the samples with several changes or automatic decisions. "
             "For each one: is the land of each year right on the images, and is the class reported right?</p>"]
    for pid in picks:
        rows = review_rows(table, classified, pid)
        sc = score.loc[pid]
        first = table[table["plot_id"].astype(str) == str(pid)].iloc[0]
        where = f"x {first['x']}, y {first['y']}" if given(first.get("x", "")) else ""
        parts.append(f"<h2>Plot {escape(str(pid))}</h2><p class='hint'>{where} - changes of class {sc['changes']}, "
                     f"rules {sc['rules']}, not possible {sc['not_possible']}, canopy {sc['canopy']}, disturbance {sc['disturbance']}</p>")
        parts.append(rows.to_html(index=False, escape=True, na_rep="", border=0, classes="t"))
    css = PAGE_STYLE + "table.t{border-collapse:collapse;font-size:13px;margin:8px 0 24px} table.t td,table.t th{border:1px solid var(--line);padding:3px 6px;text-align:left}"
    html = f"<!doctype html><html><head><meta charset='utf-8'><title>Review of samples</title><style>{css}</style></head><body><main>{''.join(parts)}</main></body></html>"
    with open(path, "w", encoding="utf-8") as f:
        f.write(html)



# ----------------------------------------------------------------------------------------------- 2027: the land-name lexicon (Step 2)

LEXICON_ADDED = {}          # names added from the country's file: {plain name: IPCC category}


def lexicon_table():
    """the names read as an IPCC category: the built-in list and the names added from the country's file."""
    rows = [{"name": n, "IPCC category": cat, "source": "built-in"} for cat, names in IPCC_NAMES.items() for n in names]
    rows += [{"name": n, "IPCC category": cat, "source": "country file"} for n, cat in LEXICON_ADDED.items()]
    return pd.DataFrame(rows)


def read_lexicon_file(source):
    """the country's lexicon (sheet 'lexicon': columns name, IPCC category): its names are added to the built-in list.
    A name already read as another category, or a category that is not one of the six, stops."""
    book = excel_of(source)
    sheet = "lexicon" if "lexicon" in book.sheet_names else book.sheet_names[0]
    t = book.parse(sheet, dtype=str)
    if not {"name", "IPCC category"} <= set(t.columns):
        raise ValueError("⛔ The lexicon needs a sheet 'lexicon' with the columns name and IPCC category.")
    bad, added = [], {}
    for n, cat in zip(t["name"], t["IPCC category"]):
        if not (given(n) and given(cat)):
            continue
        c = next((k for k in IPCC_ORDER if plain(k) == plain(cat)), None)
        if c is None:
            bad.append(f"'{n}': '{cat}' is not one of {IPCC_ORDER}")
        elif IPCC_LOOKUP.get(plain(n), c) != c:
            bad.append(f"'{n}' is already read as {IPCC_LOOKUP[plain(n)]}")
        else:
            added[plain(n)] = c
    if bad:
        raise ValueError("⛔ Correct the lexicon:\n   " + "\n   ".join(bad[:20]))
    new = {k: v for k, v in added.items() if k not in IPCC_LOOKUP}
    IPCC_LOOKUP.update(new)
    LEXICON_ADDED.update(new)
    return new


def land_names_in(survey):
    """the answers of the questions on the IPCC category (with their label when the export has one): how each was read."""
    roles, data = survey["roles"], survey["data"]
    cols = roles.loc[(roles["variable"] == "ipcc_category") & (roles["record"] != "-"), "export column"]
    seen = Counter()
    for c in cols:
        lab = f"{c}_label" if f"{c}_label" in data else None
        for v, l in zip(data[c], data[lab] if lab else [None] * len(data)):
            if given(v):
                seen[(str(v).strip(), str(l).strip() if given(l) else "")] += 1
    rows = []
    for (v, l), n in sorted(seen.items(), key=lambda x: -x[1]):
        cat = ipcc_category(v, l or None)
        key = plain(v) if plain(v) in IPCC_LOOKUP else plain(l)
        how = "" if cat is None else ("country file" if key in LEXICON_ADDED else "built-in")
        rows.append({"answer": v, "label": l, "answers": n, "read as": cat or "not recognised", "by": how})
    return pd.DataFrame(rows, columns=["answer", "label", "answers", "read as", "by"])


def write_lexicon(path, found):
    """the land names found and the lexicon, for the country to add its own names (sheet 'lexicon')."""
    guide = pd.DataFrame([("What it is", "How the answers on the IPCC category were read (sheet found) and the names the notebook knows (sheet lexicon)."),
                          ("Add", "Add rows to the sheet lexicon: a name the survey uses (column name) and one of the six IPCC categories (column IPCC category). Only names of the six categories: a national class counted in another category is a choice of Step 6 (reclassify)."),
                          ("Then", "Write this file in land_names_file (Step 2) and run Step 2 again.")], columns=["", "how"])
    with pd.ExcelWriter(path, engine="xlsxwriter") as w:
        guide.to_excel(w, sheet_name="README", index=False)
        found.to_excel(w, sheet_name="found", index=False)
        lexicon_table().to_excel(w, sheet_name="lexicon", index=False)
        w.sheets["README"].set_column(0, 0, 12); w.sheets["README"].set_column(1, 1, 130)
        for sh in ("found", "lexicon"):
            w.sheets[sh].set_column(0, 4, 26)


def add_to_page(path, title, frame, text=""):
    """a section added at the end of an HTML page of the notebook."""
    page = open(path, encoding="utf-8").read()
    part = f"<h2>{escape(title)}</h2>" + (f"<p class='hint'>{escape(text)}</p>" if text else "") + frame.to_html(index=False, border=0, classes="t")
    page = page.replace("</main>", part + "</main>", 1)
    open(path, "w", encoding="utf-8").write(page.replace("</style>", "table.t{border-collapse:collapse;font-size:13px} table.t td,table.t th{border:1px solid var(--line);padding:3px 6px;text-align:left}</style>", 1))



# ----------------------------------------------------------------------------------------------- 2027: SE in the matrices file (Step 7)

Z90 = 1.645                 # 90% confidence interval = 1.645 x SE


def strata_of(lp):
    """the strata and their areas from the pairs of Step 7 (the weights of the plots): DataFrame stratum, area_ha; None without area."""
    if not {"w", "stratum"} <= set(lp.columns):
        return None
    p = lp.drop_duplicates("plot_id")
    return p.groupby("stratum")["w"].sum().rename("area_ha").reset_index()


def areas_by_year(lp, strata, side):
    """the area of each class in each year with its standard error and 90% CI: side = 'ipcc' (IPCC category) or 'adj'
    (subdivision, as reported). A year = the land at its end: the first year of each pair, and the last year of the last pair."""
    firsts = set(lp["from_year"].astype(int))
    last = int(lp["to_year"].max())
    out = []
    for y in sorted(firsts) + [last]:
        g, col = (lp[lp["from_year"] == y], f"from_{side}") if y in firsts else (lp[lp["to_year"] == last], f"to_{side}")
        rows = g[["plot_id", "stratum", col]].rename(columns={col: "class"}).assign(from_year=y)
        e = estimate(rows, ["from_year", "class"], strata)
        e.insert(0, "year", y)
        out.append(e.drop(columns="from_year"))
    t = pd.concat(out, ignore_index=True)
    t["ci90_ha"] = Z90 * t["se_ha"]
    t["rel_ci90"] = (t["ci90_ha"] / t["area_ha"]).where(t["area_ha"] > 0)
    order = IPCC_ORDER if side == "ipcc" else list(dict.fromkeys(t["class"]))
    t["_o"] = t["class"].map({c: i for i, c in enumerate(order)}).fillna(99)
    return t.sort_values(["year", "_o", "class"]).drop(columns="_o").reset_index(drop=True)


FMT = {}


def write_block(s, row, title, keep, m, fmt, fmt_d, row_tot, col_tot, grand):
    """one matrix block: title, header, rows, totals."""
    s.write(row, 0, title, FMT["title"])
    row += 1
    s.write(row, 0, "Initial (rows) / Final (columns)", FMT["head"])
    for c, name in enumerate(keep, 1):
        s.write(row, c, name, FMT["head"])
    s.write(row, len(keep) + 1, "Total initial", FMT["head"])
    for i, name in enumerate(keep):
        row += 1
        s.write(row, 0, name, FMT["head"])
        for j, other in enumerate(keep):
            s.write_number(row, j + 1, float(m.at[name, other]), fmt_d if i == j else fmt)
        s.write_number(row, len(keep) + 1, float(row_tot.get(name, 0)), fmt)
    row += 1
    s.write(row, 0, "Total final", FMT["head"])
    for j, other in enumerate(keep):
        s.write_number(row, j + 1, float(col_tot.get(other, 0)), fmt)
    s.write_number(row, len(keep) + 1, float(grand), fmt)
    return row + 2


def write_matrix(w, sheet, title, a, b, order, per_plot, formats, wt=None, se_rows=None, strata=None):
    """rows = initial, columns = final, as in the example of the IPCC guide: plots, hectares (the sum of the weights) and,
    with the strata, the standard error of each cell and of each total (stratified estimator; the grand total is the total
    area: no error)."""
    FMT.update(formats)
    t = pd.crosstab(a, b).reindex(index=order, columns=order, fill_value=0)
    h = pd.crosstab(a, b, values=wt, aggfunc="sum").reindex(index=order, columns=order).fillna(0) if wt is not None else t * per_plot
    keep = [x for x in order if t.loc[x].sum() or t[x].sum()]
    t, h = t.loc[keep, keep], h.loc[keep, keep]
    s = w.book.add_worksheet(sheet[:31])
    w.sheets[sheet[:31]] = s
    s.set_column(0, 0, 30); s.set_column(1, len(keep) + 1, 13)
    row = write_block(s, 0, f"{title} - plots", keep, t, formats["int"], formats["int_d"], t.sum(axis=1), t.sum(axis=0), t.values.sum())
    if per_plot:
        row = write_block(s, row, f"{title} - hectares", keep, h, formats["ha"], formats["ha_d"], h.sum(axis=1), h.sum(axis=0), h.values.sum())
    if per_plot and se_rows is not None and strata is not None:
        d = se_rows.reset_index(drop=True).assign(from_year=0, _a=list(a), _b=list(b))
        cell_se = (estimate(d, ["from_year", "_a", "_b"], strata).pivot(index="_a", columns="_b", values="se_ha")
                   .reindex(index=keep, columns=keep).fillna(0))
        r_se = estimate(d, ["from_year", "_a"], strata).set_index("_a")["se_ha"]
        c_se = estimate(d, ["from_year", "_b"], strata).set_index("_b")["se_ha"]
        write_block(s, row, f"{title} - standard error (ha); 90% CI = 1.645 x SE", keep, cell_se, formats["ha"], formats["ha_d"], r_se, c_se, 0)


def write_areas(w, name, t, formats):
    t = t.rename(columns={"area_ha": "area (ha)", "se_ha": "SE (ha)", "ci90_ha": "90% CI (ha)", "rel_ci90": "90% CI / area"})
    t.to_excel(w, sheet_name=name[:31], index=False)
    s = w.sheets[name[:31]]
    s.set_column(0, 0, 8); s.set_column(1, 1, 30); s.set_column(2, 2, 9)
    s.set_column(3, 5, 15, w.book.add_format({"num_format": "#,##0.0"})); s.set_column(6, 6, 13, w.book.add_format({"num_format": "0.0%"}))


def write_matrices(path, lp, classes, per_plot, total_area):
    """2027: + the area of each class in each year with SE and 90% CI (IPCC categories and subdivisions), and the SE of the
    cells of every matrix."""
    subs = subdivision_order(classes)
    strata = strata_of(lp) if per_plot else None
    with pd.ExcelWriter(path, engine="xlsxwriter") as w:
        book = w.book
        fm = {"title": book.add_format({"bold": True}), "head": book.add_format({"bold": True, "bg_color": "#DCE6F1", "border": 1, "text_wrap": True}),
              "int": book.add_format({"num_format": "#,##0", "border": 1}), "ha": book.add_format({"num_format": "#,##0.0", "border": 1}),
              "int_d": book.add_format({"num_format": "#,##0", "border": 1, "bg_color": "#D9D9D9"}), "ha_d": book.add_format({"num_format": "#,##0.0", "border": 1, "bg_color": "#D9D9D9"})}
        info = [("What it is", "Land-use conversion matrices. Rows = land in the first year, columns = land in the second year. Grey = no change."),
                ("Land", "As reported (reclassify, merge); the lookup does not change the forest types recorded."),
                ("Area", (f"Total area used: {total_area:,.0f} ha; the hectares are the sum of the weights of the plots (sampling design: see Step 7)." if "w" in lp else f"Total area used: {total_area:,.0f} ha; each plot = {per_plot:,.2f} ha.") if per_plot else "No total area given: plots only."),
                ("Areas by year", "Sheets 'Areas by year (IPCC)' and 'Areas by year (subdivisions)': the area of each class in each year (the land at the end of the year), with its standard error and 90% CI." if strata is not None else "No total area given: no areas."),
                ("Standard error", "Stratified estimator (one stratum when systematic): area = sum of A_h x p_h, variance = sum of A_h^2 x p_h (1 - p_h) / (n_h - 1); 90% CI = 1.645 x SE. Each matrix has a third block with the SE of each cell and of each total (the grand total is the total area: no error)." if strata is not None else ""),
                ("Note", "In the IPCC software's matrix tab, rows are final and columns initial: the opposite of this layout and of the guide's Excel example.")]
        pd.DataFrame(info, columns=["", "note"]).to_excel(w, sheet_name="README", index=False)
        w.sheets["README"].set_column(0, 0, 16); w.sheets["README"].set_column(1, 1, 130)
        if strata is not None:
            write_areas(w, "Areas by year (IPCC)", areas_by_year(lp, strata, "ipcc"), fm)
            write_areas(w, "Areas by year (subdivisions)", areas_by_year(lp, strata, "adj"), fm)
        whole = whole_period(lp)
        first, last = int(lp["from_year"].min()), int(lp["to_year"].max())
        wt = whole["w"] if "w" in whole else None
        wrows = (whole[["plot_id"]].merge(lp.drop_duplicates("plot_id")[["plot_id", "stratum"]], on="plot_id", how="left")
                 if strata is not None else None)
        write_matrix(w, "IPCC whole period", f"IPCC categories {first}-{last}", whole["from_ipcc"], whole["to_ipcc"], IPCC_ORDER, per_plot, fm, wt, wrows, strata)
        write_matrix(w, "Subdivisions whole period", f"Subdivisions {first}-{last}", whole["from_adj"], whole["to_adj"], subs, per_plot, fm, wt, wrows, strata)
        for (y0, y1), g in lp.groupby(["from_year", "to_year"]):
            write_matrix(w, f"{y0}-{y1}", f"Subdivisions {y0}-{y1}", g["from_adj"], g["to_adj"], subs, per_plot, fm,
                         g["w"] if "w" in g else None, g[["plot_id", "stratum"]] if strata is not None else None, strata)



# ----------------------------------------------------------------------------------------------- 2027: the activity of a period (Step 10)

PERIOD_EVENTS = ["Deforestation", "Degradation", "Afforestation", "Enhancement"]


def period_histories(classified, first, last):
    """per plot, the changes of years first..last (pairs ending in those years), reported classes: the forest events in the
    order they happened (each event, also when the same activity comes again), joined by ' -> '; no event: Stable forest (forest in
    every year), Stable non-forest or Non-forest change. With the land at the start and at the end and the year of each event."""
    c = classified[(classified["to_year"] >= first) & (classified["to_year"] <= last)].sort_values(["plot_id", "to_year"])
    rows = []
    for pid, g in c.groupby("plot_id", sort=False):
        ev = [(r["level1"], int(r["to_year"])) for r in g.to_dict("records") if r["level1"] in PERIOD_EVENTS]
        seq = ev
        if seq:
            history = " -> ".join(e[0] for e in seq)
        elif (g["from_ipcc"].eq(FOREST_LAND) & g["to_ipcc"].eq(FOREST_LAND)).all():
            history = "Stable forest"
        elif g["level1"].eq("Stable non-forest").all():
            history = "Stable non-forest"
        else:
            history = "Non-forest change" if not g["to_ipcc"].eq(FOREST_LAND).any() else "Forest, other change"
        rows.append({"plot_id": pid, "history": history, "land_start": g["from_ipcc"].iloc[0] + " / " + str(g["from_adj"].iloc[0]),
                     "land_end": g["to_ipcc"].iloc[-1] + " / " + str(g["to_adj"].iloc[-1]),
                     "event_years": ", ".join(f"{e[0]} {e[1]}" for e in ev)})
    return pd.DataFrame(rows)


def period_proposal(history):
    """a proposal for the template (the country decides): one event = that activity; no event = as written; several
    events = 'to decide' (e.g. Deforestation -> Afforestation in the same period)."""
    parts = history.split(" -> ")
    return parts[0] if len(parts) == 1 else "to decide"


def write_period_template(path, hist, first, last):
    t = hist["history"].value_counts().rename_axis("history").reset_index(name="plots")
    t["proposal"] = t["history"].map(period_proposal)
    t["redd_activity"] = ""
    t["comment"] = ""
    guide = pd.DataFrame([("What it is", f"The histories of the plots in the period {first}-{last} (the forest events in the order they happened, reported classes)."),
                          ("Fill", "The column redd_activity: the activity of the period for each history (e.g. Deforestation, Degradation, Afforestation, Enhancement, Stable forest, Stable non-forest). 'proposal' is only a suggestion; 'to decide' = several events."),
                          ("Then", "Save the file, choose 'use my filled file' in Step 10 and run it again.")], columns=["", "how"])
    with pd.ExcelWriter(path, engine="xlsxwriter") as w:
        guide.to_excel(w, sheet_name="README", index=False)
        t.to_excel(w, sheet_name="history_lookup", index=False)
        w.sheets["README"].set_column(0, 0, 14); w.sheets["README"].set_column(1, 1, 120)
        w.sheets["history_lookup"].set_column(0, 0, 60); w.sheets["history_lookup"].set_column(1, 5, 18)


def read_period_lookup(source):
    """the filled history_lookup: {history: redd_activity}; a file of the workbook notebook works too."""
    book = excel_of(source)
    sheet = "history_lookup" if "history_lookup" in book.sheet_names else book.sheet_names[0]
    t = book.parse(sheet, dtype=str)
    if not {"history", "redd_activity"} <= set(t.columns):
        raise ValueError("⛔ The file needs a sheet history_lookup with the columns history and redd_activity.")
    return {str(a).strip(): str(b).strip() for a, b in zip(t["history"], t["redd_activity"]) if given(a) and given(b)}



# ----------------------------------------------------------------------------------------------- canopy before / after (Step 4) and the canopy cover transition table (Step 6)

def canopy_changes(table, threshold=0):
    """where the canopy recorded for a plot differs from the year before (canopy cover on forest land, tree cover on other
    land): tree_cover_before and tree_cover_after as the survey wrote them, and tree_cover_change = decrease / increase /
    stable. Classes written by the survey (e.g. '> 30-50 %'): a change of class; numbers: any difference; words whose order
    is not known (e.g. 'dense', 'open'): before and after only. Values the survey gave itself are kept; nothing is filled
    or interpolated. What a change means is the country's choice, in the canopy cover transition table (Step 6).
    Returns (table, counts, classes)."""
    t = table.copy()
    t["tree_cover_change"] = t["tree_cover_change"].map(lambda v: CANOPY_WORDS.get(plain(v), v) if given(v) else v)   # the survey's own words
    cc = t["canopy_cover"].where(t["canopy_cover"].map(given), t["tree_cover"])
    classes = survey_canopy_classes(cc.unique())
    order = t.sort_values(["plot_id", "year"]).index
    counts, prev, prev_plot = Counter(), None, None
    for i in order:
        pid, v = t.at[i, "plot_id"], cc[i]
        if pid != prev_plot:
            prev, prev_plot = None, pid
        if not given(v):
            continue
        if prev is not None and str(v).strip() != str(prev).strip() and \
                not any(given(t.at[i, c]) for c in ("tree_cover_before", "tree_cover_after", "tree_cover_change")):
            a, b = canopy_value(prev), canopy_value(v)
            if a and b:
                if classes:
                    ka, kb = canopy_class(a, classes), canopy_class(b, classes)
                    w = "stable" if ka == kb else ("decrease" if kb < ka else "increase")
                else:
                    d = (b[0] + b[1]) / 2 - (a[0] + a[1]) / 2
                    w = "stable" if d == 0 else ("decrease" if d < 0 else "increase")
            else:
                w = ""                                   # words whose order the notebook does not know
            t.at[i, "tree_cover_before"], t.at[i, "tree_cover_after"], t.at[i, "tree_cover_change"] = str(prev), str(v), w
            counts[w or "changed (order not known)"] += 1
        prev = v
    return t, dict(counts), classes


def pair_canopy(lp):
    """the canopy of the first and of the second year of each pair (canopy cover, else tree cover), as written."""
    def side(s):
        a = lp[f"{s}_canopy_cover"] if f"{s}_canopy_cover" in lp else pd.Series("", index=lp.index)
        b = lp[f"{s}_tree_cover"] if f"{s}_tree_cover" in lp else pd.Series("", index=lp.index)
        return a.where(a.map(given), b).fillna("").astype(str).str.strip()
    return side("from"), side("to")


def same_forest_type(lp):
    """the pairs where the plot is forest in both years and keeps its forest type (as reported)."""
    a = lp["from_adj"] if "from_adj" in lp else lp["from_m"] if "from_m" in lp else lp["from_sub"]
    b = lp["to_adj"] if "to_adj" in lp else lp["to_m"] if "to_m" in lp else lp["to_sub"]
    return (lp["from_ipcc"] == FOREST_LAND) & (lp["to_ipcc"] == FOREST_LAND) & (a == b), a


def canopy_direction(a, b):
    x, y = canopy_value(a), canopy_value(b)
    if not (x and y):
        return ""
    d = (y[0] + y[1]) / 2 - (x[0] + x[1]) / 2
    return "stable" if d == 0 else ("decrease" if d < 0 else "increase")


def canopy_transitions_found(lp):
    """the canopy transitions in stable forest: from_cc > to_cc as the survey wrote them, with the forest types and plots."""
    keep, ftype = same_forest_type(lp)
    f, t = pair_canopy(lp)
    m = keep & f.map(given) & t.map(given) & (f != t)
    d = pd.DataFrame({"from_cc": f[m], "to_cc": t[m], "forest_type": ftype[m]})
    g = d.groupby(["from_cc", "to_cc"]).agg(plots=("forest_type", "size"), forest_types=("forest_type", lambda x: ", ".join(sorted(set(map(str, x)))))).reset_index()
    g.insert(2, "direction", [canopy_direction(a, b) for a, b in zip(g["from_cc"], g["to_cc"])])
    return g.sort_values("plots", ascending=False).reset_index(drop=True)


def write_canopy_template(path, found):
    """the canopy cover transition table to fill: one row per canopy transition found in stable forest."""
    guide = [("What it is", "The country's reading of a change of canopy cover in stable forest (the same forest type in the two years). "
                            "Separate from the transition table between forest types, with the same words."),
             ("Sheet canopy", "One row per canopy transition: from_cc and to_cc as the survey writes the canopy (a class such as > 30-50 %, "
                              "a number, or a word). A range (e.g. 70-100) can replace numbers: a number is read as the range it falls in, and a change inside one range is no transition. "
                              "forest_type (optional): the rule only for that forest type; empty = every forest type."),
             ("classification", "Degradation, Enhancement, Stable forest (no REDD+ activity) or Not possible (not applied, listed to review) - "
                                "the words of the transition table. A canopy transition found in the data with no rule: class To review."),
             ("Read by the notebook", "from_cc, to_cc, classification and, if given, forest_type. direction, plots and forest_types are for the reader."),
             ("Then", "Save the file and write it in canopy_lookup_file (Step 6), with 'use my filled file'.")]
    with pd.ExcelWriter(path, engine="xlsxwriter") as w:
        pd.DataFrame(guide, columns=["", "how to fill it"]).to_excel(w, sheet_name="README", index=False)
        w.sheets["README"].set_column(0, 0, 20); w.sheets["README"].set_column(1, 1, 130)
        t = found.assign(forest_type="", classification="", comment="")[["from_cc", "to_cc", "direction", "forest_type", "classification", "plots", "forest_types", "comment"]]
        t.to_excel(w, sheet_name="canopy", index=False)
        s = w.sheets["canopy"]
        yellow = w.book.add_format({"bg_color": "#FFF2CC", "border": 1})
        s.set_column(0, 1, 16); s.set_column(2, 2, 11); s.set_column(3, 4, 18); s.set_column(5, 5, 8); s.set_column(6, 7, 40)
        for r in range(1, len(t) + 1):
            s.write_blank(r, 4, None, yellow)
        if len(t):
            s.data_validation(1, 4, len(t), 4, {"validate": "list", "source": LOOKUP_CHOICES})


def read_canopy_lookup(source):
    """the filled canopy cover transition table: [(from_cc, to_cc, forest_type, word)]. Stops on a word it does not know."""
    book = excel_of(source)
    sheet = "canopy" if "canopy" in book.sheet_names else book.sheet_names[0]
    t = book.parse(sheet, dtype=str)
    col = next((c for c in t.columns if simple_words(c) in ("classification", "canopy transition") or simple_words(c).startswith("transition")), None)
    if col is None or not {"from_cc", "to_cc"} <= set(t.columns):
        raise ValueError("⛔ The canopy cover transition table needs a sheet canopy with the columns from_cc, to_cc and classification.")
    rules, bad = [], []
    for r in t.to_dict("records"):
        if not (given(r["from_cc"]) and given(r["to_cc"]) and given(r[col])):
            continue
        word = LOOKUP_WORDS.get(simple_words(r[col]))
        if word is None:
            bad.append(f"{r['from_cc']} > {r['to_cc']}: '{r[col]}' is not one of {LOOKUP_CHOICES}")
        else:
            rules.append((str(r["from_cc"]).strip(), str(r["to_cc"]).strip(), str(r.get("forest_type", "") or "").strip() if given(r.get("forest_type")) else "", word))
    if bad:
        raise ValueError("⛔ Correct the canopy cover transition table:\n   " + "\n   ".join(bad[:20]))
    return rules


def canopy_matches(value, spec):
    """a canopy as written matches a rule's value: the same words, or a number / class inside the rule's range."""
    if plain(value) == plain(spec):
        return True
    v, s = canopy_value(value), canopy_value(spec)
    if not (v and s):
        return False
    if s[0] == s[1]:
        return v == s
    x = (v[0] + v[1]) / 2
    return s[0] <= x < s[1] or (x == s[1] and s[1] >= 100)


def apply_canopy_lookup(classified, lp, rules):
    """in stable forest (forest in both years, same forest type, the class not decided otherwise), a canopy transition
    takes the class of its rule: Degradation or Enhancement (based_on = canopy), Stable forest (no activity), Not possible
    (kept as stable forest, listed); with no rule: To review. Returns (classified, counts, to_review, not_possible)."""
    c = classified.copy()
    keep, ftype = same_forest_type(lp)
    f, t = pair_canopy(lp)
    m = keep & f.map(given) & t.map(given) & (f != t) & (c["level1"] == "Stable forest")
    ranges = {x for a, b, ft, w in rules for x in (a, b) if canopy_value(x) and canopy_value(x)[0] != canopy_value(x)[1]}
    counts, review, impossible = Counter(), [], []
    for i in c.index[m]:
        if any(canopy_matches(f[i], r) and canopy_matches(t[i], r) for r in ranges):
            continue                                             # a change inside one range of the table: no transition
        hit = [(ft, w) for a, b, ft, w in rules if canopy_matches(f[i], a) and canopy_matches(t[i], b)
               and (not ft or plain(ft) == plain(ftype[i]))]
        rule = next((w for ft, w in hit if ft), None) or next((w for ft, w in hit), None)   # a rule for the forest type wins
        fs = str(ftype[i]).strip().replace(" ", "_")
        if rule in ("Degradation", "Enhancement"):
            d = str(c.at[i, "driver"] or "")
            c.at[i, "level1"], c.at[i, "level2"] = rule, f"{rule}_in_{fs}"
            c.at[i, "level3"] = f"{rule}_in_{fs}" + (f"_due_{d}" if d else "")
            c.at[i, "based_on"] = f"canopy {f[i]} > {t[i]}"
        elif rule is None:
            c.at[i, "level1"], c.at[i, "level2"], c.at[i, "level3"] = "To review", f"To_review_in_{fs}", f"To_review_canopy_in_{fs}"
            c.at[i, "based_on"] = f"canopy {f[i]} > {t[i]}: no rule"
            review.append((c.at[i, "plot_id"], c.at[i, "to_year"], f[i], t[i]))
        else:
            c.at[i, "based_on"] = f"canopy {f[i]} > {t[i]}: {rule}"
            if rule == "Not possible":
                impossible.append((c.at[i, "plot_id"], c.at[i, "to_year"], f[i], t[i]))
        counts[rule or "To review (no rule)"] += 1
    cols = ["plot_id", "year", "canopy before", "canopy after"]
    return c, dict(counts), pd.DataFrame(review, columns=cols), pd.DataFrame(impossible, columns=cols)



# ----------------------------------------------------------------------------------------------- the country rules workbook (Step 6)

RULE_SHEETS = ["lookup", "canopy", "category changes", "subclass merges"]


def read_canopy_lookup(source):
    """the sheet canopy of the country rules workbook: [(from_cc, to_cc, forest_type, word)]; [] when the workbook has no such
    sheet or it is empty. Stops on a word it does not know."""
    book = excel_of(source)
    if "canopy" not in book.sheet_names:
        return []
    t = book.parse("canopy", dtype=str)
    if t.empty:
        return []
    col = next((c for c in t.columns if simple_words(c) in ("classification", "canopy transition") or simple_words(c).startswith("transition")), None)
    if col is None or not {"from_cc", "to_cc"} <= set(t.columns):
        raise ValueError("⛔ The sheet canopy needs the columns from_cc, to_cc and classification.")
    rules, bad = [], []
    for r in t.to_dict("records"):
        if not (given(r["from_cc"]) and given(r["to_cc"]) and given(r[col])):
            continue
        word = LOOKUP_WORDS.get(simple_words(r[col]))
        if word is None:
            bad.append(f"{r['from_cc']} > {r['to_cc']}: '{r[col]}' is not one of {LOOKUP_CHOICES}")
        else:
            rules.append((str(r["from_cc"]).strip(), str(r["to_cc"]).strip(), str(r.get("forest_type")).strip() if given(r.get("forest_type")) else "", word))
    if bad:
        raise ValueError("⛔ Correct the sheet canopy:\n   " + "\n   ".join(bad[:20]))
    return rules


def write_rules_workbook(path, classes, lp, canopy_found):
    """the country rules workbook to fill, one sheet per kind of rule, every sheet optional: lookup (the changes between forest
    types found in the data), canopy (the canopy transitions found in stable forest), category changes (reclassify) and
    subclass merges (merge); a sheet classes when a land name is not recognised."""
    found = forest_changes(lp)
    types = list(classes.loc[classes["IPCC category"] == FOREST_LAND, "subdivision"])
    look = pd.DataFrame([{"from_sub": a, "to_sub": b, "found in your data": "yes" if (a, b) in found.index else "no",
                          "plots with this change": int(found.get((a, b), 0)), "classification": "", "comment": ""}
                         for a in types for b in types if a != b], columns=["from_sub", "to_sub", "found in your data", "plots with this change", "classification", "comment"])
    look = look.sort_values("plots with this change", ascending=False)
    canopy = canopy_found.assign(forest_type="", classification="", comment="")[["from_cc", "to_cc", "direction", "forest_type", "classification", "plots", "forest_types", "comment"]] \
        if len(canopy_found) else pd.DataFrame(columns=["from_cc", "to_cc", "direction", "forest_type", "classification", "plots", "forest_types", "comment"])
    cat = pd.DataFrame(columns=["sub", "new category", "reason"])
    mer = pd.DataFrame(columns=["sub", "merged into", "reason"])
    unknown = classes[classes["IPCC category"].map(lambda v: not given(v))][["subdivision", "land_use", "IPCC category"]]
    guide = [("What it is", "The country's rules for Step 6, in one workbook. Every sheet is optional: an empty sheet is not used. The notebook decides nothing by itself."),
             ("Sheet lookup", "The changes between two forest types: one row per change, from_sub and to_sub (the survey's codes), classification. Fill at least the rows found in your data."),
             ("Sheet canopy", "Only when the survey records the canopy: in stable forest (the same forest type in the two years), what a change of canopy means. from_cc and to_cc as the survey writes the canopy (a class such as > 30-50 %, a number, or a word); a range (e.g. 70-100) can stand for numbers, and a change inside one range is no transition. forest_type (optional): the rule only for that type; it wins over a rule for all."),
             ("classification", "In the sheets lookup and canopy: Degradation, Enhancement, Stable forest (no REDD+ activity) or Not possible (not applied, listed to review). The French words Dégradation, Amélioration, Forêt stable, Pas possible are read too. A change of the data with no rule: class To review."),
             ("Sheet category changes", "Reclassify, only if needed: a class counted in another IPCC category. sub (the survey's code) and new category (one of the six), e.g. wooded_savanna | Forest land."),
             ("Sheet subclass merges", "Merge, only if needed: classes counted as one. sub and merged into, e.g. gallery_forest | gallery_riparian_forest and riparian_forest | gallery_riparian_forest. A change between them is then no change; the merged class needs its own rows in the sheet lookup."),
             ("Read by the notebook", "lookup: from_sub, to_sub, classification; canopy: from_cc, to_cc, classification, forest_type; category changes: sub, new category; subclass merges: sub, merged into. The other columns are notes."),
             ("Then", "Save the file. In Step 6 choose 'use my filled file' and write its path (in Colab: leave it empty and choose the file).")]
    if len(unknown):
        guide.insert(1, ("Sheet classes", "Land names the notebook does not recognise: write their IPCC category in the yellow column."))
    with pd.ExcelWriter(path, engine="xlsxwriter") as w:
        book = w.book
        head = book.add_format({"bold": True, "bg_color": "#DCE6F1", "border": 1, "text_wrap": True, "valign": "top"})
        yellow = book.add_format({"bg_color": "#FFF2CC", "border": 1})
        pd.DataFrame(guide, columns=["", "how to fill it"]).to_excel(w, sheet_name="README", index=False)
        w.sheets["README"].set_column(0, 0, 22); w.sheets["README"].set_column(1, 1, 130)
        for name, t, col in (("lookup", look, 4), ("canopy", canopy, 4), ("category changes", cat, 1), ("subclass merges", mer, 1)):
            t.to_excel(w, sheet_name=name, index=False)
            s = w.sheets[name]
            for c, cname in enumerate(t.columns):
                s.write(0, c, cname, head)
            s.set_column(0, len(t.columns), 22)
            for r in range(1, max(len(t), 20 if name in ("category changes", "subclass merges") else 0) + 1):
                s.write_blank(r, col, None, yellow)
            if name in ("lookup", "canopy") and len(t):
                s.data_validation(1, col, len(t), col, {"validate": "list", "source": LOOKUP_CHOICES})
            if name == "category changes":
                s.data_validation(1, 1, 200, 1, {"validate": "list", "source": IPCC_ORDER})
        if len(unknown):
            unknown.to_excel(w, sheet_name="classes", index=False)
            s = w.sheets["classes"]
            s.set_column(0, 2, 24)
            for r in range(1, len(unknown) + 1):
                s.write_blank(r, 2, None, yellow)
            s.data_validation(1, 2, len(unknown), 2, {"validate": "list", "source": IPCC_ORDER})



# ----------------------------------------------------------------------------------------------- the history lookup table in other languages (Step 10)

HISTORY_WORDS = {"deforestation": "Deforestation", "degradation": "Degradation", "afforestation": "Afforestation",
                 "enhancement": "Enhancement", "stable forest": "Stable forest", "stable non forest": "Stable non-forest",
                 "non forest change": "Non-forest change", "forest other change": "Forest, other change",
                 # French (e.g. a country workbook): the same activities
                 "gain forestier": "Afforestation", "boisement": "Afforestation", "amelioration": "Enhancement",
                 "foret stable": "Stable forest", "non forestiere stable": "Stable non-forest",
                 "non forestieres en changement": "Non-forest change", "non forestiere en changement": "Non-forest change",
                 # Spanish, Portuguese
                 "deforestacion": "Deforestation", "degradacion": "Degradation", "forestacion": "Afforestation",
                 "mejora": "Enhancement", "bosque estable": "Stable forest", "desmatamento": "Deforestation",
                 "degradacao": "Degradation", "florestamento": "Afforestation"}


def history_key(text):
    """a history as Step 10 writes it, from a history written in another language or spelling: each event read with
    HISTORY_WORDS ('Gain forestier -> Deforestation' -> 'Afforestation -> Deforestation'); unknown words are kept."""
    parts = [p.strip() for p in str(text).split("->")]
    return " -> ".join(HISTORY_WORDS.get(plain(p), p) for p in parts)


def read_period_lookup(source):
    """the filled history lookup table: {history as Step 10 writes it: redd_activity}. The histories may be written in
    another language (French, Spanish, Portuguese): they are read with HISTORY_WORDS. redd_activity stays as written."""
    book = excel_of(source)
    sheet = "history_lookup" if "history_lookup" in book.sheet_names else book.sheet_names[0]
    t = book.parse(sheet, dtype=str)
    if not {"history", "redd_activity"} <= set(t.columns):
        raise ValueError("⛔ The file needs a sheet history_lookup with the columns history and redd_activity.")
    return {history_key(a): str(b).strip() for a, b in zip(t["history"], t["redd_activity"]) if given(a) and given(b)}



def history_lookup_read(source):
    """how each history of the file was read: the history as written and as Step 10 writes it (for transparency)."""
    book = excel_of(source)
    sheet = "history_lookup" if "history_lookup" in book.sheet_names else book.sheet_names[0]
    t = book.parse(sheet, dtype=str)
    rows = [(str(a).strip(), history_key(a)) for a in t["history"] if given(a)]
    return pd.DataFrame(rows, columns=["history in the file", "read as"])



# ----------------------------------------------------------------------------------------------- the country's words for the activities (Step 10)

HISTORY_ACTIVITIES = ["Deforestation", "Degradation", "Afforestation", "Enhancement", "Stable forest", "Stable non-forest",
                      "Non-forest change", "Forest, other change"]
HISTORY_ADDED = {}          # the country's words (sheet words): {plain word: activity}


def history_word(part):
    """one event of a history, as Step 10 writes it: a word of the country (sheet words), a built-in word, or None."""
    p = plain(part)
    if p in HISTORY_ADDED:
        return HISTORY_ADDED[p]
    if p in HISTORY_WORDS:
        return HISTORY_WORDS[p]
    return next((a for a in HISTORY_ACTIVITIES if plain(a) == p), None)


def history_key(text):
    """a history as Step 10 writes it, from a history written with the country's words or in another language
    ('Gain forestier -> Deforestation' -> 'Afforestation -> Deforestation'); a word not known is kept as written."""
    return " -> ".join(history_word(p) or p.strip() for p in str(text).split("->"))


def read_history_words(book):
    """the sheet words of the history lookup table (columns word, activity): the country's words, added to the built-in ones.
    Stops on an activity that is not one of HISTORY_ACTIVITIES, or on a word the built-in list reads otherwise."""
    HISTORY_ADDED.clear()
    if "words" not in book.sheet_names:
        return {}
    t = book.parse("words", dtype=str)
    if t.empty or not {"word", "activity"} <= set(t.columns):
        return {}
    bad = []
    for w, a in zip(t["word"], t["activity"]):
        if not (given(w) and given(a)):
            continue
        act = next((x for x in HISTORY_ACTIVITIES if plain(x) == plain(a)), None)
        known = HISTORY_WORDS.get(plain(w)) or next((x for x in HISTORY_ACTIVITIES if plain(x) == plain(w)), None)
        if act is None:
            bad.append(f"'{w}': '{a}' is not one of {HISTORY_ACTIVITIES}")
        elif known and known != act:
            bad.append(f"'{w}' is already read as {known}")
        else:
            HISTORY_ADDED[plain(w)] = act
    if bad:
        raise ValueError("⛔ Correct the sheet words of the history lookup table:\n   " + "\n   ".join(bad[:20]))
    return dict(HISTORY_ADDED)


def read_period_lookup(source):
    """the filled history lookup table: {history as Step 10 writes it: redd_activity}. The histories may use the country's
    words (sheet words) or another language (French, Spanish, Portuguese). redd_activity stays as written."""
    book = excel_of(source)
    read_history_words(book)
    sheet = "history_lookup" if "history_lookup" in book.sheet_names else book.sheet_names[0]
    t = book.parse(sheet, dtype=str)
    if not {"history", "redd_activity"} <= set(t.columns):
        raise ValueError("⛔ The file needs a sheet history_lookup with the columns history and redd_activity.")
    return {history_key(a): str(b).strip() for a, b in zip(t["history"], t["redd_activity"]) if given(a) and given(b)}


def history_lookup_read(source):
    """how each history of the file was read, and the words not known (for transparency)."""
    book = excel_of(source)
    sheet = "history_lookup" if "history_lookup" in book.sheet_names else book.sheet_names[0]
    t = book.parse(sheet, dtype=str)
    rows = []
    for a in t["history"]:
        if given(a):
            unknown = [p.strip() for p in str(a).split("->") if history_word(p) is None]
            rows.append((str(a).strip(), history_key(a), ", ".join(unknown)))
    return pd.DataFrame(rows, columns=["history in the file", "read as", "words not known"])


_write_period_template_v1 = write_period_template


def write_period_template(path, hist, first, last):
    """the history lookup table to fill, + the sheet words: the built-in words, and the country's own words to add."""
    _write_period_template_v1(path, hist, first, last)
    import openpyxl
    wb = openpyxl.load_workbook(path)
    ws = wb.create_sheet("words")
    ws.append(["word", "activity", "source"])
    for w, a in HISTORY_WORDS.items():
        ws.append([w, a, "built-in"])
    ws.column_dimensions["A"].width = 34; ws.column_dimensions["B"].width = 22; ws.column_dimensions["C"].width = 14
    r = wb["README"]
    r.append(["Sheet words", "Optional: the words the country uses for the activities in the histories (e.g. gain | Afforestation). "
              "The built-in words are listed; add a row per word of your own. activity = one of: " + ", ".join(HISTORY_ACTIVITIES) + "."])
    wb.save(path)



# ----------------------------------------------------------------------------------------------- one lexicon: land names and activity words (Steps 2 and 10)

write_period_template = _write_period_template_v1          # the history lookup table: no sheet of words (they are in the lexicon)
_read_lexicon_names = read_lexicon_file


def read_history_words(book, sheet="activities"):
    """the sheet activities of the lexicon (columns word, activity): the country's words for the activities of the histories,
    added to the built-in ones. Stops on an activity that is not one of HISTORY_ACTIVITIES, or on a word read otherwise."""
    HISTORY_ADDED.clear()
    if sheet not in book.sheet_names:
        return {}
    t = book.parse(sheet, dtype=str)
    if t.empty or not {"word", "activity"} <= set(t.columns):
        return {}
    bad = []
    for w, a in zip(t["word"], t["activity"]):
        if not (given(w) and given(a)):
            continue
        act = next((x for x in HISTORY_ACTIVITIES if plain(x) == plain(a)), None)
        known = HISTORY_WORDS.get(plain(w)) or next((x for x in HISTORY_ACTIVITIES if plain(x) == plain(w)), None)
        if act is None:
            bad.append(f"'{w}': '{a}' is not one of {HISTORY_ACTIVITIES}")
        elif known and known != act:
            bad.append(f"'{w}' is already read as {known}")
        elif not known:
            HISTORY_ADDED[plain(w)] = act
    if bad:
        raise ValueError("⛔ Correct the sheet activities of the lexicon:\n   " + "\n   ".join(bad[:20]))
    return dict(HISTORY_ADDED)


def read_lexicon_file(source):
    """the country's lexicon: sheet lexicon (land names: name, IPCC category) and sheet activities (words of the histories:
    word, activity). Returns (land names added, activity words added)."""
    names = _read_lexicon_names(source)
    return names, read_history_words(excel_of(source))


def read_period_lookup(source):
    """the filled history lookup table: {history as Step 10 writes it: redd_activity}. The words of the histories are read
    with the built-in words and the sheet activities of the lexicon (Step 2). redd_activity stays as written."""
    book = excel_of(source)
    sheet = "history_lookup" if "history_lookup" in book.sheet_names else book.sheet_names[0]
    t = book.parse(sheet, dtype=str)
    if not {"history", "redd_activity"} <= set(t.columns):
        raise ValueError("⛔ The file needs a sheet history_lookup with the columns history and redd_activity.")
    return {history_key(a): str(b).strip() for a, b in zip(t["history"], t["redd_activity"]) if given(a) and given(b)}


def activity_words_table():
    """the words read as an activity of the histories: the activities themselves, the built-in words, the country's words."""
    rows = [{"word": a, "activity": a, "source": "built-in"} for a in HISTORY_ACTIVITIES]
    rows += [{"word": w, "activity": a, "source": "built-in"} for w, a in HISTORY_WORDS.items() if plain(w) not in {plain(x) for x in HISTORY_ACTIVITIES}]
    rows += [{"word": w, "activity": a, "source": "country file"} for w, a in HISTORY_ADDED.items()]
    return pd.DataFrame(rows)


def write_lexicon(path, found):
    """the lexicon of the survey: the land answers found and how they were read (found), the land names (lexicon) and the
    words of the histories (activities), for the country to add its own words."""
    guide = pd.DataFrame([
        ("What it is", "The words the notebook reads. Sheet found: how the answers on the IPCC category were read. Sheet lexicon: the "
                       "land names read as an IPCC category. Sheet activities: the words read as an activity in the histories of Step 10."),
        ("Sheet lexicon", "Add a row per land name the survey uses: name, and one of the six IPCC categories (Forest land, Cropland, "
                          "Grassland, Wetlands, Settlements, Other land), e.g. monte | Forest land. Only names of the six categories: a "
                          "national class counted in another category is a choice of Step 6 (reclassify)."),
        ("Sheet activities", "Add a row per word the country uses for an activity in its history lookup table: word, and one of "
                             + ", ".join(HISTORY_ACTIVITIES) + ", e.g. gain | Afforestation."),
        ("Then", "Write this file in lexicon_file (Step 2, 'use my file') and run Step 2 again; Step 10 rereads it.")], columns=["", "how"])
    with pd.ExcelWriter(path, engine="xlsxwriter") as w:
        guide.to_excel(w, sheet_name="README", index=False)
        found.to_excel(w, sheet_name="found", index=False)
        lexicon_table().to_excel(w, sheet_name="lexicon", index=False)
        activity_words_table().to_excel(w, sheet_name="activities", index=False)
        w.sheets["README"].set_column(0, 0, 16); w.sheets["README"].set_column(1, 1, 130)
        for sh in ("found", "lexicon", "activities"):
            w.sheets[sh].set_column(0, 4, 26)



# ----------------------------------------------------------------------------------------------- Step 10: more built-in activity words; the result in one file

HISTORY_WORDS.update({
    "gain": "Afforestation", "forest gain": "Afforestation", "reforestation": "Afforestation", "reboisement": "Afforestation",
    "boisement": "Afforestation", "gain de foret": "Afforestation", "reforestacion": "Afforestation", "reflorestamento": "Afforestation",
    "loss": "Deforestation", "forest loss": "Deforestation", "perte": "Deforestation", "perte forestiere": "Deforestation",
    "perte de foret": "Deforestation", "perdida de bosque": "Deforestation",
    "regrowth": "Enhancement", "regeneration": "Enhancement", "recuperation": "Enhancement",
    "degradation forestiere": "Degradation", "degradacion forestal": "Degradation",
    "non foret stable": "Stable non-forest", "non forest stable": "Stable non-forest", "foret stable": "Stable forest",
    "no forest change": "Non-forest change"})


def write_period_result(path, est, hist, first, last):
    """the activity of the period in one Excel: areas (one row per activity: plots, area, SE, 90% CI), plots (one activity per
    plot, with its history) and histories (each history, its activity, its plots)."""
    hs = hist.groupby(["history", "redd_activity"]).size().rename("plots").reset_index().sort_values("plots", ascending=False)
    guide = pd.DataFrame([("What it is", f"The activity of the period {first}-{last}: each plot gets one REDD+ activity from its history (the forest "
                                         "events of the period, reported classes) through the country's history lookup table; then the area of "
                                         "each activity with its standard error and 90% CI (stratified estimator; one stratum when systematic)."),
                          ("Sheet areas", "One row per activity of the period: plots, area (ha), SE (ha), 90% CI (ha) = 1.645 x SE, 90% CI / area."),
                          ("Sheet plots", "One row per plot: its history, its activity, the land at the start and at the end, the years of its events."),
                          ("Sheet histories", "Each history of the data, the activity given by the history lookup table, and its plots."),
                          ("Years", f"The changes of {first} to {last} (pairs of years ending in those years). The yearly values are in Step 7.")],
                         columns=["", "note"])
    with pd.ExcelWriter(path, engine="xlsxwriter") as w:
        guide.to_excel(w, sheet_name="README", index=False)
        est.to_excel(w, sheet_name="areas", index=False)
        hist.to_excel(w, sheet_name="plots", index=False)
        hs.to_excel(w, sheet_name="histories", index=False)
        w.sheets["README"].set_column(0, 0, 16); w.sheets["README"].set_column(1, 1, 130)
        w.sheets["areas"].set_column(0, 1, 26); w.sheets["areas"].set_column(2, 7, 14, w.book.add_format({"num_format": "#,##0.0"}))
        w.sheets["plots"].set_column(0, 5, 30); w.sheets["histories"].set_column(0, 2, 40)


## Step 1 - Read
**What you do:** write the file in `file_path`, or leave it empty in Colab and choose the file(s).

- **One file**, e.g. a Collect Earth export already combined, or a Collect Earth Online export.
- **Several files of the same survey**, e.g. one Collect Earth CSV per operator: write them separated by `;`, or write the **folder** that holds them. They are put together; the column `source_file` says the file of each row. A plot in two files stops the notebook in Step 3.
- Files of **different surveys** are read one at a time: their unified formats (Step 4) have the same columns and can be put together afterwards.


In [ ]:
#@title Step 1 - Read the export(s)
#@markdown **One file, several files separated by ; or a folder. In Colab you can leave it empty and choose the file(s).**
file_path = ""  #@param {type:"string"}
#@markdown <font color="grey">Optional: the folder where the results are saved. Empty = the folder where the notebook runs.</font>
output_folder = ""  #@param {type:"string"}

out_dir = os.path.abspath(output_folder.strip() or ".")
os.makedirs(out_dir, exist_ok=True)

parts = []
if file_path.strip():
    for p, in_folder in export_paths(file_path):
        try:
            t = read_export(p)
            if in_folder and not looks_like_export(t):
                raise ValueError("not a survey export")
        except ValueError as e:
            if not in_folder:
                raise
            print(f"   left out {os.path.basename(p)}: {str(e)[:80]}")
            continue
        parts.append((os.path.basename(p), t))
        print(f"   read {os.path.basename(p)}{f' (sheet {read_export.sheet})' if read_export.sheet else ''}: {t.shape[0]:,} rows")
    if not parts:
        raise ValueError("⛔ No survey export found in file_path.")
else:
    try:
        from google.colab import files  # only in Colab
    except ImportError:
        raise ValueError("⛔ Write the export file(s) in file_path.")
    up = files.upload()
    for name in up:
        parts.append((name, read_export(up[name], name)))
export, combine_notes = combine_exports(parts)
export_name = parts[0][0] if len(parts) == 1 else f"{len(parts)} files ({parts[0][0]} ...)"
sheet_read = read_export.sheet if len(parts) == 1 else ""
for n in combine_notes:
    print("⚠ " + n)
print(f"✅ {export_name}: {export.shape[0]:,} rows, {export.shape[1]} columns")
print(f"   The results will be saved in: {out_dir}")


## Step 2 - Describe
**What you do:** write `end_year` for Collect Earth. `start_year` is optional. Leave `direction` on auto.

**Look at:** the country, the end year, the picture and the questions (type, plots answered, answers).
**The lexicon (the words the notebook reads):** the answers on the IPCC category are read with a list of names in several languages (e.g. *Forest land, forêt, bosque, tierras forestales, floresta, F, FL*), and the histories of Step 10 with a list of activity words (e.g. *Afforestation, gain forestier, forestación*). Step 2 shows how each land answer was read, in the page and in `_step2_lexicon_` (Excel), which also lists all the words known. The country adds its own words there: sheet *lexicon* (columns `name`, `IPCC category`, e.g. *monte | Forest land*) and sheet *activities* (columns `word`, `activity`, e.g. *gain | Afforestation*). Then choose *use my file*, give it in `lexicon_file` and run Step 2 again; Step 10 rereads it. Only names of the six IPCC categories in the sheet *lexicon*: a national class counted in another category is a choice of Step 6 (reclassify), made only when it is needed.

`survey_name` (optional): the name used in the files; empty = the country found from the location of the plots.


In [ ]:
#@title Step 2 - Describe the survey
#@markdown **Required for Collect Earth (the export does not say it). Collect Earth Online: 0 = found by itself.**
end_year = 0  #@param {type:"integer"}
#@markdown <font color="grey">Optional: 0 = the first year with information. To see the changes of a year as a pair, start one year before (e.g. 2000 for the changes of 2001).</font>
start_year = 0  #@param {type:"integer"}
#@markdown <font color="grey">Leave it on auto: it is found from the survey.</font>
direction = "auto"  #@param ["auto", "backward", "onward"]
#@markdown <font color="grey">Optional - the country's lexicon: the words of its survey and of its tables. Sheet lexicon: land names (name, IPCC category; e.g. monte | Forest land). Sheet activities: the words of the history lookup table of Step 10 (word, activity; e.g. gain | Afforestation). Start from the file _step2_lexicon_ of a run. Built-in words only = the words the notebook knows (Step 0).</font>
lexicon_choice = "built-in words only"  #@param ["built-in words only", "use my file"]
#@markdown <font color="grey">Only on your own computer: the path of the lexicon. In Colab leave it empty and choose the file.</font>
lexicon_file = ""  #@param {type:"string"}
#@markdown <font color="grey">Optional - the name used in the files. Empty = the country found from the location of the plots.</font>
survey_name = ""  #@param {type:"string"}
lexicon_source = None
if lexicon_choice == "use my file":
    if lexicon_file.strip():
        lexicon_source, lexicon_name = lexicon_file.strip(), os.path.basename(lexicon_file.strip())
    else:
        from google.colab import files
        up = files.upload()
        lexicon_name = list(up)[0]
        lexicon_source = up[lexicon_name]
    added_names, added_words = read_lexicon_file(lexicon_source)
    print(f"Lexicon {lexicon_name}: {len(added_names)} land name(s) added {added_names or ''}; {len(added_words)} activity word(s) added {added_words or ''}")

survey = describe_survey(export, "", end_year=end_year)
survey["file"], survey["sheet"] = export_name, sheet_read
country = survey["survey_id"] = re.sub(r"[^A-Za-z0-9_-]+", "_", survey_name.strip()) if survey_name.strip() else country_from_location(survey)
stamp = datetime.datetime.now().strftime("%Y-%m-%d_%H%M%S")
print(f"Survey: {country}" + ("" if survey_name.strip() else " (the country, from the location of the plots)"))
land_found = land_names_in(survey)
print("How the answers on the IPCC category were read:")
print(land_found.head(30).to_string(index=False))
if (land_found["read as"] == "not recognised").any():
    print("⚠ Answers not recognised: add them to the sheet lexicon of _step2_lexicon_ and give the file in lexicon_file.")
print(describe_text(survey))
for line in land_names_report(survey):
    print("ℹ " + line)
survey_says = survey_direction(survey)
if direction == "auto" and not survey_says:
    raise ValueError("⛔ The direction could not be found: select backward or onward in direction, then run this step again.")
direction_used = survey_says if direction == "auto" else direction
print(f"Direction: {direction_used} ({survey['design_text']})")
if survey_says and direction_used != survey_says:
    print(f"⚠ The survey reads {survey_says}: the years outside the rows collected are filled less.")
table, picture = None, None
if survey.get("end_year"):
    try:
        table = plot_year_table(survey, previous_year, start_year, direction_used)
        picture = os.path.join(out_dir, f"{country}_step2_survey_structure_{stamp}.html")
        write_picture(picture, survey, table, direction_used)
        print(f"✅ The picture of the survey:\n   {picture}")
        FILES = [picture]
        add_to_page(picture, "How the land names were read", land_found, "Each answer on the IPCC category, how many times it is given and the category it is read as (built-in list or the country's lexicon).")
        try:
            from google.colab import files
            for f in FILES:
                files.download(f)
            print("⬇ The files are being downloaded to your Downloads folder.")
        except ImportError:
            pass
    except Exception as e:
        table, picture = None, None
        print(f"⚠ The picture could not be made yet ({e}): see the quality check in Step 3.")
lexicon_xlsx = os.path.join(out_dir, f"{country}_step2_lexicon_{stamp}.xlsx")
write_lexicon(lexicon_xlsx, land_found)
print(f"✅ The lexicon: the land answers found, the land names (sheet lexicon) and the activity words (sheet activities) - add the country's words there:\n   {lexicon_xlsx}")
print("✅ Survey described.")

## Step 2b - One sample (optional)
**What you do:** write one sample id (several: separate them with commas) and run. Empty = nothing is made. Run it again with another id: it uses the table of Step 2.

**Look at:** the answers of the sample, the years built from them, and its messages.

In [ ]:
#@title Step 2b - One sample (optional)
#@markdown The sample id, e.g. 1024 (several: separate with commas). Empty = nothing is made.
sample_id = ""  #@param {type:"string"}

import difflib
wanted = [x.strip() for x in str(sample_id).split(",") if x.strip()]
if wanted and table is None:
    raise ValueError("⛔ Run Step 2 first (its picture must be made).")
FILES = []
for one in wanted:
    page = os.path.join(out_dir, f"{country}_step2_sample_{re.sub(r'[^A-Za-z0-9_-]', '_', one)}_{stamp}.html")
    if write_sample(page, survey, table, one):
        FILES.append(page)
        print(f"✅ Sample {one}:\n   {page}")
    else:
        ids = survey["data"][survey["id_col"]].astype(str).str.strip().tolist()
        near = difflib.get_close_matches(one, ids, n=5, cutoff=0.6)
        print(f"⚠ Sample {one} is not in the file ({survey['id_col']})." + (f" Close: {', '.join(near)}" if near else ""))
try:
    from google.colab import files
    for f in FILES:
        files.download(f)
except ImportError:
    pass


## Step 3 - Quality check
**What you do:** with any ⛔ the notebook stops: correct the export and run again from Step 1. Or tick `skip_correction` to go on: the values stay as in the export and the note of their row says *to correct: ...*.

- **The same plot id twice** (in one file, or in two files) always stops: `skip_correction` does not apply.
- **Two plot ids at the same place**: ⛔ to fix.
- **Gaps**: years without land between two years with land (⚠ to check: nothing is filled).
- The other checks: the end year, the type of the values, the land and the changes recorded (e.g. a change recorded but the land stays the same).


In [ ]:
#@title Step 3 - Quality check
#@markdown <font color="grey">Optional: go on even with errors to fix (⛔). Nothing is changed or removed: the values stay as in the export and the note of their row says "to correct: ...".</font>
skip_correction = False  #@param {type:"boolean"}
qc_report, _, qc_stop = quality_check(survey, advanced=True, set_aside=False)
dups = duplicate_ids(survey["data"], survey["id_col"], survey.get("point_col"))
if dups:
    print("⛔ The same plot id more than once:\n   " + "\n   ".join(dups[:20]))
    raise ValueError(f"⛔ {len(dups)} plot id(s) more than once: keep one row per plot (or per plot and sample point), then run again from Step 1.")
same_place = qc_report["check"].eq("4 same location")
qc_report.loc[same_place, "level"] = "⛔ to fix"
if qc_stop:
    raise ValueError("⛔ " + qc_stop + " - then run Step 2 and this step again.")
if qc_report["level"].str.startswith("⛔").any() and not skip_correction:
    table = None
else:
    if table is None:
        try:
            table = plot_year_table(survey, previous_year, start_year, direction_used)
        except Exception as e:
            raise ValueError(f"⛔ The table cannot be built with these errors ({e}): correct the export, then run again from Step 1.")
    qc_report = pd.concat([qc_report, land_problems(table), year_gaps(table)], ignore_index=True)
print(quality_text(qc_report, set(), True))
to_fix = qc_report[qc_report["level"].str.startswith("⛔")]
FILES = []
if len(qc_report):
    errors_csv, correct_csv = write_error_files(survey, qc_report, country, stamp, out_dir)
    FILES = [f for f in (errors_csv, correct_csv) if f]
    print(f"\nThe errors with a description ({len(qc_report):,} lines):\n   {errors_csv}")
    if correct_csv:
        print(f"The rows to correct ({to_fix['plot'].nunique():,} plots):\n   {correct_csv}")
        print(to_fix[["plot", "detail"]].head(10).to_string(index=False))
try:
    from google.colab import files
    for f in FILES:
        files.download(f)
    print("⬇ The files are being downloaded to your Downloads folder.")
except ImportError:
    pass

if len(to_fix) and not skip_correction:
    raise ValueError("⛔ Correct the export, then run again from Step 1 (or tick skip_correction to go on). The next steps do not run until there is nothing to correct.")
if len(to_fix):
    table = flag_to_correct(table, to_fix)
    print(f"⚠ {len(to_fix)} errors left uncorrected ({to_fix['plot'].nunique()} plots): the notebook goes on. Their values stay as in the export; the note of their row says 'to correct: ...'.")
else:
    print("✅ Nothing to correct.")

## Step 4 - Unified format
**What you do:** run it.

**You get:** one row per plot and year, always the same 41 columns.
- `source`: **raw** = the survey recorded the sample in that year (a date or the year of a change): every value of the row is the survey's answer for that year, empty when not answered. **filled** = the year is carried from the year recorded before it: the same land and the same attributes of the land.
- The attributes of the land (canopy cover, tree cover, ...) are filled like the land. The attributes of a change (its type, the drivers, `temporary_loss`, `gain_origin` = planted / natural) are only in the year of the change, never filled.
- A canopy that differs between two recorded years with the same land and no change is **not a change**: the row keeps its value and the note says, for example, *canopy 100 → 92 between 2000 and 2008 (year not known), no change recorded*.
- **Canopy before and after**: in the year where the canopy recorded differs from the year before, `tree_cover_before` and `tree_cover_after` (as the survey wrote them) and `tree_cover_change`: *stable*, *decrease* or *increase*.
  - **The survey's classes are found by themselves** (e.g. *> 30-50 %*, *> 50 %*): a change of class is a decrease or an increase. A number (e.g. a tree cover of 20 on other land) is placed among the classes.
  - A survey that writes **numbers** (e.g. 80, 72): any difference is a decrease or an increase; its size stays visible in before and after (80 → 72). Words whose order the notebook cannot know (e.g. *dense*, *open*): before and after only.
  - The values the survey gave itself are kept. The year is the first year of the new value (the change happened between the two years recorded; see the note). Nothing to fill here: what a canopy change means (degradation, enhancement...) is the country's choice, in the sheet `canopy` of the **country rules workbook** of Step 6.

**Look at:** the plot shown.


In [ ]:
#@title Step 4 - Unified format
if table is None or (len(to_fix) and not skip_correction):
    raise ValueError("⛔ Step 3 found things to fix: fix the export, then run again from Step 1 (or tick skip_correction in Step 3).")
if len(to_fix):
    print(f"⚠ {len(to_fix)} errors left uncorrected ({to_fix['plot'].nunique()} plots): see 'to correct' in the note column.")
table, canopy_counts, canopy_found = canopy_changes(table)
if canopy_found:
    print("Canopy classes written by the survey: " + ", ".join(f"{lo:g}-{hi:g}" for lo, hi in canopy_found))
if canopy_counts:
    print(f"Canopy before -> after ({'by class' if canopy_found else 'as written'}): {canopy_counts} - what a change means: the sheet canopy of the country rules workbook (Step 6)")
elif canopy_found:
    print("Canopy before -> after: no canopy that differs from one year to the next.")
year_csv = os.path.join(out_dir, f"{country}_step4_unified_format_{stamp}.csv")
table.to_csv(year_csv, index=False, encoding="utf-8-sig")
if survey.get("previous_year"):
    print(f"Earlier assessment: {survey['previous_year']} (found from the data: please check it)")
print(f"✅ {table['plot_id'].nunique():,} plots, {len(table):,} rows, years {table['year'].min()}-{table['year'].max()}")
print(f"   source: {table['source'].value_counts().to_dict()}")
print(f"   canopy that differs between two recorded years, no change: {table['note'].astype(str).str.contains('no change recorded').sum():,} rows")
print(f"   change = yes: {(table['change'] == 'yes').sum():,}, disturbance = yes: {(table['disturbance'] == 'yes').sum():,}")
print(f"✅ The unified format ({len(table.columns)} columns, the same in every file):\n   {year_csv}")
changed = table.loc[(table["change"] == "yes") | (table["disturbance"] == "yes"), "plot_id"]
pid = changed.iloc[0] if len(changed) else table["plot_id"].iloc[0]
print(f"\nPlot {pid}:")
print(table[table["plot_id"].astype(str) == str(pid)].drop(columns=["survey_id"]).to_string(index=False))
try:
    from google.colab import files
    for f in [year_csv]:
        files.download(f)
    print("⬇ The files are being downloaded to your Downloads folder.")
except ImportError:
    pass

## Step 5 - History of the samples
**What you do:** run it.

**You get:** from the unified format only, before any rule of the country (the history *after* the country's rules, `_step5_history_reported_`, is written in Step 6, where the rules are read):
- the **pairs**: one row per plot and pair of years following each other (`from_` = the first year, `to_` = the second year);
- the **history**: one row per plot, the IPCC category, the subdivision and the canopy of every year side by side: `lu_ipcc_1990, lu_sub_1990, cc_1990, lu_ipcc_1991, ...` (`cc` = canopy cover on forest land, tree cover on other land), and `temporary_loss_<year>` (yes = cyclical, no = true loss) and `gain_origin_<year>` (planted / natural), only for the years where a plot has one. Natural or planted forest: the subdivision says it (e.g. Private plantations).

**How to read the years:** the land of a year is the land at the end of that year (31 December). A pair 2012–2013 is the land at the end of 2012 and at the end of 2013: a change in the pair happened at any time in 2013, from 1 January to 31 December (29 December 2013 is still a change of 2013), and the 2013 row already shows the land after it. The type and the drivers of the change are in the second year of the pair (2013).


In [ ]:
#@title Step 5 - History of the samples (pairs and years side by side)
if globals().get("table") is None or (len(to_fix) and not skip_correction):
    raise ValueError("⛔ Run Step 4 first.")
pairs = pairs_table(table)
pairs_csv = os.path.join(out_dir, f"{country}_step5_pairs_{stamp}.csv")
pairs.to_csv(pairs_csv, index=False, encoding="utf-8-sig")
history = history_wide(table)
history_csv = os.path.join(out_dir, f"{country}_step5_history_{stamp}.csv")
history.to_csv(history_csv, index=False, encoding="utf-8-sig")
print(YEARS_EXPLAINED)
print(f"\n✅ {len(pairs):,} pairs of years (change = yes: {(pairs['change'] == 'yes').sum():,}, disturbance = yes: {(pairs['disturbance'] == 'yes').sum():,}):\n   {pairs_csv}")
print(f"✅ The history: {len(history):,} plots, one row each, lu_ipcc_, lu_sub_ and cc_ for {table['year'].nunique()} years:\n   {history_csv}")
changed = table.loc[table["change"] == "yes", "plot_id"]
for pid in list(dict.fromkeys(changed))[:3]:
    print(f"   {pid}: {history_line(table, pid)}")
try:
    from google.colab import files
    for f in [pairs_csv, history_csv]:
        files.download(f)
    print("⬇ The files are being downloaded to your Downloads folder.")
except ImportError:
    pass


## Step 6 - The country's rules and the REDD+ classes
**What you do:** run it with *make the template*: it writes the **country rules workbook** with what is found in the data. Fill the sheets you need, choose *use my filled file*, give the file and run it again. Or choose *no rules*.

**Recorded and reported classes.** *Recorded* = the classes as the interpreters recorded them, no rule applied: the record, kept for checking. *Reported* = the classes after the country's rules: what the country reports, used by Steps 7, 8 and 10. The unified format (Steps 4-5) always keeps the recorded classes. The table *recorded / reported / difference* shows what the rules change.

**The country rules workbook** - one file, one sheet per kind of rule, every sheet optional (an empty sheet is not used):

| Sheet | What | Columns read | Example row |
|---|---|---|---|
| `lookup` | the transitions between forest types | `from_sub`, `to_sub`, `classification` | dense_forest \| open_forest \| Degradation |
| `canopy` | the canopy cover transitions in stable forest, only when the survey records the canopy | `from_cc`, `to_cc`, `classification`, optional `forest_type` | > 50 % \| > 30-50 % \| Degradation |
| `category changes` | reclassify: a class counted in another IPCC category | `sub`, `new category` | wooded_savanna \| Forest land |
| `subclass merges` | merge: classes counted as one | `sub`, `merged into` | gallery_forest \| gallery_riparian_forest |

- **classification** (sheets `lookup` and `canopy`): *Degradation*, *Enhancement*, *Stable forest* (no REDD+ activity) or *Not possible* (not applied: the types recorded are kept, the pair is listed for review). The French words (Dégradation, Amélioration, Forêt stable, Pas possible) are read too. A change of the data with no rule becomes *To review*.
- **canopy**: in stable forest (forest in both years, the same forest type, the class not already decided by the land, the lookup, a disturbance or a regrowth), a change of canopy from one value to another, whatever the survey writes: classes (*> 30-50 %*), numbers or words. A range (e.g. `70-100`) can stand for numbers, and a change inside one range is no transition; a rule for one forest type wins over a rule for all. The class says `based_on = canopy ...`, and the history of Step 10 includes it.
- **merge**: a change between merged classes is no change; the merged class needs its own rows in the sheet `lookup`.
- Other columns are notes for the reader. A workbook made by the country in the same layout works (e.g. the transition table of the Extract Tool, with its column `classification`).

**Look at:** the table recorded / reported / difference; the changes with no rule (*To review*); the *Not possible* changes; the canopy transitions applied.


In [ ]:
#@title Step 6 - The country's rules (lookup), reclassify / merge, and the REDD+ classes
lookup_choice = "make the template"  #@param ["make the template", "use my filled file", "no rules"]
#@markdown <font color="grey">The country rules workbook, one sheet per kind of rule, every sheet optional: lookup (transitions between forest types), canopy (canopy cover transitions, only when the survey records the canopy), category changes (reclassify), subclass merges (merge). 'make the template' writes it with what is found in the data. Only on your own computer: the path of the filled file. In Colab leave it empty and choose the file.</font>
lookup_file = ""  #@param {type:"string"}

if "pairs" not in globals():
    raise ValueError("⛔ Run Step 5 first.")
if not len(pairs):
    raise ValueError("⛔ There is no pair of years: the survey gives the land at one date only (or its change years are codes, "
                     "not years). Steps 6-8 need the land in at least two years.")
classes_recorded = land_classes(table)
names = class_names(survey)
source = None
if lookup_choice == "use my filled file":
    if lookup_file.strip():
        source = lookup_file.strip()
    else:
        from google.colab import files
        up = files.upload()
        source = up[list(up)[0]]                         # the bytes of the file: read for each sheet

# the country's reading of the classes: the sheets category changes (reclassify) and subclass merges (merge) of the workbook
changes, merges = read_adjustments(source) if source is not None else ({}, {})
from_file = True
absent = [c for c in list(changes) + list(merges) if c not in set(classes_recorded["subdivision"])]
if absent:
    print(f"⚠ reclassify / merge: {', '.join(absent)} not found in the data - no effect.")
if changes or merges:
    print("The country's reading of the classes" + (" (from the lookup file)" if from_file else " (from the fields)") + ":")
    for sub, cat in changes.items():
        print(f"   reclassify: {sub} ({names.get(sub, sub)}) -> {cat}")
    for new in dict.fromkeys(merges.values()):
        print(f"   merge: {' + '.join(k for k, v in merges.items() if v == new)} -> {new}")

rules, rules_ready = {}, False
classes_cat = with_category_changes(classes_recorded, changes)
lp_base, years_text = complete_years(land_pairs(pairs, classes_cat))
print(years_text)
if lookup_choice == "make the template":
    template = os.path.join(out_dir, f"{country}_step6_country_rules_template_{stamp}.xlsx")
    canopy_found_pairs = canopy_transitions_found(with_merges(lp_base, merges))
    write_rules_workbook(template, merged_classes(classes_cat, merges), with_merges(lp_base, merges), canopy_found_pairs)
    FILES = [template]
    print(f"✅ The template:\n   {template}")
    print(f"   {len(forest_changes(with_merges(lp_base, merges)))} changes between forest types found in the data.")
    if len(canopy_found_pairs):
        print(f"   {len(canopy_found_pairs)} canopy transitions in stable forest ({int(canopy_found_pairs['plots'].sum())} plot-pairs): sheet canopy.")
    else:
        print("   No canopy transition in stable forest: the sheet canopy can stay empty.")
    unknown = classes_cat[classes_cat["IPCC category"].map(lambda v: not given(v))]
    if len(unknown):
        print(f"⚠ {len(unknown)} land names are not recognised: write their IPCC category in the sheet classes: {list(unknown['subdivision'])[:12]}")
    try:
        from google.colab import files
        for f in FILES:
            files.download(f)
        print("⬇ The files are being downloaded to your Downloads folder.")
    except ImportError:
        pass
    print("⏸ Fill the yellow cells, choose 'use my filled file' and run this step again. Do not forget to give the filled file: its path in the file field, or in Colab leave it empty and upload it.")
else:
    if source is not None:
        rules, classes_recorded = read_lookup(source, classes_recorded)
        classes_cat = with_category_changes(classes_recorded, changes)
        print(f"✅ {len(rules)} rules read: {pd.Series(list(rules.values())).value_counts().to_dict()}")
    empty = classes_cat[classes_cat["IPCC category"].map(lambda v: not given(v))]
    if len(empty):
        print(empty[["land_use", "subdivision"]].to_string(index=False))
        raise ValueError("⛔ These land names are not recognised as an IPCC category: make the template, write their category in its sheet classes, and use that file.")
    # as recorded: the interpreters' classes, no rule, no reclassify or merge
    recorded = classify_pairs(apply_rules(complete_years(land_pairs(pairs, classes_recorded))[0], {}), False)
    # reported: reclassify, merge, then the rules of the lookup
    lp = apply_rules(with_merges(complete_years(land_pairs(pairs, classes_cat))[0], merges), rules)
    classes = merged_classes(classes_cat, merges)
    missing_merged = sorted(set(merges.values()) - {k for pair in rules for k in pair})
    if rules and missing_merged:
        print(f"⚠ The lookup has no rule for the merged class(es) {', '.join(missing_merged)}.")
    missing = forest_changes(lp[lp["rule"] == "no rule"])
    if rules and len(missing):
        print(f"⚠ {len(missing)} changes between forest types have no rule ({int(missing.sum())} plot-pairs): class 'To review'.")
        print(missing.sort_values(ascending=False).head(10).to_string())
    impossible = lp[(lp["rule"] == "Not possible") & (lp["from_m"] != lp["to_m"])]       # the pair where the change is recorded
    if len(impossible):
        print(f"⚠ {len(impossible)} change(s) 'Not possible' in the lookup: the types recorded are kept (the most recent class is taken as right), the pair counts as stable forest (to review on the images):")
        print(impossible[["plot_id", "to_year", "from_m", "to_m"]].rename(columns={"to_year": "year", "from_m": "from", "to_m": "to"}).head(20).to_string(index=False))
    rules_ready = True
    classified = classify_pairs(lp, bool(rules))
    canopy_done = {}
    canopy_rules = read_canopy_lookup(source) if source is not None else []
    if canopy_rules:                                             # the sheet canopy of the workbook
        classified, canopy_done, canopy_review, canopy_impossible = apply_canopy_lookup(classified, lp, canopy_rules)
        print(f"✅ Sheet canopy: {len(canopy_rules)} rules; canopy transitions in stable forest: {canopy_done or 'none'}")
        if len(canopy_review):
            print(f"⚠ {len(canopy_review)} canopy transition(s) with no rule: class 'To review':")
            print(canopy_review.groupby(["canopy before", "canopy after"]).size().rename("plot-pairs").to_string())
        if len(canopy_impossible):
            print(f"⚠ {len(canopy_impossible)} canopy transition(s) 'Not possible': kept as stable forest, to review on the images:")
            print(canopy_impossible.head(20).to_string(index=False))
    adjusted = bool(rules or changes or merges or canopy_done)   # reported differs from recorded
    classes_csv = os.path.join(out_dir, f"{country}_step6_classes_{'reported_' if adjusted else ''}{stamp}.csv")
    classes_wide(classified).to_csv(classes_csv, index=False, encoding="utf-8-sig")
    FILES = [classes_csv]
    if adjusted:
        recorded_csv = os.path.join(out_dir, f"{country}_step6_classes_recorded_{stamp}.csv")
        classes_wide(recorded).to_csv(recorded_csv, index=False, encoding="utf-8-sig")
        FILES.append(recorded_csv)
        history_reported_csv = os.path.join(out_dir, f"{country}_step5_history_reported_{stamp}.csv")
        history_reported(history, lp).to_csv(history_reported_csv, index=False, encoding="utf-8-sig")
        FILES.append(history_reported_csv)
        print("\nLevel 1, all pairs of years - recorded / reported / difference:")
        print(compare_levels(recorded, classified).to_string())
    print()
    print(class_table(classified).to_string())
    print(f"\n✅ One row per plot; per pair of years: from > to, the class at 3 levels, the driver and why{' (reported: used by Steps 7-8)' if adjusted else ' (recorded = reported: no rule, reclassify or merge)'}:\n   {classes_csv}")
    if adjusted:
        print(f"✅ The same, as recorded by the interpreters (no rule, no reclassify or merge):\n   {recorded_csv}")
        print(f"✅ The history of Step 5 after the country's rules (same columns as _step5_history_):\n   {history_reported_csv}")
    try:
        from google.colab import files
        for f in FILES:
            files.download(f)
        print("⬇ The files are being downloaded to your Downloads folder.")
    except ImportError:
        pass
    print("✅ Ready for Step 7.")


## Step 7 - Area, standard errors and land-use conversion matrices
**What you do:** write `total_area_ha` and choose the sampling design. Step 8 needs hectares.

**Sampling design**
- *systematic* - every plot has the same weight: `total_area_ha` / the number of plots.
- *stratified* - each stratum has its own weight. The notebook looks in the export for a column of hectares per plot (one value per stratum) and the column of the strata (e.g. `exp_fct (ha/plot)` and `Prob_Zone`) and lists the strata it uses (1, 2, 3 ... named as in the data). Only if it does not find them: write `stratum_column` and `stratum_areas` (`stratum: hectares; ...`).

**Standard error:** each area of a class or a transition has its standard error (stratified estimator: per stratum, the share of its plots in the class; one stratum when systematic).

**The transitions for mocaredd** (`_step7_mocaredd_transitions_`): one row per change of class and year with area, standard error and REDD+ activity (DF deforestation, DG degradation, EN afforestation and enhancement), in the columns of the activity data of mocaredd. **Natural or planted** (`planted_classes`, only for this file): each deforestation is labelled *planted* or *natural* from the forest class before the change, each afforestation from the forest class after it (column `redd_activity_details`, e.g. *Deforestation (planted)*, *Deforestation (natural)*), so that planted and natural forest can be given different carbon stocks in mocaredd. It changes no area, class or matrix. Write the planted forest classes of the survey (e.g. `private_plantation; public_plantation`); empty = the forest classes whose name has "plantation" (English only: a class named *plantacion* or *reboisement* is not found, write it). A degradation without a change of class (a disturbance recorded) is not in this file: it is counted in the class areas. Many rows rest on one plot: then the standard error equals the area (see the note printed).


In [ ]:
#@title Step 7 - Area, standard errors and land-use conversion matrices
#@markdown <font color="grey">The total area in hectares. 0 = plots only (stratified: the areas of the strata are used).</font>
total_area_ha = 0  #@param {type:"number"}
sampling_design = "systematic"  #@param ["systematic", "stratified"]
#@markdown <font color="grey">Stratified, only if the strata are not found in the export. Example: stratum ; and 1: 1500000; 2: 800000; ...</font>
stratum_column = ""  #@param {type:"string"}
stratum_areas = ""  #@param {type:"string"}
#@markdown <font color="grey">Optional - only for the mocaredd file: the forest classes that are planted (separate with ;). Each deforestation is labelled planted or natural from the forest class before the change, each afforestation from the class after it (column redd_activity_details, e.g. Deforestation (planted)), so that planted and natural forest can get different carbon stocks. No area, class or matrix changes. Empty = the forest classes whose name has "plantation" (English only: write the classes when they are named otherwise, e.g. plantacion, reboisement). Example: private_plantation; public_plantation</font>
planted_classes = ""  #@param {type:"string"}

if "classified" not in globals() or not globals().get("rules_ready"):
    raise ValueError("⛔ Finish Step 6 first.")
all_plots = lp["plot_id"].nunique()
plot_w, strata, strata_text = plot_strata(survey, lp["plot_id"], sampling_design, total_area_ha, stratum_column, stratum_areas)
print(strata_text)
if sampling_design == "stratified":
    total_area_ha = float(strata["area_ha"].sum())
per_plot = float(total_area_ha) / all_plots if total_area_ha else 0
lp = lp.drop(columns=[c for c in ("w", "stratum") if c in lp]).merge(plot_w, on="plot_id", how="left")
classified = classified.drop(columns=[c for c in ("w", "stratum") if c in classified]).merge(plot_w, on="plot_id", how="left")
strata_csv = os.path.join(out_dir, f"{country}_step7_strata_{stamp}.csv")
strata.to_csv(strata_csv, index=False, encoding="utf-8-sig")
print(strata.to_string(index=False, formatters={"ha per plot": "{:,.2f}".format, "area_ha": "{:,.0f}".format}))
areas_csv = os.path.join(out_dir, f"{country}_step7_class_areas_{stamp}.csv")
class_summary = class_areas(classified, per_plot)
if per_plot:                                                        # v17: area from the weights, with its standard error
    keys = ["from_year", "to_year", "level1", "decided_by", "forest_type", "driver"]
    c7 = classified.copy()
    c7["decided_by"] = c7["based_on"].fillna("").astype(str).str.strip().map(lambda v: "land use" if not v or v == "nan" else ("canopy" if v.startswith("canopy") else "disturbance" if v == "disturbance" else "rule"))
    c7["driver"] = c7["driver"].fillna("").astype(str).str.strip().replace("nan", "")
    c7["forest_type"] = ""
    gone = c7["level1"].isin(["Stable forest", "Deforestation", "Degradation", "Enhancement", "Forest subclass change", "To review"])
    c7.loc[gone, "forest_type"] = c7.loc[gone, "from_adj"]
    c7.loc[c7["level1"] == "Afforestation", "forest_type"] = c7.loc[c7["level1"] == "Afforestation", "to_adj"]
    est = estimate(c7, keys, strata).rename(columns={"level1": "class"})
    class_summary = class_summary.drop(columns="area_ha").merge(est.drop(columns="plots"), on=["from_year", "to_year", "class", "decided_by", "forest_type", "driver"], how="left")
class_summary.to_csv(areas_csv, index=False, encoding="utf-8-sig")
split = class_summary[class_summary["class"] == "Degradation"].groupby("decided_by")["plots"].sum()
if len(split):
    print("Degradation (plot-pairs): " + ", ".join(f"{k} {v}" for k, v in split.items()))
matrix_xlsx = os.path.join(out_dir, f"{country}_step7_matrices_{stamp}.xlsx")
matrix_csv = os.path.join(out_dir, f"{country}_step7_transitions_{stamp}.csv")
write_matrices(matrix_xlsx, lp, classes, per_plot, total_area_ha)
tl = transitions_long(lp, classes, per_plot)
if per_plot:
    est = estimate(lp, ["from_year", "to_year", "from_ipcc", "from_adj", "to_ipcc", "to_adj"], strata)
    tl = tl.drop(columns="area_ha").merge(est.drop(columns="plots").rename(columns={"from_adj": "from_subdivision", "to_adj": "to_subdivision"}),
                                          on=["from_year", "to_year", "from_ipcc", "from_subdivision", "to_ipcc", "to_subdivision"], how="left")
tl.to_csv(matrix_csv, index=False, encoding="utf-8-sig")
moca_xlsx = None
if per_plot:
    planted = planted_of(classes, planted_classes)
    moca = mocaredd_transitions(classified, strata, planted)
    if len(moca):
        moca_xlsx = os.path.join(out_dir, f"{country}_step7_mocaredd_transitions_{stamp}.xlsx")
        moca.to_excel(moca_xlsx, index=False)
        one = (moca["plots"] == 1).sum()
        print(f"\nmocaredd: {len(moca):,} transitions ({moca['redd_activity'].value_counts().to_dict()}); planted classes: {sorted(planted) or 'none'}.")
        print(f"⚠ {one:,} of them rest on one plot: their standard error equals their area (a Normal draw is below 0 one time in six).")
    nodist = classified[(classified["level1"] == "Degradation") & (classified["from_adj"] == classified["to_adj"])]
    if len(nodist):
        print(f"ℹ {len(nodist):,} plot-pairs of Degradation with no change of class (a disturbance or a canopy transition): in the class areas, not in the mocaredd file.")
w = whole_period(lp)
print(f"\nWhole period, IPCC categories (plots, rows = first year, columns = last year):")
print(pd.crosstab(w["from_ipcc"], w["to_ipcc"]).reindex(index=IPCC_ORDER, columns=IPCC_ORDER, fill_value=0).to_string())
if per_plot and sampling_design != "stratified":
    print(f"\nEach plot = {per_plot:,.2f} ha ({total_area_ha:,.0f} ha / {all_plots:,} plots).")
print(f"\n✅ Plots{' and area' if per_plot else ''} of each class per pair of years, with the REDD+ activity:\n   {areas_csv}")
print(f"✅ The matrices:\n   {matrix_xlsx}\n✅ The same numbers in one list:\n   {matrix_csv}")
FILES = [strata_csv, areas_csv, matrix_xlsx, matrix_csv] + ([moca_xlsx] if moca_xlsx else [])
if moca_xlsx:
    print(f"✅ The transitions for mocaredd (area, standard error, REDD+ activity):\n   {moca_xlsx}")
print(f"✅ The strata and the weights:\n   {strata_csv}")
try:
    from google.colab import files
    for f in FILES:
        files.download(f)
    print("⬇ The files are being downloaded to your Downloads folder.")
except ImportError:
    pass

## Step 8 - Units of land for the IPCC software
**What you do:** run it with *make the template*, check the yellow column, then choose *use my filled file* and run it again. Leave `transition_period` at 20 unless the country uses another one.

**Look at:** the total of each year must equal the total area.

In [ ]:
#@title Step 8 - Units of land for the IPCC software
#@markdown <font color="grey">Years. IPCC default = 20.</font>
transition_period = 20  #@param {type:"integer"}
subcategory_choice = "make the template"  #@param ["make the template", "use my filled file"]
#@markdown <font color="grey">Only on your own computer: the path of the filled file. In Colab leave it empty and choose the file.</font>
subcategory_file = ""  #@param {type:"string"}

if not globals().get("per_plot"):
    raise ValueError("⛔ Write total_area_ha in Step 7 and run Step 7 again: the software needs hectares.")
FILES = []
if subcategory_choice == "make the template":
    sub_template = os.path.join(out_dir, f"{country}_step8_ipcc_subcategories_{stamp}.xlsx")
    make_subcategory_template(sub_template, classes)
    FILES = [sub_template]
    print(f"✅ The template ({len(classes)} subdivisions, IPCC subcategory proposed):\n   {sub_template}")
    print("⏸ Check the yellow column, choose 'use my filled file' and run this step again. Do not forget to give the filled file: its path in the file field, or in Colab leave it empty and upload it.")
else:
    if subcategory_file.strip():
        sub_source = subcategory_file.strip()
    else:
        from google.colab import files
        up = files.upload()
        sub_source = io.BytesIO(up[list(up)[0]])
    classes = read_subcategories(sub_source, classes)
    sub_info = {r["subdivision"]: (r["IPCC category"], r["IPCC subcategory"]) for r in classes.to_dict("records")}
    units, unit_years, area_check = units_of_land(ipcc_periods(lp, per_plot), sub_info, transition_period)
    units_xlsx = os.path.join(out_dir, f"{country}_step8_ipcc_units_of_land_{stamp}.xlsx")
    land_table = write_ipcc_table(units_xlsx, units, unit_years, total_area_ha, transition_period)
    in_conv = (units["Year of conversion"] != "").sum()
    print(f"{len(units):,} rows: {len(units) - in_conv:,} with no change, {in_conv:,} with a change. Years {unit_years[0]}-{unit_years[-1]}.")
    bad = {y: v for y, v in area_check.items() if abs(v - total_area_ha) > 1}
    print("✅ The total of each year equals the total area." if not bad else f"⚠ The total differs from {total_area_ha:,.0f} ha in: {bad}")
    below = units[units[unit_years].apply(lambda r: any(isinstance(v, (int, float)) and v < 0 for v in r), axis=1)]
    if len(below):
        print(f"⚠ {len(below)} Remaining units go below zero: changes out of a class that gained its area only through conversions. Approach 2 cannot follow these; check them: {list(below['Current subdivision'])[:12]}")
    print(land_table.head(6).to_string(index=False))
    print(f"\n✅ The Excel for the Land Representation Manager (first sheet: what to do):\n   {units_xlsx}")
    FILES = [units_xlsx]
try:
    from google.colab import files
    for f in FILES:
        files.download(f)
    print("⬇ The files are being downloaded to your Downloads folder.")
except ImportError:
    pass

## Step 9 - Review of a few samples (optional)
**What you do:** run it after Step 6 (or 7). `n_samples` and `random_seed` can be changed (the same seed gives the same samples).

**You get:** a page with each sample year by year: the land and canopy recorded, what was recorded (change, disturbance), the class reported for the pair ending that year and **what decided it** (land use, a rule of the lookup, the canopy, a disturbance). The samples are chosen at random among those with several changes or automatic decisions. A CSV lists them with two empty columns for the reviewer (`agrees` yes / no, `comment`).

**Look at:** each sample on the images, year by year: is the land right, and is the class reported right?


In [ ]:
#@title Step 9 - Review of a few samples (optional)
n_samples = 5  #@param {type:"integer"}
random_seed = 1  #@param {type:"integer"}
if "classified" not in globals():
    raise ValueError("⛔ Run Step 6 first.")
score = complexity(lp, classified)
picks = pick_samples(score, n_samples, random_seed)
review_html = os.path.join(out_dir, f"{country}_step9_review_samples_{stamp}.html")
write_review(review_html, table, classified, picks, score, survey)
review_csv = os.path.join(out_dir, f"{country}_step9_review_samples_{stamp}.csv")
score.loc[picks].reset_index().assign(agrees="", comment="").to_csv(review_csv, index=False, encoding="utf-8-sig")
print(score.loc[picks].to_string())
print(f"\n✅ The samples to review, year by year:\n   {review_html}\n✅ The list, with columns for the reviewer:\n   {review_csv}")
try:
    from google.colab import files
    for f in [review_html, review_csv]:
        files.download(f)
except ImportError:
    pass


## Step 10 - The activity of a period (optional)
**What you do:** write the period (`period_start` = the first year of change, `period_end` = the last; e.g. 2014 and 2019 for a reference period, 2020 and 2024 for a monitoring period; 0 = all), run it with *make the template*, fill `redd_activity` in the template, choose *use my filled file*, **give the filled file** (its path in `history_lookup_file`, or in Colab leave it empty and upload it when the button appears) and run it again. Needs Steps 6 and 7.

- **The histories** are written in English, as the template writes them; a table written in French, Spanish or Portuguese is read too (e.g. *Gain forestier* = Afforestation, *Foret stable* = Stable forest), and **the country's own words** go in the sheet `activities` of the lexicon of Step 2 (columns `word`, `activity`, e.g. *gain* | Afforestation); Step 10 rereads the lexicon, so after adding a word only this step runs again. The step prints how each history was read; a word not known is listed.

**You get:** one Excel `_step10_period_<first>_<last>_` with the **area of each activity for the period, its standard error and 90% CI** (sheet *areas*), the activity of each plot (sheet *plots*) and of each history (sheet *histories*). Each plot gets one activity for the period, decided by the country from the **history** of the plot: the forest events of the period in the order they happened, from the reported classes (e.g. *Deforestation*, *Afforestation -> Deforestation*). The template lists the histories found with the number of plots and a proposal; nothing is decided by the code. Then the area of each activity for the period, with its standard error and 90% CI (one value per activity for the whole period; the yearly values are in Step 7). Each plot counts once in the period. For two periods (e.g. reference and monitoring), run the step twice.


In [ ]:
#@title Step 10 - The activity of a period (optional)
period_start = 0  #@param {type:"integer"}
period_end = 0  #@param {type:"integer"}
history_lookup_choice = "make the template"  #@param ["make the template", "use my filled file"]
#@markdown <font color="grey">The history lookup table: sheet history_lookup, columns history and redd_activity (one row per history of the data). The words of the histories are read with the lexicon of Step 2 (sheet activities, e.g. gain | Afforestation). 'make the template' writes it with the histories found. Only on your own computer: the path of the filled file. In Colab leave it empty and choose the file.</font>
history_lookup_file = ""  #@param {type:"string"}
if "classified" not in globals():
    raise ValueError("⛔ Run Step 6 first.")
first = int(period_start or classified["to_year"].min())
last = int(period_end or classified["to_year"].max())
if first > last:
    raise ValueError("⛔ period_start is after period_end.")
hist = period_histories(classified, first, last)
print(f"Period {first}-{last} (changes of {first} to {last}): {len(hist):,} plots")
print(hist["history"].value_counts().head(15).to_string())
FILES = []
if history_lookup_choice == "make the template":
    tpl = os.path.join(out_dir, f"{country}_step10_history_lookup_template_{first}_{last}_{stamp}.xlsx")
    write_period_template(tpl, hist, first, last)
    FILES = [tpl]
    print(f"\n✅ The template:\n   {tpl}\n⏸ Fill redd_activity, then choose 'use my filled file' and run this step again. "
          "Do not forget to give the filled file: its path in history_lookup_file, or in Colab leave it empty and upload it.")
else:
    if history_lookup_file.strip():
        psrc = history_lookup_file.strip()
    else:
        from google.colab import files
        up = files.upload()
        psrc = up[list(up)[0]]
    if globals().get("lexicon_source") is not None:
        read_history_words(excel_of(lexicon_source))     # the lexicon of Step 2, reread: words added since are read
    plook = read_period_lookup(psrc)
    seen = history_lookup_read(psrc)
    if HISTORY_ADDED:
        print(f"The country's activity words (lexicon of Step 2, sheet activities): {HISTORY_ADDED}")
    moved = seen[(seen["history in the file"] != seen["read as"]) | seen["words not known"].ne("")]
    if len(moved):
        print("How the histories of the file were read (the country's words, another language or spelling):")
        print(moved.to_string(index=False))
    unknown_words = sorted({w.strip() for x in seen["words not known"] for w in x.split(",") if w.strip()})
    absent = sorted(set(hist["history"]) - set(plook))
    if absent:
        print("Histories of the data, as Step 10 writes them, missing from the file:")
        print(hist[hist["history"].isin(absent)]["history"].value_counts().to_string())
        if unknown_words:
            print(f"Words of the file the notebook does not know: {unknown_words} - add them to the sheet activities of the lexicon (Step 2, lexicon_file: word, activity), save it and run this step again.")
        raise ValueError(f"⛔ {len(absent)} histories of the data are not in the file: add a row for each (history, redd_activity), "
                         "or start from the template of this step (make the template), which lists them all. Histories written in "
                         "French, Spanish or Portuguese are read too (e.g. Gain forestier = Afforestation); your own words go in the sheet activities of the lexicon (Step 2).")
    hist["redd_activity"] = hist["history"].map(plook)
    if globals().get("strata") is None or not globals().get("per_plot"):
        raise ValueError("⛔ The areas need Step 7 (total_area_ha and the sampling design): run Step 7, then this step again.")
    rows = hist.merge(plot_w, on="plot_id", how="left").assign(from_year=first)
    est = estimate(rows, ["from_year", "redd_activity"], strata).drop(columns="from_year")
    est["ci90_ha"] = 1.645 * est["se_ha"]
    est["rel_ci90"] = (est["ci90_ha"] / est["area_ha"]).where(est["area_ha"] > 0)
    est.insert(0, "period", f"{first}-{last}")
    est = est.sort_values("area_ha", ascending=False).reset_index(drop=True)
    print(f"\nActivity of the period {first}-{last} - one activity per plot (history lookup table), area and uncertainty:")
    show = est.assign(**{"90% CI (%)": (100 * est["rel_ci90"]).round(1)}).drop(columns=["rel_ci90", "period"])
    print(show.to_string(index=False, float_format=lambda v: f"{v:,.1f}"))
    out_xlsx = os.path.join(out_dir, f"{country}_step10_period_{first}_{last}_{stamp}.xlsx")
    write_period_result(out_xlsx, est, hist, first, last)
    FILES = [out_xlsx]
    print(f"\n✅ The activity of the period {first}-{last} (sheets areas, plots, histories):\n   {out_xlsx}")
try:
    from google.colab import files
    for f in FILES:
        files.download(f)
except ImportError:
    pass
